In [1]:
# Install only when the Kaggle image does not already provide the package.
try:
    import kaggle_environments
except ImportError:
    %pip install -q "kaggle-environments>=1.28.0"

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

if Path("/kaggle/working").exists():
    WORK_DIR = Path("/kaggle/working")
else:
    local_root = Path.cwd()
    WORK_DIR = (
        local_root / "whole_plan_runtime"
        if local_root.name.lower() == "lastrl"
        else local_root / "lastRL" / "whole_plan_runtime"
    )
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("runtime directory:", WORK_DIR.resolve())


[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

## Heuristic & PPO Implementation

These two cells contain the implementation of heuristic + PPO (included in the original folder). We inject them here just for usability on Kaggle.


In [4]:
HEURISTIC_PATH = WORK_DIR / 'whole_plan_heuristic.py'
HEURISTIC_PATH.write_text('\nfrom __future__ import annotations\nimport dataclasses\nimport math\nimport os\nimport sys\nfrom dataclasses import dataclass\nfrom typing import Any, Sequence\nimport torch\nfrom torch import Tensor\n\n# MAIN V12: midpoint sweep for the opening production tiebreaker after\n# weights 0.75 and 3.0 tied baseline while 1.25 was strong but volatile.\n\nB_DEFAULT: int = 1024\nP_MAX: int = 64\nF_MAX: int = 256\nA: int = 2\nBOARD_SIZE: float = 100.0\nCENTER: float = 50.0\nSUN_RADIUS: float = 10.0\nMAX_SHIP_SPEED: float = 6.0\nROT_RADIUS_LIMIT: float = 50.0\nOWN: int = 0\nENEMY: int = 1\nNEUTRAL: int = 2\nDEAD: int = 3\nLIBRARY_K_DEFAULT: int = 100000\nCOMET_EVENTS: int = 5\nCOMETS_PER_EVENT: int = 4\nCOMET_PATH_MAX: int = 40\nCOMET_SPAWN_STEPS: tuple[int, ...] = (50, 150, 250, 350, 450)\nCOMET_RADIUS: float = 1.0\nCOMET_PRODUCTION: float = 1.0\nEARLY_TERM_MARGIN: float = 2.0\nEARLY_TERM_STREAK_2P: int = 5\nEARLY_TERM_STREAK_4P: int = 20\nEARLY_TERM_PROD_WEIGHT_2P: float = 5.0\nEARLY_TERM_SHIP_WEIGHT_2P: float = 1.0\nEARLY_TERM_PROD_WEIGHT_4P: float = 1.0\nEARLY_TERM_SHIP_WEIGHT_4P: float = 0.0\nDEFAULT_EPISODE_STEPS: int = 500\nTARGET_SHORTLIST_PROD_WEIGHT: float = 1.0\nTARGET_SHORTLIST_PROD_TURN_LIMIT: int = 40\n\ndef orbit_phase_index_from_obs_step(obs_step: Tensor) -> Tensor:\n    s = obs_step.float()\n    return (s - (s > 0).to(s.dtype)).clamp(min=0.0)\n_LOG_1000: float = float(torch.log(torch.tensor(1000.0)).item())\n_FLEET_SPEED_LUT_MAX: int = 400\n\ndef _fleet_speed_formula(ships: Tensor) -> Tensor:\n    ratio = (torch.log(ships) / _LOG_1000).clamp(max=1.0)\n    return 1.0 + (MAX_SHIP_SPEED - 1.0) * ratio.pow(1.5)\n\ndef _build_fleet_speed_lut(max_ships: int) -> Tensor:\n    idx = torch.arange(max_ships + 1, dtype=torch.float32).clamp(min=1.0)\n    return _fleet_speed_formula(idx)\n_FLEET_SPEED_LUT: Tensor = _build_fleet_speed_lut(_FLEET_SPEED_LUT_MAX)\n_FLEET_SPEED_LUT_CACHE: dict[tuple, Tensor] = {}\n\ndef _fleet_speed_lut_on(device: torch.device, dtype: torch.dtype) -> Tensor:\n    key = (device, dtype)\n    cached = _FLEET_SPEED_LUT_CACHE.get(key)\n    if cached is None:\n        cached = _FLEET_SPEED_LUT.to(device=device, dtype=dtype)\n        _FLEET_SPEED_LUT_CACHE[key] = cached\n    return cached\n\ndef fleet_speed(ships: Tensor) -> Tensor:\n    s = ships.clamp(min=1.0)\n    s_lut = s.clamp(max=float(_FLEET_SPEED_LUT_MAX))\n    lo = torch.floor(s_lut).long()\n    hi = torch.ceil(s_lut).long()\n    frac = s_lut - lo.to(dtype=s.dtype)\n    lut = _fleet_speed_lut_on(s.device, s.dtype)\n    speed = lut[lo] + (lut[hi] - lut[lo]) * frac\n    over = s > float(_FLEET_SPEED_LUT_MAX)\n    speed_formula = _fleet_speed_formula(s)\n    return torch.where(over, speed_formula, speed)\n\n@dataclass\nclass ParsedObs:\n    alive: Tensor\n    x: Tensor\n    y: Tensor\n    r: Tensor\n    ships: Tensor\n    prod: Tensor\n    owner_abs: Tensor\n    owned: Tensor\n    is_enemy: Tensor\n    is_neutral: Tensor\n    orb_r: Tensor\n    orb_a0: Tensor\n    is_orbiting: Tensor\n    angvel: Tensor\n    step: Tensor\n    f_alive: Tensor\n    f_owner: Tensor\n    f_x: Tensor\n    f_y: Tensor\n    f_angle: Tensor\n    f_ships: Tensor\n    player_id: int\n    P: int\n    F: int\n    device: torch.device\n\ndef parse_obs(obs_tensors: dict, player_id: int | None=None) -> ParsedObs:\n    planets = obs_tensors[\'planets\']\n    initial = obs_tensors[\'initial_planets\']\n    fleets = obs_tensors[\'fleets\']\n    angvel = obs_tensors[\'angular_velocity\'].float()\n    step = obs_tensors[\'step\'].float()\n    if player_id is None:\n        player_id = int(obs_tensors[\'player\'].flatten()[0].item())\n    P, _ = planets.shape\n    F, _ = fleets.shape\n    device = planets.device\n    pid = planets[..., 0]\n    owner_abs = planets[..., 1]\n    x = planets[..., 2]\n    y = planets[..., 3]\n    r = planets[..., 4]\n    ships = planets[..., 5]\n    prod = planets[..., 6]\n    alive = pid >= 0.0\n    owned = alive & (owner_abs == float(player_id))\n    is_enemy = alive & (owner_abs >= 0.0) & (owner_abs != float(player_id))\n    is_neutral = alive & (owner_abs < 0.0)\n    ix = initial[..., 2]\n    iy = initial[..., 3]\n    i_r = initial[..., 4]\n    dx0 = ix - CENTER\n    dy0 = iy - CENTER\n    orb_r_raw = torch.sqrt(dx0 * dx0 + dy0 * dy0)\n    orb_a0 = torch.atan2(dy0, dx0)\n    is_orbiting = alive & (orb_r_raw + i_r < ROT_RADIUS_LIMIT) & (orb_r_raw > 0.5)\n    orb_r = torch.where(is_orbiting, orb_r_raw, torch.zeros_like(orb_r_raw))\n    f_pid = fleets[..., 0]\n    f_alive = f_pid >= 0.0\n    f_owner = fleets[..., 1]\n    f_x = fleets[..., 2]\n    f_y = fleets[..., 3]\n    f_angle = fleets[..., 4]\n    f_ships = fleets[..., 6]\n    return ParsedObs(alive=alive, x=x, y=y, r=r, ships=ships, prod=prod, owner_abs=owner_abs, owned=owned, is_enemy=is_enemy, is_neutral=is_neutral, orb_r=orb_r, orb_a0=orb_a0, is_orbiting=is_orbiting, angvel=angvel, step=step, f_alive=f_alive, f_owner=f_owner, f_x=f_x, f_y=f_y, f_angle=f_angle, f_ships=f_ships, player_id=player_id, P=P, F=F, device=device)\nLAUNCH_SURFACE_OFFSET: float = 0.1\nTARGET_HIT_SURFACE_OFFSET: float = 0.0\nKAGGLE_SUN_RADIUS: float = SUN_RADIUS\n\ndef _swept_pair_hit_mask(ax: Tensor, ay: Tensor, bx: Tensor, by: Tensor, p0x: Tensor, p0y: Tensor, p1x: Tensor, p1y: Tensor, r: Tensor) -> Tensor:\n    d0x = ax - p0x\n    d0y = ay - p0y\n    dvx = bx - ax - (p1x - p0x)\n    dvy = by - ay - (p1y - p0y)\n    a = dvx * dvx + dvy * dvy\n    b = 2.0 * (d0x * dvx + d0y * dvy)\n    c = d0x * d0x + d0y * d0y - r * r\n    near_static = a < 1e-12\n    c_hit = c <= 0.0\n    disc = b * b - 4.0 * a * c\n    has_root = disc >= 0.0\n    safe_a = torch.where(near_static, torch.ones_like(a), a)\n    sq = torch.sqrt(torch.clamp(disc, min=0.0))\n    t1 = (-b - sq) / (2.0 * safe_a)\n    t2 = (-b + sq) / (2.0 * safe_a)\n    quad_hit = has_root & (t2 >= 0.0) & (t1 <= 1.0)\n    return torch.where(near_static, c_hit, quad_hit)\nDEFAULT_MOVEMENT_HORIZON = 20\nDEFAULT_DRIFT_EPSILON = 0.0001\nDEFAULT_MAX_TRACKED_FLEETS = 64\n\n@dataclass(frozen=True)\nclass MovementConfig:\n    movement_horizon: int = DEFAULT_MOVEMENT_HORIZON\n    drift_epsilon: float = DEFAULT_DRIFT_EPSILON\n    track_fleets: bool = False\n    player_count: int | None = None\n    max_tracked_fleets: int = DEFAULT_MAX_TRACKED_FLEETS\n\n@dataclass(frozen=True)\nclass PlanetGarrisonStatus:\n    owner: Tensor\n    ships: Tensor\n    pre_combat_owner: Tensor | None = None\n    pre_combat_ships: Tensor | None = None\n    arrivals_by_owner: Tensor | None = None\n\n@dataclass\nclass PlanetMovement:\n    x: Tensor\n    y: Tensor\n    alive_by_step: Tensor\n    planet_ids: Tensor\n    radii: Tensor\n    planet_owner: Tensor\n    planet_ships: Tensor\n    planet_prod: Tensor\n    base_step: Tensor\n    comet_planet_ids: Tensor\n    comet_path_index: Tensor\n    movement_horizon: int = DEFAULT_MOVEMENT_HORIZON\n    drift_epsilon: float = DEFAULT_DRIFT_EPSILON\n    track_fleets: bool = False\n    player_count: int | None = None\n    max_tracked_fleets: int = DEFAULT_MAX_TRACKED_FLEETS\n    fleet_buckets: Tensor | None = None\n    fleet_last_step: Tensor | None = None\n    tracked_fleet_ids: Tensor | None = None\n    tracked_fleet_eta: Tensor | None = None\n    tracked_fleet_target_slot: Tensor | None = None\n    tracked_fleet_owner: Tensor | None = None\n    tracked_fleet_ships: Tensor | None = None\n    garrison_owner_cache: Tensor | None = None\n    garrison_ships_cache: Tensor | None = None\n    garrison_pre_combat_owner_cache: Tensor | None = None\n    garrison_pre_combat_ships_cache: Tensor | None = None\n    garrison_dirty_from: Tensor | None = None\n    pending_source_planets: Tensor | None = None\n    pending_ships: Tensor | None = None\n    pending_angle: Tensor | None = None\n    pending_target_slots: Tensor | None = None\n    pending_eta: Tensor | None = None\n    pending_owners: Tensor | None = None\n    pending_prev_nfid: Tensor | None = None\n    pending_stash_step: Tensor | None = None\n\n    @property\n    def P(self) -> int:\n        return int(self.planet_ids.shape[0])\n\n    @property\n    def device(self) -> torch.device:\n        return self.x.device\n\n    @property\n    def dtype(self) -> torch.dtype:\n        return self.x.dtype\n\n    @property\n    def config(self) -> MovementConfig:\n        return MovementConfig(movement_horizon=int(self.movement_horizon), drift_epsilon=float(self.drift_epsilon), track_fleets=bool(self.track_fleets), player_count=self.player_count, max_tracked_fleets=int(self.max_tracked_fleets))\n\n    @classmethod\n    def from_obs_tensors(cls, obs_tensors: dict, *, config: MovementConfig | None=None, movement_horizon: int=DEFAULT_MOVEMENT_HORIZON, drift_epsilon: float=DEFAULT_DRIFT_EPSILON, track_fleets: bool=False, player_count: int | None=None, max_tracked_fleets: int=DEFAULT_MAX_TRACKED_FLEETS) -> \'PlanetMovement\':\n        cfg = config if config is not None else MovementConfig(movement_horizon=int(movement_horizon), drift_epsilon=float(drift_epsilon), track_fleets=bool(track_fleets), player_count=player_count, max_tracked_fleets=int(max_tracked_fleets))\n        built = _build_future_from_obs(obs_tensors, int(cfg.movement_horizon))\n        resolved_player_count = _resolve_player_count(obs_tensors, cfg.player_count) if cfg.track_fleets else cfg.player_count\n        movement = cls(x=built[\'x\'], y=built[\'y\'], alive_by_step=built[\'alive_by_step\'], planet_ids=built[\'planet_ids\'], radii=built[\'radii\'], planet_owner=built[\'owner\'], planet_ships=built[\'ships\'], planet_prod=built[\'prod\'], base_step=built[\'step\'], comet_planet_ids=built[\'comet_planet_ids\'], comet_path_index=built[\'comet_path_index\'], movement_horizon=int(cfg.movement_horizon), drift_epsilon=float(cfg.drift_epsilon), track_fleets=bool(cfg.track_fleets), player_count=resolved_player_count, max_tracked_fleets=int(cfg.max_tracked_fleets))\n        if movement.track_fleets:\n            movement._init_fleet_tracking(obs_tensors, reset_ledger=True)\n            movement._ingest_obs_fleets(obs_tensors)\n        return movement\n\n    def update(self, obs_tensors: dict) -> \'PlanetMovement\':\n        planets = obs_tensors[\'planets\']\n        if planets.device != self.device or planets.shape[0] != self.P or int(self.x.shape[0]) != int(self.movement_horizon) + 1:\n            fresh = type(self).from_obs_tensors(obs_tensors, movement_horizon=self.movement_horizon, drift_epsilon=self.drift_epsilon, track_fleets=self.track_fleets, player_count=self.player_count, max_tracked_fleets=int(self.max_tracked_fleets))\n            self._copy_from(fresh)\n            return self\n        if self.track_fleets:\n            current_player_count = _resolve_player_count(obs_tensors, self.player_count)\n            if self.fleet_buckets is None or self.fleet_last_step is None or self.tracked_fleet_ids is None or (tuple(self.fleet_buckets.shape) != (self.P, int(self.movement_horizon), int(current_player_count))) or (self.fleet_buckets.device != self.device) or (int(self.tracked_fleet_ids.shape[0]) < int(self.max_tracked_fleets)):\n                self.player_count = int(current_player_count)\n                self._init_fleet_tracking(obs_tensors, reset_ledger=True)\n        obs_for_decision = parse_obs(obs_tensors)\n        H = int(self.movement_horizon)\n        planet_ids_now = planets[..., 0].long()\n        radii_now = planets[..., 4].to(dtype=self.dtype)\n        owner_now = planets[..., 1].to(device=self.device, dtype=torch.long)\n        owner_now = torch.where(obs_for_decision.alive, owner_now, torch.full_like(owner_now, -1))\n        ships_now = planets[..., 5].to(device=self.device, dtype=self.dtype)\n        prod_now = planets[..., 6].to(device=self.device, dtype=self.dtype)\n        step_now = obs_for_decision.step.to(device=self.device, dtype=torch.long)\n        comet_ids_now, comet_idx_now = _comet_metadata(obs_tensors, self.device)\n        current_obs_x = planets[..., 2].to(device=self.device, dtype=self.dtype)\n        current_obs_y = planets[..., 3].to(device=self.device, dtype=self.dtype)\n        current_alive = obs_for_decision.alive\n        ids_same = bool((planet_ids_now == self.planet_ids).all())\n        same_step = bool(step_now == self.base_step)\n        next_step = bool(step_now == self.base_step + 1)\n        comet_same = _same_2d(comet_ids_now, self.comet_planet_ids)\n        comet_idx_same = _same_2d(comet_idx_now, self.comet_path_index)\n        expected_next_idx = torch.where(self.comet_path_index >= 0, self.comet_path_index + 1, self.comet_path_index)\n        comet_idx_next = _same_2d(comet_idx_now, expected_next_idx)\n        same_alive_ok = bool((current_alive == self.alive_by_step[0]).all())\n        next_alive_ok = bool((current_alive == self.alive_by_step[1]).all())\n        same_drift_ok = _position_matches(self.x[0], self.y[0], current_obs_x, current_obs_y, current_alive, float(self.drift_epsilon))\n        next_drift_ok = _position_matches(self.x[1], self.y[1], current_obs_x, current_obs_y, current_alive, float(self.drift_epsilon))\n        keep = ids_same and same_step and comet_same and comet_idx_same and same_alive_ok and same_drift_ok\n        roll = ids_same and next_step and comet_same and comet_idx_next and next_alive_ok and next_drift_ok\n        rebuild = not (keep or roll)\n        if rebuild:\n            built = _build_future_from_obs(obs_tensors, H)\n        elif roll:\n            last_offset = torch.tensor([H], dtype=torch.long, device=self.device)\n            built = _build_future_from_obs(obs_tensors, H, offsets=last_offset)\n        else:\n            built = None\n        if roll:\n            assert built is not None\n            self.x[:-1] = self.x[1:].clone()\n            self.y[:-1] = self.y[1:].clone()\n            self.alive_by_step[:-1] = self.alive_by_step[1:].clone()\n            self.x[-1] = built[\'x\'][-1]\n            self.y[-1] = built[\'y\'][-1]\n            self.alive_by_step[-1] = built[\'alive_by_step\'][-1]\n            self._roll_garrison_projection()\n        if rebuild:\n            assert built is not None\n            self.x[:] = built[\'x\']\n            self.y[:] = built[\'y\']\n            self.alive_by_step[:] = built[\'alive_by_step\']\n            self._mark_garrison_dirty_all(0)\n        if roll or rebuild:\n            self.planet_ids[:] = planet_ids_now\n            self.radii[:] = radii_now\n            self.base_step = step_now\n            self.comet_planet_ids = comet_ids_now\n            self.comet_path_index = comet_idx_now\n        self._refresh_garrison_base({\'planet_ids\': planet_ids_now, \'radii\': radii_now, \'owner\': owner_now, \'ships\': ships_now, \'prod\': prod_now, \'step\': step_now})\n        if self.track_fleets:\n            self._roll_fleet_buckets_phase1(step_now)\n            if rebuild and (not ids_same):\n                self._reset_fleet_tracking()\n            self._reconcile_pending_own_launches(obs_tensors)\n            self._ingest_obs_fleets(obs_tensors)\n            self._reconcile_obs_fleets(obs_tensors)\n        return self\n\n    def all_positions(self, k: int) -> tuple[Tensor, Tensor]:\n        idx = self._k_index(k)\n        return (self.x[idx], self.y[idx])\n\n    def alive_at(self, k: int) -> Tensor:\n        return self.alive_by_step[self._k_index(k)]\n\n    def position_at_slots(self, slots: Tensor, k: int) -> tuple[Tensor, Tensor]:\n        slots = slots.to(device=self.device, dtype=torch.long).clamp(0, max(self.P - 1, 0))\n        px, py = self.all_positions(k)\n        out_x = px[slots].to(dtype=self.dtype)\n        out_y = py[slots].to(dtype=self.dtype)\n        return (out_x, out_y)\n\n    def pairwise_distance(self, k: int) -> Tensor:\n        px, py = self.all_positions(k)\n        dx = px.unsqueeze(1) - px.unsqueeze(0)\n        dy = py.unsqueeze(1) - py.unsqueeze(0)\n        return torch.sqrt((dx * dx + dy * dy).clamp(min=0.0))\n\n    def garrison_status(self, planet_slots: Tensor | None=None, *, max_horizon: int | None=None) -> PlanetGarrisonStatus:\n        self._require_fleet_buckets()\n        slots, out_prefix = self._normalize_garrison_slots(planet_slots)\n        requested_horizon = int(self.movement_horizon if max_horizon is None else max(0, min(int(max_horizon), int(self.movement_horizon))))\n        self._refresh_garrison_projection(slots, requested_horizon=requested_horizon)\n        assert self.garrison_owner_cache is not None\n        assert self.garrison_ships_cache is not None\n        assert self.garrison_dirty_from is not None\n        owner = self.garrison_owner_cache[slots][:, :requested_horizon + 1].reshape(*out_prefix, requested_horizon + 1)\n        ships = self.garrison_ships_cache[slots][:, :requested_horizon + 1].reshape(*out_prefix, requested_horizon + 1)\n        pre_combat_owner: Tensor | None = None\n        pre_combat_ships: Tensor | None = None\n        if self.garrison_pre_combat_owner_cache is not None and self.garrison_pre_combat_ships_cache is not None:\n            pre_combat_owner = self.garrison_pre_combat_owner_cache[slots][:, :requested_horizon + 1].reshape(*out_prefix, requested_horizon + 1)\n            pre_combat_ships = self.garrison_pre_combat_ships_cache[slots][:, :requested_horizon + 1].reshape(*out_prefix, requested_horizon + 1)\n        arrivals_by_owner: Tensor | None = None\n        if self.fleet_buckets is not None and requested_horizon > 0:\n            A = int(self.fleet_buckets.shape[-1])\n            arrivals_full = self.fleet_buckets[slots].reshape(*out_prefix, int(self.movement_horizon), A)\n            arrivals_trimmed = arrivals_full[..., :requested_horizon, :]\n            zero_frame = torch.zeros(*out_prefix, 1, A, dtype=arrivals_trimmed.dtype, device=self.device)\n            arrivals_by_owner = torch.cat([zero_frame, arrivals_trimmed], dim=-2)\n        status = PlanetGarrisonStatus(owner=owner, ships=ships, pre_combat_owner=pre_combat_owner, pre_combat_ships=pre_combat_ships, arrivals_by_owner=arrivals_by_owner)\n        return status\n\n    def _clear_pending_mask(self, mask: Tensor) -> None:\n        if self.pending_owners is None:\n            return\n        self.pending_owners[mask] = -1\n        assert self.pending_source_planets is not None\n        self.pending_source_planets[mask] = -1\n        assert self.pending_ships is not None\n        self.pending_ships[mask] = 0\n        assert self.pending_angle is not None\n        self.pending_angle[mask] = 0.0\n        assert self.pending_target_slots is not None\n        self.pending_target_slots[mask] = -1\n        assert self.pending_eta is not None\n        self.pending_eta[mask] = 0.0\n        assert self.pending_prev_nfid is not None\n        self.pending_prev_nfid[mask] = 0\n        assert self.pending_stash_step is not None\n        self.pending_stash_step[mask] = -1\n\n    def _ensure_pending_capacity(self, needed: int) -> None:\n        device = self.device\n        if self.pending_owners is None:\n            initial = max(4, int(needed))\n            shape = (initial,)\n            self.pending_owners = torch.full(shape, -1, dtype=torch.long, device=device)\n            self.pending_source_planets = torch.full(shape, -1, dtype=torch.long, device=device)\n            self.pending_ships = torch.zeros(shape, dtype=torch.long, device=device)\n            self.pending_angle = torch.zeros(shape, dtype=self.dtype, device=device)\n            self.pending_target_slots = torch.full(shape, -1, dtype=torch.long, device=device)\n            self.pending_eta = torch.zeros(shape, dtype=self.dtype, device=device)\n            self.pending_prev_nfid = torch.zeros(shape, dtype=torch.long, device=device)\n            self.pending_stash_step = torch.full(shape, -1, dtype=torch.long, device=device)\n            return\n        assert self.pending_owners is not None\n        empty_count = int((self.pending_owners == -1).sum().item())\n        shortage = int(needed) - empty_count\n        if shortage <= 0:\n            return\n        cur_L = int(self.pending_owners.shape[0])\n        extra = max(shortage, cur_L)\n        new_L = cur_L + extra\n\n        def _grow(t: Tensor, fill: float | int) -> Tensor:\n            extension = torch.full((new_L - cur_L,), fill, dtype=t.dtype, device=device)\n            return torch.cat([t, extension], dim=0)\n        self.pending_owners = _grow(self.pending_owners, -1)\n        assert self.pending_source_planets is not None\n        self.pending_source_planets = _grow(self.pending_source_planets, -1)\n        assert self.pending_ships is not None\n        self.pending_ships = _grow(self.pending_ships, 0)\n        assert self.pending_angle is not None\n        self.pending_angle = _grow(self.pending_angle, 0.0)\n        assert self.pending_target_slots is not None\n        self.pending_target_slots = _grow(self.pending_target_slots, -1)\n        assert self.pending_eta is not None\n        self.pending_eta = _grow(self.pending_eta, 0.0)\n        assert self.pending_prev_nfid is not None\n        self.pending_prev_nfid = _grow(self.pending_prev_nfid, 0)\n        assert self.pending_stash_step is not None\n        self.pending_stash_step = _grow(self.pending_stash_step, -1)\n\n    def stash_pending_own_launches(self, *, owner_id: int | Tensor, source_slots: Tensor, ships: Tensor, angle: Tensor, target_slots: Tensor, eta: Tensor, valid: Tensor, prev_next_fleet_id: int | Tensor) -> None:\n        if not self.track_fleets:\n            return\n        device = self.device\n        valid_mask = valid.to(device=device, dtype=torch.bool).reshape(-1)\n        if not bool(valid_mask.any()):\n            return\n        src = source_slots.to(device=device, dtype=torch.long).reshape(-1)\n        ships_t = ships.to(device=device, dtype=torch.long).reshape(-1)\n        angle_t = angle.to(device=device, dtype=self.dtype).reshape(-1)\n        tgt_t = target_slots.to(device=device, dtype=torch.long).reshape(-1)\n        eta_t = eta.to(device=device, dtype=self.dtype).reshape(-1)\n        src_safe = src.clamp(min=0, max=max(int(self.P) - 1, 0))\n        source_planet_ids = self.planet_ids[src_safe]\n        L_in = int(valid_mask.shape[0])\n        if isinstance(prev_next_fleet_id, Tensor):\n            prev_nfid_scalar = int(prev_next_fleet_id.flatten()[0].item())\n        else:\n            prev_nfid_scalar = int(prev_next_fleet_id)\n        prev_nfid_L = torch.full((L_in,), prev_nfid_scalar, dtype=torch.long, device=device)\n        owner_scalar = int(owner_id.flatten()[0].item()) if isinstance(owner_id, Tensor) else int(owner_id)\n        owner_L = torch.full((L_in,), owner_scalar, dtype=torch.long, device=device)\n        stash_step_scalar = int(self.base_step.item()) if isinstance(self.base_step, Tensor) else -1\n        stash_step_L = torch.full((L_in,), stash_step_scalar, dtype=torch.long, device=device)\n        if self.pending_owners is not None:\n            same_owner = self.pending_owners == owner_scalar\n            if bool(same_owner.any()):\n                self._clear_pending_mask(same_owner)\n        per_needed = int(valid_mask.sum().item())\n        self._ensure_pending_capacity(per_needed)\n        assert self.pending_owners is not None\n        empty_slots = torch.nonzero(self.pending_owners == -1, as_tuple=True)[0]\n        k_in = torch.nonzero(valid_mask, as_tuple=True)[0]\n        slot_in_pending = empty_slots[:k_in.numel()]\n        self.pending_owners[slot_in_pending] = owner_L[k_in]\n        assert self.pending_source_planets is not None\n        self.pending_source_planets[slot_in_pending] = source_planet_ids[k_in]\n        assert self.pending_ships is not None\n        self.pending_ships[slot_in_pending] = ships_t[k_in]\n        assert self.pending_angle is not None\n        self.pending_angle[slot_in_pending] = angle_t[k_in]\n        assert self.pending_target_slots is not None\n        self.pending_target_slots[slot_in_pending] = tgt_t[k_in]\n        assert self.pending_eta is not None\n        self.pending_eta[slot_in_pending] = eta_t[k_in]\n        assert self.pending_prev_nfid is not None\n        self.pending_prev_nfid[slot_in_pending] = prev_nfid_L[k_in]\n        assert self.pending_stash_step is not None\n        self.pending_stash_step[slot_in_pending] = stash_step_L[k_in]\n\n    def _reconcile_pending_own_launches(self, obs_tensors: dict) -> None:\n        if not self.track_fleets:\n            return\n        if self.pending_owners is None or self.tracked_fleet_ids is None:\n            return\n        active_mask = self.pending_owners != -1\n        if not bool(active_mask.any()):\n            return\n        device = self.device\n        step_tensor = obs_tensors.get(\'step\')\n        if step_tensor is not None:\n            assert self.pending_stash_step is not None\n            step_scalar = int(step_tensor.flatten()[0].item()) if isinstance(step_tensor, Tensor) else int(step_tensor)\n            advanced = step_scalar > self.pending_stash_step\n            active_mask = active_mask & advanced\n        if not bool(active_mask.any()):\n            return\n        fleets = obs_tensors[\'fleets\'].to(device=device)\n        fleet_ids = fleets[..., 0].to(dtype=torch.long)\n        obs_owner = fleets[..., 1].to(dtype=torch.long)\n        obs_angle = fleets[..., 4].to(dtype=self.dtype)\n        obs_from = fleets[..., 5].to(dtype=torch.long)\n        obs_ships = fleets[..., 6].to(dtype=torch.long)\n        assert self.pending_owners is not None\n        assert self.pending_source_planets is not None\n        assert self.pending_ships is not None\n        assert self.pending_angle is not None\n        assert self.pending_target_slots is not None\n        assert self.pending_eta is not None\n        assert self.pending_prev_nfid is not None\n        match_FL = active_mask.unsqueeze(0) & (fleet_ids.unsqueeze(1) >= 0) & (obs_owner.unsqueeze(1) == self.pending_owners.unsqueeze(0)) & (obs_from.unsqueeze(1) == self.pending_source_planets.unsqueeze(0)) & (obs_ships.unsqueeze(1) == self.pending_ships.unsqueeze(0)) & (obs_angle.unsqueeze(1) == self.pending_angle.unsqueeze(0)) & (fleet_ids.unsqueeze(1) >= self.pending_prev_nfid.unsqueeze(0))\n        INF = torch.iinfo(torch.long).max\n        id_for_match = torch.where(match_FL, fleet_ids.unsqueeze(1).expand_as(match_FL), torch.full_like(match_FL, INF, dtype=torch.long))\n        chosen_id, _ = id_for_match.min(dim=0)\n        eta_now = torch.ceil(self.pending_eta).to(dtype=torch.long) - 1\n        expect_obs_match = active_mask & (eta_now > 0)\n        no_match = expect_obs_match & (chosen_id == INF)\n        matched = expect_obs_match & (chosen_id != INF)\n        if int(active_mask.shape[0]) > 1:\n            chosen_for_matched = torch.where(matched, chosen_id, torch.full_like(chosen_id, INF))\n            sorted_ids, _ = chosen_for_matched.sort()\n            dup = bool(((sorted_ids[1:] == sorted_ids[:-1]) & (sorted_ids[1:] != INF)).any())\n            if dup:\n                raise AssertionError(\'Pending-launch reconciliation: multiple pending entries resolved to the same engine fleet id. This usually means multi-launch from the same source with identical (ships, angle) tuples processed in an unexpected order.\')\n        if bool(matched.any()):\n            l_idx = torch.where(matched)[0]\n            real_ids = chosen_id[l_idx]\n            self._ledger_bulk_insert(real_ids, eta_now[l_idx], self.pending_target_slots[l_idx], self.pending_owners[l_idx], self.pending_ships[l_idx].to(dtype=self.dtype))\n        if bool(no_match.any()):\n            self._decrement_unmatched_arrivals(no_match)\n        self._clear_pending_mask(active_mask)\n\n    def _decrement_unmatched_arrivals(self, no_match: Tensor) -> None:\n        assert self.pending_eta is not None\n        assert self.pending_owners is not None\n        assert self.pending_ships is not None\n        assert self.pending_target_slots is not None\n        buckets = self._require_fleet_buckets()\n        eta_now = torch.ceil(self.pending_eta).to(dtype=torch.long) - 1\n        h_idx_now = eta_now - 1\n        H = int(self.movement_horizon)\n        Aowner = int(buckets.shape[2])\n        valid = no_match & (h_idx_now >= 0) & (h_idx_now < H) & (self.pending_target_slots >= 0) & (self.pending_target_slots < int(self.P)) & (self.pending_owners >= 0) & (self.pending_owners < Aowner) & (self.pending_ships > 0)\n        if not bool(valid.any()):\n            return\n        target = self.pending_target_slots[valid]\n        h_idx_sel = h_idx_now[valid]\n        owner_sel = self.pending_owners[valid]\n        ships_sel = self.pending_ships[valid].to(dtype=self.dtype)\n        buckets.index_put_((target, h_idx_sel, owner_sel), -ships_sel, accumulate=True)\n        self._mark_garrison_dirty(target, h_idx_sel + 1)\n\n    def record_fleet_arrivals(self, *, target_slots: Tensor, owner_ids: Tensor | int, ships: Tensor, eta: Tensor, valid: Tensor | None=None) -> None:\n        buckets = self._require_fleet_buckets()\n        target_slots, ships, eta = torch.broadcast_tensors(target_slots.to(device=self.device, dtype=torch.long), ships.to(device=self.device, dtype=self.dtype), eta.to(device=self.device, dtype=self.dtype))\n        if isinstance(owner_ids, int):\n            owner = torch.full_like(target_slots, int(owner_ids), dtype=torch.long, device=self.device)\n        else:\n            owner = torch.broadcast_to(owner_ids.to(device=self.device, dtype=torch.long), target_slots.shape)\n        if valid is None:\n            valid_mask = torch.ones_like(target_slots, dtype=torch.bool)\n        else:\n            valid_mask = torch.broadcast_to(valid.to(device=self.device, dtype=torch.bool), target_slots.shape)\n        h_idx = torch.ceil(eta).to(dtype=torch.long) - 1\n        valid_mask = valid_mask & (target_slots >= 0) & (target_slots < self.P) & (owner >= 0) & (owner < int(buckets.shape[2])) & (h_idx >= 0) & (h_idx < int(self.movement_horizon)) & (ships > 0.0)\n        if not bool(valid_mask.any()):\n            return\n        buckets.index_put_((target_slots[valid_mask], h_idx[valid_mask], owner[valid_mask]), ships[valid_mask], accumulate=True)\n        self._mark_garrison_dirty(target_slots[valid_mask], h_idx[valid_mask] + 1)\n\n    def _normalize_garrison_slots(self, planet_slots: Tensor | None) -> tuple[Tensor, torch.Size]:\n        if planet_slots is None:\n            slots = torch.arange(self.P, dtype=torch.long, device=self.device)\n            return (slots, slots.shape)\n        raw = planet_slots.to(device=self.device, dtype=torch.long)\n        out_prefix = raw.shape\n        slots = raw.reshape(-1).clamp(0, max(self.P - 1, 0))\n        return (slots, out_prefix)\n\n    def _ensure_garrison_cache(self) -> None:\n        self._ensure_garrison_cache_impl()\n\n    def _ensure_garrison_cache_impl(self) -> None:\n        expected_owner = (self.P, int(self.movement_horizon) + 1)\n        expected_dirty = (self.P,)\n        if self.garrison_owner_cache is not None and self.garrison_ships_cache is not None and (self.garrison_pre_combat_owner_cache is not None) and (self.garrison_pre_combat_ships_cache is not None) and (self.garrison_dirty_from is not None) and (tuple(self.garrison_owner_cache.shape) == expected_owner) and (tuple(self.garrison_ships_cache.shape) == expected_owner) and (tuple(self.garrison_pre_combat_owner_cache.shape) == expected_owner) and (tuple(self.garrison_pre_combat_ships_cache.shape) == expected_owner) and (tuple(self.garrison_dirty_from.shape) == expected_dirty) and (self.garrison_owner_cache.device == self.device) and (self.garrison_ships_cache.device == self.device):\n            return\n        horizon = int(self.movement_horizon)\n        self.garrison_owner_cache = torch.full((self.P, horizon + 1), -1, dtype=torch.long, device=self.device)\n        self.garrison_ships_cache = torch.zeros(self.P, horizon + 1, dtype=self.dtype, device=self.device)\n        self.garrison_pre_combat_owner_cache = self.garrison_owner_cache.clone()\n        self.garrison_pre_combat_ships_cache = self.garrison_ships_cache.clone()\n        self.garrison_owner_cache[:, 0] = self.planet_owner\n        self.garrison_ships_cache[:, 0] = self.planet_ships\n        self.garrison_pre_combat_owner_cache[:, 0] = self.planet_owner\n        self.garrison_pre_combat_ships_cache[:, 0] = self.planet_ships\n        self.garrison_dirty_from = torch.zeros(self.P, dtype=torch.long, device=self.device)\n\n    def _refresh_garrison_projection(self, slots: Tensor, *, requested_horizon: int | None=None) -> None:\n        self._ensure_garrison_cache()\n        assert self.fleet_buckets is not None\n        assert self.garrison_owner_cache is not None\n        assert self.garrison_ships_cache is not None\n        assert self.garrison_dirty_from is not None\n        p_idx = torch.unique(slots.reshape(-1).clamp(min=0, max=max(self.P - 1, 0)))\n        if p_idx.numel() == 0:\n            return\n        dirty = self.garrison_dirty_from[p_idx]\n        horizon = int(self.movement_horizon if requested_horizon is None else max(0, min(int(requested_horizon), int(self.movement_horizon))))\n        needs_refresh = dirty <= horizon\n        if not bool(needs_refresh.any()):\n            return\n        p_idx = p_idx[needs_refresh]\n        owner = self.planet_owner[p_idx].clone()\n        ships = self.planet_ships[p_idx].clone()\n        self.garrison_owner_cache[p_idx, 0] = owner\n        self.garrison_ships_cache[p_idx, 0] = ships\n        assert self.garrison_pre_combat_owner_cache is not None\n        assert self.garrison_pre_combat_ships_cache is not None\n        self.garrison_pre_combat_owner_cache[p_idx, 0] = owner\n        self.garrison_pre_combat_ships_cache[p_idx, 0] = ships\n        prod = self.planet_prod[p_idx]\n        if horizon == 0:\n            self.garrison_dirty_from[p_idx] = horizon + 1\n            return\n        self._fill_garrison_trajectory(p_idx=p_idx, init_owner=owner, init_ships=ships, prod=prod, horizon=horizon)\n        self.garrison_dirty_from[p_idx] = horizon + 1\n\n    def _fill_garrison_trajectory(self, *, p_idx: Tensor, init_owner: Tensor, init_ships: Tensor, prod: Tensor, horizon: int) -> None:\n        assert self.fleet_buckets is not None\n        assert self.garrison_owner_cache is not None\n        assert self.garrison_ships_cache is not None\n        assert self.garrison_pre_combat_owner_cache is not None\n        assert self.garrison_pre_combat_ships_cache is not None\n        H = int(horizon)\n        N = int(p_idx.numel())\n        if N == 0 or H == 0:\n            return\n        alive_step = self.alive_by_step[:, p_idx].transpose(0, 1)\n        alive_before = alive_step[:, :H]\n        alive_now = alive_step[:, 1:]\n        arrivals = self.fleet_buckets[p_idx, :H, :]\n        has_any_arrival = (arrivals > 0.0).any(dim=-1).any(dim=-1)\n        alive_all_true = alive_step.all(dim=1)\n        simple_mask = ~has_any_arrival & alive_all_true\n        alive_step_full = alive_step\n        n_simple = int(simple_mask.sum().item())\n        n_complex = N - n_simple\n        if n_simple > 0:\n            simple_p = p_idx[simple_mask]\n            simple_owner = init_owner[simple_mask]\n            simple_ships = init_ships[simple_mask]\n            simple_prod = prod[simple_mask]\n            owner_alive_factor = (simple_owner >= 0).to(dtype=simple_ships.dtype)\n            k_range = torch.arange(1, H + 1, device=self.device, dtype=simple_ships.dtype)\n            ships_traj = simple_ships.unsqueeze(1) + simple_prod.unsqueeze(1) * owner_alive_factor.unsqueeze(1) * k_range.unsqueeze(0)\n            owner_traj = simple_owner.unsqueeze(1).expand(-1, H)\n            self.garrison_owner_cache[simple_p, 1:H + 1] = owner_traj\n            self.garrison_ships_cache[simple_p, 1:H + 1] = ships_traj\n            self.garrison_pre_combat_owner_cache[simple_p, 1:H + 1] = owner_traj\n            self.garrison_pre_combat_ships_cache[simple_p, 1:H + 1] = ships_traj\n        if n_complex == 0:\n            return\n        complex_mask = ~simple_mask\n        cp = p_idx[complex_mask]\n        arrivals_c = arrivals[complex_mask]\n        alive_before_c = alive_before[complex_mask]\n        alive_now_c = alive_now[complex_mask]\n        alive_step_c = alive_step_full[complex_mask]\n        state_owner = init_owner[complex_mask].clone()\n        state_ships = init_ships[complex_mask].clone()\n        prod_c = prod[complex_mask]\n        A = int(arrivals_c.shape[-1])\n        if A >= 2:\n            top2 = arrivals_c.topk(k=2, dim=-1)\n            top_ships_traj = top2.values[..., 0]\n            second_ships_traj = top2.values[..., 1]\n            top_owner_traj = top2.indices[..., 0].to(dtype=torch.long)\n        else:\n            top_ships_traj, top_owner_traj = arrivals_c.max(dim=-1)\n            second_ships_traj = torch.zeros_like(top_ships_traj)\n            top_owner_traj = top_owner_traj.to(dtype=torch.long)\n        tied = top_ships_traj == second_ships_traj\n        survivor_ships_traj = torch.where(tied, torch.zeros_like(top_ships_traj), (top_ships_traj - second_ships_traj).clamp(min=0.0))\n        survivor_owner_traj = top_owner_traj\n        zero_ships_scalar = torch.zeros((), dtype=state_ships.dtype, device=self.device)\n        neg_one_owner_scalar = torch.full((), -1, dtype=state_owner.dtype, device=self.device)\n        zero_prod_scalar = torch.zeros((), dtype=prod_c.dtype, device=self.device)\n        combat_event_per_step = (survivor_ships_traj > 0.0) & alive_now_c\n        alive_change_per_step = alive_before_c != alive_now_c\n        any_event_per_step = (combat_event_per_step | alive_change_per_step).any(dim=0)\n        arange_h = torch.arange(1, H + 1, device=self.device, dtype=torch.long)\n        k_last_tensor = torch.where(any_event_per_step, arange_h, torch.zeros_like(arange_h)).max()\n        k_last = int(k_last_tensor.item())\n        loop_iters = max(0, k_last)\n        tail_steps = H - loop_iters\n        if loop_iters > 0:\n            for k in range(1, loop_iters + 1):\n                a_before = alive_before_c[:, k - 1]\n                a_now = alive_now_c[:, k - 1]\n                s_owner = survivor_owner_traj[:, k - 1]\n                s_ships = survivor_ships_traj[:, k - 1]\n                produces = a_before & (state_owner >= 0)\n                state_ships = state_ships + torch.where(produces, prod_c, zero_prod_scalar)\n                pre_owner = torch.where(a_now, state_owner, neg_one_owner_scalar)\n                pre_ships = torch.where(a_now, state_ships, zero_ships_scalar)\n                self.garrison_pre_combat_owner_cache[cp, k] = pre_owner\n                self.garrison_pre_combat_ships_cache[cp, k] = pre_ships\n                has_combat = (s_ships > 0.0) & a_now\n                same = state_owner == s_owner\n                diff = state_ships - s_ships\n                attacker_wins = ~same & (diff < 0.0)\n                combat_ships = torch.where(same, state_ships + s_ships, diff.abs())\n                combat_owner = torch.where(attacker_wins, s_owner, state_owner)\n                state_ships = torch.where(has_combat, combat_ships, state_ships)\n                state_owner = torch.where(has_combat, combat_owner, state_owner)\n                state_owner = torch.where(a_now, state_owner, neg_one_owner_scalar)\n                state_ships = torch.where(a_now, state_ships, zero_ships_scalar)\n                self.garrison_owner_cache[cp, k] = state_owner\n                self.garrison_ships_cache[cp, k] = state_ships\n        if tail_steps > 0:\n            alive_at_k_last = alive_step_c[:, k_last]\n            state_owner = torch.where(alive_at_k_last, state_owner, neg_one_owner_scalar)\n            state_ships = torch.where(alive_at_k_last, state_ships, zero_ships_scalar)\n            owner_alive_factor = (state_owner >= 0).to(dtype=state_ships.dtype) * alive_at_k_last.to(dtype=state_ships.dtype)\n            dk_range = torch.arange(1, tail_steps + 1, device=self.device, dtype=state_ships.dtype)\n            ships_traj_tail = state_ships.unsqueeze(1) + prod_c.unsqueeze(1) * owner_alive_factor.unsqueeze(1) * dk_range.unsqueeze(0)\n            owner_traj_tail = state_owner.unsqueeze(1).expand(-1, tail_steps)\n            self.garrison_owner_cache[cp, k_last + 1:H + 1] = owner_traj_tail\n            self.garrison_ships_cache[cp, k_last + 1:H + 1] = ships_traj_tail\n            self.garrison_pre_combat_owner_cache[cp, k_last + 1:H + 1] = owner_traj_tail\n            self.garrison_pre_combat_ships_cache[cp, k_last + 1:H + 1] = ships_traj_tail\n\n    def _roll_garrison_projection(self) -> None:\n        if self.garrison_owner_cache is None or self.garrison_ships_cache is None or self.garrison_pre_combat_owner_cache is None or (self.garrison_pre_combat_ships_cache is None) or (self.garrison_dirty_from is None):\n            return\n        horizon = int(self.movement_horizon)\n        if horizon > 0:\n            self.garrison_owner_cache[:, :-1] = self.garrison_owner_cache[:, 1:].clone()\n            self.garrison_ships_cache[:, :-1] = self.garrison_ships_cache[:, 1:].clone()\n            self.garrison_pre_combat_owner_cache[:, :-1] = self.garrison_pre_combat_owner_cache[:, 1:].clone()\n            self.garrison_pre_combat_ships_cache[:, :-1] = self.garrison_pre_combat_ships_cache[:, 1:].clone()\n            self.garrison_dirty_from = (self.garrison_dirty_from - 1).clamp(min=0)\n            self.garrison_dirty_from = torch.minimum(self.garrison_dirty_from, torch.full_like(self.garrison_dirty_from, horizon))\n        else:\n            self.garrison_dirty_from[:] = 0\n\n    def _refresh_garrison_base(self, built: dict[str, Tensor]) -> None:\n        owner = built[\'owner\'].to(device=self.device, dtype=torch.long)\n        ships = built[\'ships\'].to(device=self.device, dtype=self.dtype)\n        prod = built[\'prod\'].to(device=self.device, dtype=self.dtype)\n        prod_changed = tuple(self.planet_prod.shape) != tuple(prod.shape) or self.planet_prod != prod\n        self.planet_owner = owner\n        self.planet_ships = ships\n        self.planet_prod = prod\n        if self.garrison_owner_cache is None or self.garrison_ships_cache is None or self.garrison_dirty_from is None:\n            return\n        base_changed = (self.garrison_owner_cache[:, 0] != owner) | (self.garrison_ships_cache[:, 0] != ships)\n        self.garrison_owner_cache[:, 0] = owner\n        self.garrison_ships_cache[:, 0] = ships\n        if self.garrison_pre_combat_owner_cache is not None:\n            self.garrison_pre_combat_owner_cache[:, 0] = owner\n        if self.garrison_pre_combat_ships_cache is not None:\n            self.garrison_pre_combat_ships_cache[:, 0] = ships\n        if bool(base_changed.any()):\n            self.garrison_dirty_from[base_changed] = 0\n        if isinstance(prod_changed, Tensor) and bool(prod_changed.any()):\n            self.garrison_dirty_from[prod_changed] = torch.minimum(self.garrison_dirty_from[prod_changed], torch.ones_like(self.garrison_dirty_from[prod_changed]))\n        elif not isinstance(prod_changed, Tensor) and prod_changed:\n            self.garrison_dirty_from[:] = torch.minimum(self.garrison_dirty_from, torch.ones_like(self.garrison_dirty_from))\n\n    def _mark_garrison_dirty(self, planet_idx: Tensor, start_step: Tensor | int) -> None:\n        if self.garrison_dirty_from is None:\n            return\n        p = planet_idx.to(device=self.device, dtype=torch.long)\n        if isinstance(start_step, int):\n            start = torch.full((), int(start_step), dtype=torch.long, device=self.device)\n        else:\n            start = start_step.to(device=self.device, dtype=torch.long)\n        p, start = torch.broadcast_tensors(p, start)\n        p = p.reshape(-1)\n        start = start.reshape(-1)\n        if p.numel() == 0:\n            return\n        start = start.clamp(min=0, max=int(self.movement_horizon))\n        valid = (p >= 0) & (p < self.P)\n        if not bool(valid.any()):\n            return\n        p = p[valid]\n        start = start[valid]\n        flat = self.garrison_dirty_from\n        unique_idx, inverse = torch.unique(p, return_inverse=True)\n        if unique_idx.numel() == p.numel():\n            flat[unique_idx] = torch.minimum(flat[unique_idx], start)\n            return\n        sentinel = int(self.movement_horizon) + 1\n        candidate = torch.full((unique_idx.shape[0],), sentinel, dtype=torch.long, device=self.device)\n        candidate.scatter_reduce_(0, inverse, start, reduce=\'amin\', include_self=True)\n        flat[unique_idx] = torch.minimum(flat[unique_idx], candidate)\n\n    def _mark_garrison_dirty_all(self, start_step: int) -> None:\n        if self.garrison_dirty_from is None:\n            return\n        self.garrison_dirty_from = torch.minimum(self.garrison_dirty_from, torch.full_like(self.garrison_dirty_from, int(start_step)))\n\n    def _init_fleet_tracking(self, obs_tensors: dict, *, reset_ledger: bool) -> None:\n        _ = reset_ledger\n        player_count = _resolve_player_count(obs_tensors, self.player_count)\n        self.player_count = int(player_count)\n        self.fleet_buckets = torch.zeros(self.P, int(self.movement_horizon), int(player_count), dtype=self.dtype, device=self.device)\n        step = obs_tensors[\'step\'].to(device=self.device, dtype=torch.long)\n        self.fleet_last_step = step.detach().clone()\n        M = max(1, int(self.max_tracked_fleets))\n        self.max_tracked_fleets = M\n        self.tracked_fleet_ids = torch.full((M,), -1, dtype=torch.long, device=self.device)\n        self.tracked_fleet_eta = torch.zeros((M,), dtype=torch.long, device=self.device)\n        self.tracked_fleet_target_slot = torch.full((M,), -1, dtype=torch.long, device=self.device)\n        self.tracked_fleet_owner = torch.zeros((M,), dtype=torch.long, device=self.device)\n        self.tracked_fleet_ships = torch.zeros((M,), dtype=self.dtype, device=self.device)\n        if self.garrison_dirty_from is not None:\n            self.garrison_dirty_from[:] = torch.minimum(self.garrison_dirty_from, torch.full_like(self.garrison_dirty_from, 1))\n\n    def _clear_tracked_rows(self) -> None:\n        if self.tracked_fleet_ids is None or self.tracked_fleet_eta is None or self.tracked_fleet_target_slot is None or (self.tracked_fleet_owner is None) or (self.tracked_fleet_ships is None):\n            return\n        self.tracked_fleet_ids[:] = -1\n        self.tracked_fleet_eta[:] = 0\n        self.tracked_fleet_target_slot[:] = -1\n        self.tracked_fleet_owner[:] = 0\n        self.tracked_fleet_ships[:] = 0.0\n\n    def _ledger_bulk_insert(self, fleet_ids: Tensor, eta_remaining: Tensor, target_slots: Tensor, owners: Tensor, ships: Tensor) -> None:\n        if fleet_ids.numel() == 0:\n            return\n        assert self.tracked_fleet_ids is not None\n        assert self.tracked_fleet_eta is not None\n        assert self.tracked_fleet_target_slot is not None\n        assert self.tracked_fleet_owner is not None\n        assert self.tracked_fleet_ships is not None\n        M = int(self.tracked_fleet_ids.shape[0])\n        fleet_ids = fleet_ids.to(device=self.device, dtype=torch.long).reshape(-1)\n        eta_remaining = eta_remaining.to(device=self.device, dtype=torch.long).reshape(-1)\n        target_slots = target_slots.to(device=self.device, dtype=torch.long).reshape(-1)\n        owners = owners.to(device=self.device, dtype=torch.long).reshape(-1)\n        ships = ships.to(device=self.device, dtype=self.dtype).reshape(-1)\n        valid_rows = fleet_ids >= 0\n        if not bool(valid_rows.any()):\n            return\n        fleet_ids = fleet_ids[valid_rows]\n        eta_remaining = eta_remaining[valid_rows]\n        target_slots = target_slots[valid_rows]\n        owners = owners[valid_rows]\n        ships = ships[valid_rows]\n        n = int(fleet_ids.numel())\n        empty_mask = self.tracked_fleet_ids == -1\n        empty_count = int(empty_mask.sum().item())\n        if n > empty_count:\n            occupied_count = M - empty_count\n            self._grow_ledger_capacity(occupied_count + n)\n            assert self.tracked_fleet_ids is not None\n            empty_mask = self.tracked_fleet_ids == -1\n        empty_slots = torch.nonzero(empty_mask, as_tuple=True)[0]\n        slot_idx = empty_slots[:n]\n        self.tracked_fleet_ids[slot_idx] = fleet_ids\n        self.tracked_fleet_eta[slot_idx] = eta_remaining\n        self.tracked_fleet_target_slot[slot_idx] = target_slots\n        self.tracked_fleet_owner[slot_idx] = owners\n        self.tracked_fleet_ships[slot_idx] = ships\n\n    def _grow_ledger_capacity(self, required_capacity: int) -> None:\n        if self.tracked_fleet_ids is None or self.tracked_fleet_eta is None or self.tracked_fleet_target_slot is None or (self.tracked_fleet_owner is None) or (self.tracked_fleet_ships is None):\n            return\n        old_capacity = int(self.tracked_fleet_ids.shape[0])\n        target_capacity = max(int(required_capacity), old_capacity)\n        if target_capacity <= old_capacity:\n            return\n        new_capacity = max(target_capacity, old_capacity * 2)\n        old_ids = self.tracked_fleet_ids\n        old_eta = self.tracked_fleet_eta\n        old_tgt = self.tracked_fleet_target_slot\n        old_owner = self.tracked_fleet_owner\n        old_ships = self.tracked_fleet_ships\n        self.tracked_fleet_ids = torch.full((new_capacity,), -1, dtype=torch.long, device=self.device)\n        self.tracked_fleet_eta = torch.zeros((new_capacity,), dtype=torch.long, device=self.device)\n        self.tracked_fleet_target_slot = torch.full((new_capacity,), -1, dtype=torch.long, device=self.device)\n        self.tracked_fleet_owner = torch.zeros((new_capacity,), dtype=torch.long, device=self.device)\n        self.tracked_fleet_ships = torch.zeros((new_capacity,), dtype=self.dtype, device=self.device)\n        self.tracked_fleet_ids[:old_capacity] = old_ids\n        self.tracked_fleet_eta[:old_capacity] = old_eta\n        self.tracked_fleet_target_slot[:old_capacity] = old_tgt\n        self.tracked_fleet_owner[:old_capacity] = old_owner\n        self.tracked_fleet_ships[:old_capacity] = old_ships\n\n    def _ledger_decrement_and_expire(self) -> None:\n        if self.tracked_fleet_ids is None or self.tracked_fleet_eta is None or self.tracked_fleet_target_slot is None or (self.tracked_fleet_owner is None) or (self.tracked_fleet_ships is None):\n            return\n        valid = self.tracked_fleet_ids >= 0\n        eta = torch.where(valid, self.tracked_fleet_eta - 1, self.tracked_fleet_eta)\n        expire = valid & (eta <= 0)\n        self.tracked_fleet_eta = eta\n        self.tracked_fleet_ids = torch.where(expire, torch.full_like(self.tracked_fleet_ids, -1), self.tracked_fleet_ids)\n        self.tracked_fleet_eta = torch.where(expire, torch.zeros_like(self.tracked_fleet_eta), self.tracked_fleet_eta)\n        self.tracked_fleet_target_slot = torch.where(expire, torch.full_like(self.tracked_fleet_target_slot, -1), self.tracked_fleet_target_slot)\n        self.tracked_fleet_owner = torch.where(expire, torch.zeros_like(self.tracked_fleet_owner), self.tracked_fleet_owner)\n        self.tracked_fleet_ships = torch.where(expire, torch.zeros_like(self.tracked_fleet_ships), self.tracked_fleet_ships)\n\n    def _roll_fleet_buckets_phase1(self, current_step: Tensor) -> None:\n        if self.fleet_buckets is None or self.fleet_last_step is None:\n            return\n        step = current_step.to(device=self.device, dtype=torch.long)\n        delta = step - self.fleet_last_step.to(device=self.device, dtype=torch.long)\n        horizon = int(self.movement_horizon)\n        reset = bool((delta < 0) | (step <= 0))\n        if reset:\n            self.fleet_buckets[:] = 0.0\n            self._clear_tracked_rows()\n            self._mark_garrison_dirty_all(1)\n        rolled_once = not reset and bool(delta == 1)\n        if rolled_once and horizon > 0:\n            self.fleet_buckets[:, :-1, :] = self.fleet_buckets[:, 1:, :].clone()\n            self.fleet_buckets[:, -1, :] = 0.0\n            self._ledger_decrement_and_expire()\n            self._mark_garrison_dirty_all(1)\n        delta_bad = not reset and bool(delta > 1)\n        if delta_bad:\n            self._reset_fleet_tracking()\n        self.fleet_last_step = step.detach().clone()\n\n    def _reset_fleet_tracking(self) -> None:\n        if self.fleet_buckets is None:\n            return\n        self.fleet_buckets[:] = 0.0\n        self._clear_tracked_rows()\n        self._mark_garrison_dirty_all(1)\n\n    def _ingest_obs_fleets(self, obs_tensors: dict) -> None:\n        if self.fleet_buckets is None or self.tracked_fleet_ids is None or int(self.movement_horizon) <= 0:\n            return\n        fleets = obs_tensors[\'fleets\'].to(device=self.device, dtype=self.dtype)\n        fleet_ids = fleets[..., 0].to(dtype=torch.long)\n        alive = fleet_ids >= 0\n        tracked = (fleet_ids.unsqueeze(1) == self.tracked_fleet_ids.unsqueeze(0)).any(dim=1)\n        process_mask = alive & ~tracked\n        n_alive = int(alive.sum().item())\n        n_tracked = int((alive & tracked).sum().item())\n        n_to_process = n_alive - n_tracked\n        if n_to_process == 0:\n            return\n        fleet_slot = torch.where(process_mask)[0]\n        proc_ids = fleet_ids[fleet_slot]\n        estimate = _estimate_new_fleet_arrivals(movement=self, obs_fleets=fleets, fleet_slot=fleet_slot)\n        valid_owner = (estimate[\'owner\'] >= 0) & (estimate[\'owner\'] < int(self.fleet_buckets.shape[2]))\n        valid_hit = estimate[\'has_hit\'] & valid_owner\n        if not bool(valid_hit.any()):\n            return\n        buckets = self._require_fleet_buckets()\n        buckets.index_put_((estimate[\'target_slot\'][valid_hit], estimate[\'eta_index\'][valid_hit], estimate[\'owner\'][valid_hit]), estimate[\'ships\'][valid_hit], accumulate=True)\n        self._mark_garrison_dirty(estimate[\'target_slot\'][valid_hit], estimate[\'eta_index\'][valid_hit] + 1)\n        eta_remaining = estimate[\'eta_index\'][valid_hit].to(dtype=torch.long) + 1\n        self._ledger_bulk_insert(proc_ids[valid_hit], eta_remaining, estimate[\'target_slot\'][valid_hit], estimate[\'owner\'][valid_hit], estimate[\'ships\'][valid_hit])\n\n    def _reconcile_obs_fleets(self, obs_tensors: dict) -> None:\n        if self.fleet_buckets is None or self.tracked_fleet_ids is None or self.tracked_fleet_eta is None or (self.tracked_fleet_target_slot is None) or (self.tracked_fleet_owner is None) or (self.tracked_fleet_ships is None) or (int(self.movement_horizon) <= 0):\n            return\n        obs_ids = obs_tensors[\'fleets\'][..., 0].to(device=self.device, dtype=torch.long)\n        in_flight = (self.tracked_fleet_ids >= 0) & (self.tracked_fleet_eta > 0)\n        if not bool(in_flight.any()):\n            return\n        match = (self.tracked_fleet_ids.unsqueeze(1) == obs_ids.unsqueeze(0)).any(dim=1)\n        phantom = in_flight & ~match\n        if not bool(phantom.any()):\n            return\n        m_idx = torch.where(phantom)[0]\n        h_idx = (self.tracked_fleet_eta[m_idx] - 1).clamp(min=0)\n        P = int(self.fleet_buckets.shape[0])\n        H = int(self.fleet_buckets.shape[1])\n        A = int(self.fleet_buckets.shape[2])\n        in_horizon = h_idx < H\n        if not bool(in_horizon.any()):\n            self.tracked_fleet_ids[m_idx] = -1\n            self.tracked_fleet_eta[m_idx] = 0\n            self.tracked_fleet_target_slot[m_idx] = -1\n            self.tracked_fleet_owner[m_idx] = 0\n            self.tracked_fleet_ships[m_idx] = 0.0\n            return\n        m_sel = m_idx[in_horizon]\n        h_sel = h_idx[in_horizon]\n        slots = self.tracked_fleet_target_slot[m_sel].clamp(min=0, max=max(P - 1, 0))\n        owners = self.tracked_fleet_owner[m_sel].clamp(min=0, max=max(A - 1, 0))\n        ships = self.tracked_fleet_ships[m_sel]\n        self.fleet_buckets.index_put_((slots, h_sel, owners), -ships, accumulate=True)\n        self._mark_garrison_dirty(slots, h_sel + 1)\n        self.tracked_fleet_ids[m_idx] = -1\n        self.tracked_fleet_eta[m_idx] = 0\n        self.tracked_fleet_target_slot[m_idx] = -1\n        self.tracked_fleet_owner[m_idx] = 0\n        self.tracked_fleet_ships[m_idx] = 0.0\n\n    def _require_fleet_buckets(self) -> Tensor:\n        if self.fleet_buckets is None:\n            raise RuntimeError(\'PlanetMovement fleet tracking is not enabled\')\n        return self.fleet_buckets\n\n    def _k_index(self, k: int) -> int:\n        if k < 0 or k > int(self.movement_horizon):\n            raise IndexError(f\'k must be in [0, {self.movement_horizon}], got {k}\')\n        return int(k)\n\n    def _copy_from(self, other: \'PlanetMovement\') -> None:\n        self.x = other.x\n        self.y = other.y\n        self.alive_by_step = other.alive_by_step\n        self.planet_ids = other.planet_ids\n        self.radii = other.radii\n        self.planet_owner = other.planet_owner\n        self.planet_ships = other.planet_ships\n        self.planet_prod = other.planet_prod\n        self.base_step = other.base_step\n        self.comet_planet_ids = other.comet_planet_ids\n        self.comet_path_index = other.comet_path_index\n        self.movement_horizon = other.movement_horizon\n        self.drift_epsilon = other.drift_epsilon\n        self.track_fleets = other.track_fleets\n        self.player_count = other.player_count\n        self.max_tracked_fleets = other.max_tracked_fleets\n        self.fleet_buckets = other.fleet_buckets\n        self.fleet_last_step = other.fleet_last_step\n        self.tracked_fleet_ids = other.tracked_fleet_ids\n        self.tracked_fleet_eta = other.tracked_fleet_eta\n        self.tracked_fleet_target_slot = other.tracked_fleet_target_slot\n        self.tracked_fleet_owner = other.tracked_fleet_owner\n        self.tracked_fleet_ships = other.tracked_fleet_ships\n        self.garrison_owner_cache = other.garrison_owner_cache\n        self.garrison_ships_cache = other.garrison_ships_cache\n        self.garrison_dirty_from = other.garrison_dirty_from\n\ndef _resolve_player_count(obs_tensors: dict, player_count: int | None) -> int:\n    if player_count is not None:\n        if int(player_count) not in (2, 4):\n            raise ValueError(\'player_count must be 2 or 4\')\n        return int(player_count)\n    metadata_count = obs_tensors.get(\'player_count\')\n    if metadata_count is not None:\n        count = int(metadata_count.flatten()[0].item()) if isinstance(metadata_count, Tensor) else int(metadata_count)\n        if count not in (2, 4):\n            raise ValueError(\'player_count metadata must be 2 or 4\')\n        return count\n    planets = obs_tensors[\'planets\']\n    fleets = obs_tensors[\'fleets\']\n    planet_alive = planets[..., 0] >= 0\n    fleet_alive = fleets[..., 0] >= 0\n    owner_values = []\n    if bool(planet_alive.any()):\n        owner_values.append(planets[..., 1][planet_alive].to(dtype=torch.long))\n    if bool(fleet_alive.any()):\n        owner_values.append(fleets[..., 1][fleet_alive].to(dtype=torch.long))\n    if not owner_values:\n        return 2\n    owners = torch.cat(owner_values)\n    owners = owners[owners >= 0]\n    if owners.numel() == 0:\n        return 2\n    return 4 if int(owners.max().item()) >= 2 else 2\n\ndef _estimate_new_fleet_arrivals(*, movement: PlanetMovement, obs_fleets: Tensor, fleet_slot: Tensor) -> dict[str, Tensor]:\n    N = int(fleet_slot.numel())\n    device = movement.device\n    dtype = movement.dtype\n    H = int(movement.movement_horizon)\n    P = int(movement.P)\n    if N == 0:\n        empty_long = torch.empty(0, dtype=torch.long, device=device)\n        empty_bool = torch.empty(0, dtype=torch.bool, device=device)\n        empty_float = torch.empty(0, dtype=dtype, device=device)\n        return {\'owner\': empty_long, \'target_slot\': empty_long, \'eta_index\': empty_long, \'has_hit\': empty_bool, \'ships\': empty_float}\n    rows = obs_fleets[fleet_slot]\n    owner = rows[:, 1].to(dtype=torch.long)\n    x = rows[:, 2].to(dtype=dtype)\n    y = rows[:, 3].to(dtype=dtype)\n    angle = rows[:, 4].to(dtype=dtype)\n    ships = rows[:, 6].to(dtype=dtype)\n    times = torch.arange(1, H + 1, dtype=dtype, device=device).view(1, H)\n    speed = fleet_speed(ships).clamp(min=1e-06)\n    ux = torch.cos(angle)\n    uy = torch.sin(angle)\n    old_x = x.view(N, 1) + ux.view(N, 1) * speed.view(N, 1) * (times - 1.0)\n    old_y = y.view(N, 1) + uy.view(N, 1) * speed.view(N, 1) * (times - 1.0)\n    new_x = x.view(N, 1) + ux.view(N, 1) * speed.view(N, 1) * times\n    new_y = y.view(N, 1) + uy.view(N, 1) * speed.view(N, 1) * times\n    in_bounds = (new_x >= 0.0) & (new_x <= BOARD_SIZE) & (new_y >= 0.0) & (new_y <= BOARD_SIZE)\n    sun_dist_sq = _point_to_segment_distance_sq(torch.full_like(new_x, CENTER), torch.full_like(new_y, CENTER), old_x, old_y, new_x, new_y)\n    env_kill = ~in_bounds | (sun_dist_sq < SUN_RADIUS * SUN_RADIUS)\n    planet_x = movement.x.unsqueeze(0).expand(N, H + 1, P)\n    planet_y = movement.y.unsqueeze(0).expand(N, H + 1, P)\n    planet_alive = movement.alive_by_step.unsqueeze(0).expand(N, H + 1, P)\n    radii = movement.radii.unsqueeze(0).expand(N, P).to(dtype=dtype)\n    old_px = planet_x[:, :-1, :]\n    old_py = planet_y[:, :-1, :]\n    new_px = planet_x[:, 1:, :]\n    new_py = planet_y[:, 1:, :]\n    alive_old = planet_alive[:, :-1, :]\n    check_collision = alive_old & (old_px >= 0.0) & (old_py >= 0.0)\n    swept_collides = _swept_pair_hit_mask_mv(old_x.unsqueeze(2), old_y.unsqueeze(2), new_x.unsqueeze(2), new_y.unsqueeze(2), old_px, old_py, new_px, new_py, radii.view(N, 1, P)) & check_collision\n    step_raw_has_hit = swept_collides.any(dim=2)\n    hit_rank = swept_collides.to(torch.int32).cumsum(dim=2)\n    first_hit = swept_collides & (hit_rank == 1)\n    step_hit_slot = first_hit.to(torch.int64).argmax(dim=2)\n    step_hit_slot = step_hit_slot.where(step_raw_has_hit, torch.full_like(step_hit_slot, -1))\n    kill_event = step_raw_has_hit | env_kill\n    cum_kill_inclusive = kill_event.cummax(dim=1).values\n    alive_before_t = torch.cat([torch.ones((N, 1), dtype=torch.bool, device=device), ~cum_kill_inclusive[:, :-1]], dim=1)\n    step_has_hit = step_raw_has_hit & alive_before_t\n    has_hit = step_has_hit.any(dim=1)\n    eta_index = step_has_hit.to(torch.int64).argmax(dim=1)\n    target_slot = step_hit_slot.gather(1, eta_index.view(N, 1)).squeeze(1).clamp(min=0, max=max(P - 1, 0))\n    return {\'owner\': owner, \'target_slot\': target_slot, \'eta_index\': eta_index, \'has_hit\': has_hit, \'ships\': ships}\n\ndef _point_to_segment_distance_sq(px: Tensor, py: Tensor, x1: Tensor, y1: Tensor, x2: Tensor, y2: Tensor) -> Tensor:\n    dx = x2 - x1\n    dy = y2 - y1\n    denom = dx * dx + dy * dy\n    safe_denom = torch.where(denom > 0, denom, torch.ones_like(denom))\n    t = ((px - x1) * dx + (py - y1) * dy) / safe_denom\n    t = t.clamp(0.0, 1.0)\n    proj_x = x1 + t * dx\n    proj_y = y1 + t * dy\n    return (px - proj_x) ** 2 + (py - proj_y) ** 2\n\ndef _swept_pair_hit_mask_mv(ax: Tensor, ay: Tensor, bx: Tensor, by: Tensor, p0x: Tensor, p0y: Tensor, p1x: Tensor, p1y: Tensor, r: Tensor) -> Tensor:\n    d0x = ax - p0x\n    d0y = ay - p0y\n    dvx = bx - ax - (p1x - p0x)\n    dvy = by - ay - (p1y - p0y)\n    a = dvx * dvx + dvy * dvy\n    b = 2.0 * (d0x * dvx + d0y * dvy)\n    c = d0x * d0x + d0y * d0y - r * r\n    near_static = a < 1e-12\n    c_hit = c <= 0.0\n    disc = b * b - 4.0 * a * c\n    has_root = disc >= 0.0\n    safe_a = torch.where(near_static, torch.ones_like(a), a)\n    sq = torch.sqrt(torch.clamp(disc, min=0.0))\n    t1 = (-b - sq) / (2.0 * safe_a)\n    t2 = (-b + sq) / (2.0 * safe_a)\n    quad_hit = has_root & (t2 >= 0.0) & (t1 <= 1.0)\n    return torch.where(near_static, c_hit, quad_hit)\n\ndef _build_future_from_obs(obs_tensors: dict, movement_horizon: int, *, offsets: Tensor | None=None) -> dict[str, Tensor]:\n    obs = parse_obs(obs_tensors)\n    H = int(movement_horizon)\n    planets = obs_tensors[\'planets\']\n    dtype = planets.dtype\n    device = planets.device\n    P, _ = planets.shape\n    planet_ids = planets[..., 0].long()\n    radii = planets[..., 4].to(dtype=dtype)\n    owner = planets[..., 1].to(device=device, dtype=torch.long)\n    owner = torch.where(obs.alive, owner, torch.full_like(owner, -1))\n    ships = planets[..., 5].to(device=device, dtype=dtype)\n    prod = planets[..., 6].to(device=device, dtype=dtype)\n    step = obs.step.to(device=device, dtype=torch.long)\n    if offsets is None:\n        offsets_long = torch.arange(H + 1, dtype=torch.long, device=device)\n    else:\n        offsets_long = offsets.to(device=device, dtype=torch.long).reshape(-1)\n    M = int(offsets_long.shape[0])\n    offsets_d = offsets_long.to(dtype=dtype)\n    future_phase = orbit_phase_index_from_obs_step(obs.step.to(dtype=dtype) + offsets_d).to(device=device, dtype=dtype)\n    angle = obs.orb_a0.to(dtype=dtype).view(1, P) + obs.angvel.to(dtype=dtype) * future_phase.view(M, 1)\n    orb_x = CENTER + obs.orb_r.to(dtype=dtype).view(1, P) * torch.cos(angle)\n    orb_y = CENTER + obs.orb_r.to(dtype=dtype).view(1, P) * torch.sin(angle)\n    is_orbiting = obs.is_orbiting.view(1, P)\n    x = torch.where(is_orbiting, orb_x, obs.x.to(dtype=dtype).view(1, P).expand(M, P)).contiguous()\n    y = torch.where(is_orbiting, orb_y, obs.y.to(dtype=dtype).view(1, P).expand(M, P)).contiguous()\n    alive_by_step = obs.alive.view(1, P).expand(M, P).clone()\n    comet_planet_ids, comet_path_index = _comet_metadata(obs_tensors, device)\n    x, y, alive_by_step = _apply_comet_paths(x=x, y=y, alive_by_step=alive_by_step, planet_ids=planet_ids, comet_planet_ids=comet_planet_ids, comet_path_index=comet_path_index, obs_tensors=obs_tensors, offsets=offsets_long)\n    zero_idx = (offsets_long == 0).nonzero(as_tuple=True)[0]\n    if int(zero_idx.numel()) > 0:\n        x[zero_idx, :] = obs.x.to(dtype=dtype).view(1, P)\n        y[zero_idx, :] = obs.y.to(dtype=dtype).view(1, P)\n        alive_by_step[zero_idx, :] = obs.alive.view(1, P)\n    return {\'x\': x, \'y\': y, \'alive_by_step\': alive_by_step, \'planet_ids\': planet_ids, \'radii\': radii, \'owner\': owner, \'ships\': ships, \'prod\': prod, \'step\': step, \'comet_planet_ids\': comet_planet_ids, \'comet_path_index\': comet_path_index, \'_offsets\': offsets_long}\n\ndef _comet_metadata(obs_tensors: dict, device: torch.device) -> tuple[Tensor, Tensor]:\n    comets = obs_tensors.get(\'comets\') or {}\n    comet_ids = comets.get(\'planet_ids\')\n    if comet_ids is None:\n        flat_ids = obs_tensors.get(\'comet_planet_ids\')\n        if flat_ids is None:\n            flat_ids = torch.full((0,), -1, dtype=torch.long, device=device)\n        else:\n            flat_ids = flat_ids.to(device=device, dtype=torch.long)\n        path_index = torch.full((0,), -1, dtype=torch.long, device=device)\n        return (flat_ids, path_index)\n    comet_ids = comet_ids.to(device=device, dtype=torch.long)\n    flat_ids = comet_ids.reshape(-1)\n    path_index = comets.get(\'path_index\')\n    if path_index is None:\n        path_index = torch.full((comet_ids.shape[0],), -1, dtype=torch.long, device=device)\n    else:\n        path_index = path_index.to(device=device, dtype=torch.long)\n    return (flat_ids, path_index)\n\ndef _apply_comet_paths(*, x: Tensor, y: Tensor, alive_by_step: Tensor, planet_ids: Tensor, comet_planet_ids: Tensor, comet_path_index: Tensor, obs_tensors: dict, offsets: Tensor) -> tuple[Tensor, Tensor, Tensor]:\n    comets = obs_tensors.get(\'comets\') or {}\n    paths = comets.get(\'paths\')\n    ids_grid = comets.get(\'planet_ids\')\n    if paths is None or ids_grid is None or comet_planet_ids.numel() == 0:\n        return (x, y, alive_by_step)\n    M, P = x.shape\n    paths = paths.to(device=x.device, dtype=x.dtype)\n    ids_grid = ids_grid.to(device=x.device, dtype=torch.long)\n    E = int(ids_grid.shape[0])\n    C = int(ids_grid.shape[1])\n    T = int(paths.shape[2])\n    if E == 0 or C == 0 or T == 0:\n        return (x, y, alive_by_step)\n    flat_ids = ids_grid.reshape(E * C)\n    matches = (planet_ids.unsqueeze(1) == flat_ids.unsqueeze(0)) & (flat_ids.unsqueeze(0) >= 0)\n    is_comet = matches.any(dim=1)\n    flat_slot = matches.to(torch.float32).argmax(dim=1).long()\n    flat_paths_x = paths[..., 0].reshape(E * C, T)\n    flat_paths_y = paths[..., 1].reshape(E * C, T)\n    path_x_by_slot = flat_paths_x[flat_slot]\n    path_y_by_slot = flat_paths_y[flat_slot]\n    finite = torch.isfinite(flat_paths_x)\n    path_len = finite.sum(dim=1).to(dtype=torch.long)\n    len_by_slot = path_len[flat_slot]\n    group_idx = (flat_slot // C).clamp(min=0, max=max(E - 1, 0))\n    path_idx_by_slot = comet_path_index[group_idx]\n    offsets_v = offsets.to(device=x.device, dtype=torch.long).view(M, 1)\n    future_idx = path_idx_by_slot.view(1, P) + offsets_v\n    valid_future = is_comet.view(1, P) & (future_idx >= 0) & (future_idx < len_by_slot.view(1, P))\n    idx_clamped = future_idx.clamp(min=0, max=max(T - 1, 0))\n    p_index = torch.arange(P, device=x.device).view(1, P).expand(M, P)\n    comet_x = path_x_by_slot[p_index, idx_clamped]\n    comet_y = path_y_by_slot[p_index, idx_clamped]\n    x = torch.where(valid_future, comet_x, x)\n    y = torch.where(valid_future, comet_y, y)\n    alive_by_step = torch.where(is_comet.view(1, P), valid_future, alive_by_step)\n    return (x, y, alive_by_step)\n\ndef _same_2d(a: Tensor, b: Tensor) -> bool:\n    if a.shape != b.shape:\n        return False\n    if a.numel() == 0:\n        return True\n    return bool((a == b.to(device=a.device, dtype=a.dtype)).all())\n\ndef _position_matches(pred_x: Tensor, pred_y: Tensor, cur_x: Tensor, cur_y: Tensor, alive: Tensor, epsilon: float) -> bool:\n    diff = torch.maximum((pred_x - cur_x).abs(), (pred_y - cur_y).abs())\n    diff = torch.where(alive, diff, torch.zeros_like(diff))\n    return bool((diff <= float(epsilon)).all())\n\'What-if-I-launch flow projection over a :class:`PlanetGarrisonStatus`.\\n\\n\\n\\n``PlanetGarrisonStatus`` is a per-planet ledger of projected owner / ships over a\\n\\nfuture horizon, computed from the fleets we currently know about, assuming we do\\n\\nnothing. :func:`sparse_launch_flow_delta` answers the forward-looking question an\\n\\nagent faces — *"if I launch these ships, how does each player\\\'s net ship flow\\n\\n(production minus combat losses) change?"* — by recomputing the production→combat\\n\\nrecurrence only for the planets a launch touches and diffing against the baseline.\\n\\n\\n\\nA launch is two-sided: it debits the source planet\\\'s garrison (ships leave now,\\n\\nbefore that turn\\\'s production) and credits the target\\\'s arrival at step ``k``.\\n\\n\\n\\nTwo leading axes are supported (single game):\\n\\n\\n\\n- ``C`` — candidates: the different launches / launch-sets being scored.\\n\\n- ``L`` — launches within a candidate: a candidate *is* a set of launches; ``L``\\n\\n  is summed away during aggregation and is not an output axis.\\n\\n\\n\\nPass launches as ``[L]`` (no candidate axis) or ``[C, L]``.\\n\\n\'\nfrom dataclasses import dataclass\nimport torch\nfrom torch import Tensor\n\n@dataclass(frozen=True)\nclass LaunchSet:\n    source_slots: Tensor\n    target_slots: Tensor\n    ships: Tensor\n    eta: Tensor\n    owner: Tensor\n    valid: Tensor\n\n    @property\n    def has_candidate_axis(self) -> bool:\n        return self.source_slots.dim() >= 2\n\ndef _per_step_survivor(arrivals: Tensor) -> tuple[Tensor, Tensor]:\n    A = int(arrivals.shape[-1])\n    if A >= 2:\n        top2 = arrivals.topk(k=2, dim=-1)\n        top_ships = top2.values[..., 0]\n        second_ships = top2.values[..., 1]\n        top_owner = top2.indices[..., 0].to(dtype=torch.long)\n    else:\n        top_ships, top_owner = arrivals.max(dim=-1)\n        second_ships = torch.zeros_like(top_ships)\n        top_owner = top_owner.to(dtype=torch.long)\n    tied = top_ships == second_ships\n    survivor_ships = torch.where(tied, torch.zeros_like(top_ships), (top_ships - second_ships).clamp(min=0.0))\n    return (top_owner, survivor_ships)\n\ndef _run_exact_recurrence(*, init_owner: Tensor, init_ships: Tensor, prod: Tensor, alive: Tensor, arrivals: Tensor) -> tuple[Tensor, Tensor, Tensor, Tensor]:\n    N, P = init_owner.shape\n    H = int(arrivals.shape[2])\n    device = init_ships.device\n    owner_out = torch.empty(N, P, H + 1, dtype=init_owner.dtype, device=device)\n    ships_out = torch.empty(N, P, H + 1, dtype=init_ships.dtype, device=device)\n    pre_owner_out = torch.empty_like(owner_out)\n    pre_ships_out = torch.empty_like(ships_out)\n    owner_out[..., 0] = init_owner\n    ships_out[..., 0] = init_ships\n    pre_owner_out[..., 0] = init_owner\n    pre_ships_out[..., 0] = init_ships\n    survivor_owner, survivor_ships = _per_step_survivor(arrivals)\n    state_owner = init_owner.clone()\n    state_ships = init_ships.clone()\n    zero_ships = torch.zeros((), dtype=state_ships.dtype, device=device)\n    neg_one = torch.full((), -1, dtype=state_owner.dtype, device=device)\n    zero_prod = torch.zeros((), dtype=prod.dtype, device=device)\n    for k in range(1, H + 1):\n        a_before = alive[..., k - 1]\n        a_now = alive[..., k]\n        s_owner = survivor_owner[..., k - 1]\n        s_ships = survivor_ships[..., k - 1]\n        produces = a_before & (state_owner >= 0)\n        state_ships = state_ships + torch.where(produces, prod, zero_prod)\n        pre_owner_out[..., k] = torch.where(a_now, state_owner, neg_one)\n        pre_ships_out[..., k] = torch.where(a_now, state_ships, zero_ships)\n        has_combat = (s_ships > 0.0) & a_now\n        same = state_owner == s_owner\n        diff = state_ships - s_ships\n        attacker_wins = ~same & (diff < 0.0)\n        combat_ships = torch.where(same, state_ships + s_ships, diff.abs())\n        combat_owner = torch.where(attacker_wins, s_owner, state_owner)\n        state_ships = torch.where(has_combat, combat_ships, state_ships)\n        state_owner = torch.where(has_combat, combat_owner, state_owner)\n        state_owner = torch.where(a_now, state_owner, neg_one)\n        state_ships = torch.where(a_now, state_ships, zero_ships)\n        owner_out[..., k] = state_owner\n        ships_out[..., k] = state_ships\n    return (owner_out, ships_out, pre_owner_out, pre_ships_out)\n\ndef _validate_inputs(status: PlanetGarrisonStatus, prod: Tensor, alive_by_step: Tensor, player_count: int) -> tuple[int, int, int, int]:\n    if status.arrivals_by_owner is None:\n        raise ValueError(\'garrison status must carry arrivals_by_owner (build it from a PlanetMovement with track_fleets=True)\')\n    if status.pre_combat_owner is None or status.pre_combat_ships is None:\n        raise ValueError(\'garrison status must carry pre_combat_owner/ships\')\n    if status.owner.dim() != 2:\n        raise ValueError(f\'expected a full-board status with owner shaped [P, H+1]; got {tuple(status.owner.shape)}\')\n    P, H1 = status.owner.shape\n    H = H1 - 1\n    A = int(status.arrivals_by_owner.shape[-1])\n    if int(player_count) != A:\n        raise ValueError(f\'player_count={player_count} disagrees with arrivals owner axis A={A}\')\n    if tuple(prod.shape) != (P,):\n        raise ValueError(f\'prod must be [P]=({P},); got {tuple(prod.shape)}\')\n    if tuple(alive_by_step.shape) != (H1, P):\n        raise ValueError(f\'alive_by_step must be [H+1, P]=({H1}, {P}); got {tuple(alive_by_step.shape)}\')\n    return (P, H, A)\n\n@dataclass(frozen=True)\nclass GarrisonFlowDiff:\n    player_id: int\n    ships_produced_current: Tensor\n    ships_produced_hypothetical: Tensor\n    ships_produced_delta: Tensor\n    ships_lost_combat_current: Tensor\n    ships_lost_combat_hypothetical: Tensor\n    ships_lost_combat_delta: Tensor\n    net_ship_delta: Tensor\n\n    @property\n    def player_count(self) -> int:\n        return int(self.ships_produced_delta.shape[-1])\n\ndef _flow_terms_per_planet(*, owner: Tensor, pre_owner: Tensor, pre_ships: Tensor, arr_full: Tensor, prod: Tensor, alive_pmajor: Tensor) -> tuple[Tensor, Tensor]:\n    A = int(arr_full.shape[-1])\n    H = int(owner.shape[-1]) - 1\n    fdtype = pre_ships.dtype\n    a_idx = torch.arange(A, device=owner.device)\n    producing_owner = owner[..., :H]\n    amount = prod.unsqueeze(-1) * alive_pmajor[..., :H].to(fdtype)\n    prod_owner_oh = producing_owner.unsqueeze(-1) == a_idx\n    produced = (amount.unsqueeze(-1) * prod_owner_oh.to(fdtype)).sum(dim=-2)\n    arr_k = arr_full[..., 1:, :]\n    survivor_owner, survivor_ships = _per_step_survivor(arr_k)\n    survived = torch.where(a_idx == survivor_owner.unsqueeze(-1), survivor_ships.unsqueeze(-1), torch.zeros_like(survivor_ships).unsqueeze(-1))\n    attacker_lost = (arr_k - survived).clamp(min=0.0)\n    prior_owner = pre_owner[..., 1:]\n    prior_ships = pre_ships[..., 1:]\n    fights_garrison = (survivor_ships > 0.0) & (survivor_owner != prior_owner) & (survivor_owner >= 0)\n    garrison_loss = torch.where(fights_garrison, torch.minimum(prior_ships, survivor_ships), torch.zeros_like(prior_ships))\n    is_survivor = (a_idx == survivor_owner.unsqueeze(-1)) & fights_garrison.unsqueeze(-1)\n    is_prior = (a_idx == prior_owner.unsqueeze(-1)) & fights_garrison.unsqueeze(-1) & (prior_owner >= 0).unsqueeze(-1)\n    garrison_lost = garrison_loss.unsqueeze(-1) * (is_survivor.to(fdtype) + is_prior.to(fdtype))\n    combat_lost = (attacker_lost + garrison_lost).sum(dim=-2)\n    return (produced, combat_lost)\n\ndef _normalize_launches_bcl(launches: LaunchSet) -> tuple[Tensor, ...]:\n    fields = (launches.source_slots, launches.target_slots, launches.ships, launches.eta, launches.owner, launches.valid)\n    if launches.has_candidate_axis:\n        return fields\n    return tuple((f.unsqueeze(0) for f in fields))\n\ndef sparse_launch_flow_delta(status: PlanetGarrisonStatus, *, prod: Tensor, alive_by_step: Tensor, player_count: int, launches: LaunchSet, player_id: int=0) -> GarrisonFlowDiff:\n    P, H, A = _validate_inputs(status, prod, alive_by_step, player_count)\n    device = status.owner.device\n    fdtype = status.ships.dtype\n    assert status.pre_combat_owner is not None and status.pre_combat_ships is not None\n    assert status.arrivals_by_owner is not None\n    src, tgt, ships, eta, owner, valid = _normalize_launches_bcl(launches)\n    C = int(src.shape[0])\n    L = int(src.shape[-1])\n    src = src.to(device=device, dtype=torch.long)\n    tgt = tgt.to(device=device, dtype=torch.long)\n    ships = ships.to(device=device, dtype=fdtype)\n    owner = owner.to(device=device, dtype=torch.long)\n    valid = valid.to(device=device, dtype=torch.bool)\n    h_idx = torch.ceil(eta.to(device=device, dtype=fdtype)).to(torch.long) - 1\n    valid_t = valid & (ships > 0) & (tgt >= 0) & (tgt < P) & (owner >= 0) & (owner < A) & (h_idx >= 0) & (h_idx < H)\n    valid_s = valid & (ships > 0) & (src >= 0) & (src < P)\n    src_safe = src.clamp(0, max(P - 1, 0))\n    tgt_safe = tgt.clamp(0, max(P - 1, 0))\n    affected = torch.zeros(C, P, dtype=fdtype, device=device)\n    affected.scatter_add_(1, src_safe, valid_s.to(fdtype))\n    affected.scatter_add_(1, tgt_safe, valid_t.to(fdtype))\n    affected_mask = affected > 0\n    base_prod_pp, base_combat_pp = _flow_terms_per_planet(owner=status.owner, pre_owner=status.pre_combat_owner, pre_ships=status.pre_combat_ships, arr_full=status.arrivals_by_owner, prod=prod, alive_pmajor=alive_by_step.permute(1, 0))\n    base_prod = base_prod_pp.sum(dim=0)\n    base_combat = base_combat_pp.sum(dim=0)\n    produced_delta = torch.zeros(C, A, dtype=fdtype, device=device)\n    combat_delta = torch.zeros(C, A, dtype=fdtype, device=device)\n    if bool(affected_mask.any()):\n        c_aff, p_aff = affected_mask.nonzero(as_tuple=True)\n        N = int(c_aff.numel())\n        cell_id = torch.full((C, P), -1, dtype=torch.long, device=device)\n        cell_id[c_aff, p_aff] = torch.arange(N, device=device)\n        debit_cp = torch.zeros(C, P, dtype=fdtype, device=device)\n        debit_cp.scatter_add_(1, src_safe, torch.where(valid_s, ships, torch.zeros_like(ships)))\n        debit_aff = debit_cp[c_aff, p_aff]\n        arr_aff = torch.zeros(N, H, A, dtype=fdtype, device=device)\n        launch_cell = cell_id.gather(1, tgt_safe)\n        m = valid_t\n        cells, hh, oo, ss = (launch_cell[m], h_idx[m], owner[m], ships[m])\n        ok = cells >= 0\n        arr_aff.index_put_((cells[ok], hh[ok], oo[ok]), ss[ok], accumulate=True)\n        base_arr_k = status.arrivals_by_owner[..., 1:, :]\n        arrivals_cell = base_arr_k[p_aff] + arr_aff\n        init_owner = status.owner[p_aff, 0]\n        init_ships = (status.ships[p_aff, 0] - debit_aff).clamp(min=0.0)\n        prod_aff = prod[p_aff]\n        alive_aff = alive_by_step[:, p_aff].transpose(0, 1)\n        o_t, _s_t, po_t, ps_t = _run_exact_recurrence(init_owner=init_owner.unsqueeze(1), init_ships=init_ships.unsqueeze(1), prod=prod_aff.unsqueeze(1), alive=alive_aff.unsqueeze(1), arrivals=arrivals_cell.unsqueeze(1))\n        zero_frame = torch.zeros(N, 1, 1, A, dtype=fdtype, device=device)\n        arr_full_cell = torch.cat([zero_frame, arrivals_cell.unsqueeze(1)], dim=-2)\n        hyp_prod_pp, hyp_combat_pp = _flow_terms_per_planet(owner=o_t, pre_owner=po_t, pre_ships=ps_t, arr_full=arr_full_cell, prod=prod_aff.unsqueeze(1), alive_pmajor=alive_aff.unsqueeze(1))\n        dprod = hyp_prod_pp.squeeze(1) - base_prod_pp[p_aff]\n        dcombat = hyp_combat_pp.squeeze(1) - base_combat_pp[p_aff]\n        produced_delta.index_put_((c_aff,), dprod, accumulate=True)\n        combat_delta.index_put_((c_aff,), dcombat, accumulate=True)\n    produced_current = base_prod.unsqueeze(0)\n    combat_current = base_combat.unsqueeze(0)\n    diff = GarrisonFlowDiff(player_id=int(player_id), ships_produced_current=produced_current, ships_produced_hypothetical=produced_current + produced_delta, ships_produced_delta=produced_delta, ships_lost_combat_current=combat_current, ships_lost_combat_hypothetical=combat_current + combat_delta, ships_lost_combat_delta=combat_delta, net_ship_delta=produced_delta - combat_delta)\n    if not launches.has_candidate_axis:\n\n        def _sq(t: Tensor) -> Tensor:\n            return t.squeeze(0)\n        diff = GarrisonFlowDiff(player_id=diff.player_id, ships_produced_current=base_prod, ships_produced_hypothetical=_sq(diff.ships_produced_hypothetical), ships_produced_delta=_sq(diff.ships_produced_delta), ships_lost_combat_current=base_combat, ships_lost_combat_hypothetical=_sq(diff.ships_lost_combat_hypothetical), ships_lost_combat_delta=_sq(diff.ships_lost_combat_delta), net_ship_delta=_sq(diff.net_ship_delta))\n    return diff\n\'Cross-k distance cache for the movement-backed planner.\\n\\n\\n\\nEntry ``cross_dist[k, s, t]`` is the Euclidean distance from planet ``s`` at step\\n\\n0 to planet ``t`` at step ``k`` — the *cross-time* distance a fleet must travel if\\n\\nit launches now from ``s`` to intercept ``t`` at time ``k``. For static planets\\n\\nthis equals same-step pairwise distance; for orbiting sources the cross-time form\\n\\nis the geometrically correct quantity for fleet-intercept feasibility. A\\n\\nprecomputed ``[K+1, P, P]`` window gives exact per-step lookups for free.\\n\\n\'\nfrom dataclasses import dataclass\nimport torch\nfrom torch import Tensor\n\n@dataclass\nclass DistanceCache:\n    cross_dist: Tensor\n    alive_by_step: Tensor\n    K: int\n\n    @property\n    def P(self) -> int:\n        return int(self.cross_dist.shape[-1])\n\n    @property\n    def device(self) -> torch.device:\n        return self.cross_dist.device\n\n    @property\n    def dtype(self) -> torch.dtype:\n        return self.cross_dist.dtype\n\ndef build_distance_cache(movement: PlanetMovement, *, max_k: int) -> DistanceCache:\n    K = max(0, min(int(max_k), int(movement.movement_horizon)))\n    P = int(movement.P)\n    src_x0 = movement.x[0]\n    src_y0 = movement.y[0]\n    tgt_x = movement.x[:K + 1]\n    tgt_y = movement.y[:K + 1]\n    dx = src_x0.view(1, P, 1) - tgt_x.unsqueeze(1)\n    dy = src_y0.view(1, P, 1) - tgt_y.unsqueeze(1)\n    cross_dist = torch.sqrt((dx * dx + dy * dy).clamp(min=0.0))\n    alive_by_step = movement.alive_by_step[:K + 1]\n    return DistanceCache(cross_dist=cross_dist, alive_by_step=alive_by_step, K=K)\n\ndef min_distance_to_targets(cache: DistanceCache, source_mask: Tensor, target_mask: Tensor, *, max_k: int) -> Tensor:\n    if source_mask.shape[-1] != cache.P or target_mask.shape[-1] != cache.P:\n        raise ValueError(\'source_mask and target_mask must have shape [P]\')\n    K = max(0, min(int(max_k), int(cache.K)))\n    if K <= 0:\n        return torch.zeros(cache.P, dtype=cache.dtype, device=cache.device)\n    cross = cache.cross_dist[1:K + 1].clone()\n    alive_steps = cache.alive_by_step[1:K + 1]\n    src_mask = source_mask.to(device=cache.device, dtype=torch.bool)\n    tgt_mask = target_mask.to(device=cache.device, dtype=torch.bool)\n    inf_v = float(\'inf\')\n    cross.masked_fill_(~alive_steps.unsqueeze(1), inf_v)\n    cross.masked_fill_(~src_mask.view(1, cache.P, 1), inf_v)\n    cross.masked_fill_(~tgt_mask.view(1, 1, cache.P), inf_v)\n    best_per_target = cross.amin(dim=(0, 1))\n    return torch.where(torch.isfinite(best_per_target), best_per_target, torch.zeros_like(best_per_target))\n"Fixed-fleet intercept aim — sub-turn-accurate angle for an orbiting target.\\n\\n\\n\\nSolves the **continuous** intercept time ``t*`` (root of\\n\\n``v·t = dist(target_pos(t), source) − gap`` with the target on its analytic\\n\\norbit), aims at ``target_pos(t*)``, and verifies that angle with a\\n\\nfully-vectorized analytic first-contact check.\\n\\n\\n\\n* **Root** — a continuous fixed-point iteration (no grid scan / argmax /\\n\\n  bisection), free of grid-resolution artifacts.\\n\\n* **Verify** — :func:`_analytic_first_contact` reproduces the engine\'s\\n\\n  first-contact verdict exactly (swept-pair vs every planet, sun, bounds,\\n\\n  lowest-slot same-step tie-break) with no engine state and no per-step loop.\\n\\n  A shot is viable iff it contacts the target first.\\n\\n\\n\\nReturns ``angle`` / ``eta`` / ``viable``.\\n\\n"\nimport torch\nfrom torch import Tensor\n_FP_ITERS = 6\n_BIG = 1000000.0\n\ndef intercept_angle(movement: PlanetMovement, source_slots: Tensor, target_slots: Tensor, fleet_sizes: Tensor, *, fp_iters: int=_FP_ITERS, active: Tensor | None=None) -> dict[str, Tensor]:\n    dev = movement.device\n    dt = movement.dtype\n    H = int(movement.movement_horizon)\n    src, tgt, ships = torch.broadcast_tensors(source_slots.to(device=dev), target_slots.to(device=dev), fleet_sizes.to(device=dev, dtype=dt))\n    shape = src.shape\n    src = src.long().clamp(0, max(movement.P - 1, 0)).reshape(-1)\n    tgt = tgt.long().clamp(0, max(movement.P - 1, 0)).reshape(-1)\n    ships = ships.to(dt).clamp(min=1.0).reshape(-1)\n    M = src.shape[0]\n    sx, sy = movement.position_at_slots(src, 0)\n    src_r = movement.radii[src]\n    tgt_r = movement.radii[tgt]\n    speed = fleet_speed(ships).clamp(min=1e-06)\n    t0x, t0y = movement.position_at_slots(tgt, 0)\n    t1x, t1y = movement.position_at_slots(tgt, 1)\n    R = torch.sqrt(((t0x - CENTER) ** 2 + (t0y - CENTER) ** 2).clamp(min=0.0))\n    a0 = torch.atan2(t0y - CENTER, t0x - CENTER)\n    a1 = torch.atan2(t1y - CENTER, t1x - CENTER)\n    omega = torch.atan2(torch.sin(a1 - a0), torch.cos(a1 - a0))\n    gap = src_r + LAUNCH_SURFACE_OFFSET + tgt_r + TARGET_HIT_SURFACE_OFFSET\n\n    def target_pos(t: Tensor):\n        ang = a0 + omega * t\n        return (CENTER + R * torch.cos(ang), CENTER + R * torch.sin(ang))\n    d0 = torch.sqrt(((t0x - sx) ** 2 + (t0y - sy) ** 2).clamp(min=0.0))\n    t_star = ((d0 - gap) / speed).clamp(min=0.0, max=float(H))\n    for _ in range(int(fp_iters)):\n        tx, ty = target_pos(t_star)\n        d = torch.sqrt(((tx - sx) ** 2 + (ty - sy) ** 2).clamp(min=0.0))\n        t_star = ((d - gap) / speed).clamp(min=0.0, max=float(H))\n    tx, ty = target_pos(t_star)\n    angle = torch.atan2(ty - sy, tx - sx)\n    cos_a = torch.cos(angle)\n    sin_a = torch.sin(angle)\n    launch_x = sx + cos_a * (src_r + LAUNCH_SURFACE_OFFSET)\n    launch_y = sy + sin_a * (src_r + LAUNCH_SURFACE_OFFSET)\n    eta_cap = (t_star + 2.0).clamp(max=float(H))\n    seg_len = speed * eta_cap + tgt_r + 2.0\n    px = movement.x[:H + 1, :]\n    py = movement.y[:H + 1, :]\n    radii_p = movement.radii\n    alive0 = movement.alive_at(0)\n    if active is None:\n        contact, eta_c = _analytic_first_contact(launch_x=launch_x, launch_y=launch_y, cos_a=cos_a, sin_a=sin_a, speed=speed, px=px, py=py, p_alive0=alive0, radii=radii_p, H=H, seg_len=seg_len)\n    else:\n        act = active.broadcast_to(shape).reshape(M).to(torch.bool)\n        n_max = max(1, int(act.sum().item()))\n        order = (~act).to(torch.int8).argsort(stable=True)\n        midx = order[:n_max]\n        keep = act[midx]\n        contact_m, eta_cm = _analytic_first_contact(launch_x=launch_x[midx], launch_y=launch_y[midx], cos_a=cos_a[midx], sin_a=sin_a[midx], speed=speed[midx], px=px, py=py, p_alive0=alive0, radii=radii_p, H=H, seg_len=seg_len[midx])\n        contact = torch.full((M,), -1, dtype=contact_m.dtype, device=dev)\n        eta_c = torch.full((M,), float(H), dtype=eta_cm.dtype, device=dev)\n        contact[midx] = torch.where(keep, contact_m, torch.full_like(contact_m, -1))\n        eta_c[midx] = torch.where(keep, eta_cm, torch.full_like(eta_cm, float(H)))\n    viable = contact == tgt\n    eta_out = torch.where(viable, eta_c.to(dt), torch.full_like(eta_c.to(dt), float(\'inf\')))\n    return {\'angle\': angle.reshape(shape), \'eta\': eta_out.reshape(shape), \'viable\': viable.reshape(shape)}\n\ndef _analytic_first_contact(*, launch_x: Tensor, launch_y: Tensor, cos_a: Tensor, sin_a: Tensor, speed: Tensor, px: Tensor, py: Tensor, p_alive0: Tensor, radii: Tensor, H: int, seg_len: Tensor | None=None, max_bytes: int=256 * 1024 * 1024):\n    M = cos_a.shape[0]\n    P = px.shape[-1]\n    dev = cos_a.device\n    dt = launch_x.dtype\n    N = M\n    big = _BIG\n    lx = launch_x.reshape(N)\n    ly = launch_y.reshape(N)\n    ca = cos_a.reshape(N)\n    sa = sin_a.reshape(N)\n    sp = speed.reshape(N)\n    slen = sp * float(H) if seg_len is None else seg_len.reshape(N)\n    end_x = lx + ca * slen\n    end_y = ly + sa * slen\n    seg_xmin = torch.minimum(lx, end_x)\n    seg_xmax = torch.maximum(lx, end_x)\n    seg_ymin = torch.minimum(ly, end_y)\n    seg_ymax = torch.maximum(ly, end_y)\n    bb_xmin = px.amin(0) - radii\n    bb_xmax = px.amax(0) + radii\n    bb_ymin = py.amin(0) - radii\n    bb_ymax = py.amax(0) + radii\n    keep = ~((seg_xmax.unsqueeze(1) < bb_xmin) | (seg_xmin.unsqueeze(1) > bb_xmax) | (seg_ymax.unsqueeze(1) < bb_ymin) | (seg_ymin.unsqueeze(1) > bb_ymax))\n    K = max(1, int(keep.sum(1).amax().item()))\n    order = (~keep).to(torch.int8).argsort(dim=1, stable=True)\n    shortlist = order[:, :K]\n    valid = keep.gather(1, shortlist)\n    k = torch.arange(H + 1, device=dev, dtype=dt)\n    t_ax = torch.arange(H + 1, device=dev).view(1, H + 1, 1)\n    step_h = torch.arange(1, H + 1, device=dev, dtype=dt).view(1, H, 1)\n    bytes_per = max(1, 16 * H * K * 4)\n    chunk = max(4096, max_bytes // bytes_per)\n    chunk = min(chunk, max(N, 1))\n    contacts: list[Tensor] = []\n    etas: list[Tensor] = []\n    for s in range(0, N, chunk):\n        e = min(s + chunk, N)\n        sl = shortlist[s:e]\n        fx = lx[s:e].view(-1, 1) + ca[s:e].view(-1, 1) * sp[s:e].view(-1, 1) * k\n        fy = ly[s:e].view(-1, 1) + sa[s:e].view(-1, 1) * sp[s:e].view(-1, 1) * k\n        sl_e = sl.view(-1, 1, K)\n        pxc = px[t_ax, sl_e]\n        pyc = py[t_ax, sl_e]\n        radc = radii[sl]\n        alivec = p_alive0[sl] & valid[s:e]\n        real_slot = sl.to(dt)\n        fx0 = fx[:, :-1].unsqueeze(-1)\n        fy0 = fy[:, :-1].unsqueeze(-1)\n        fx1 = fx[:, 1:].unsqueeze(-1)\n        fy1 = fy[:, 1:].unsqueeze(-1)\n        hit = _swept_pair_hit_mask(fx0, fy0, fx1, fy1, pxc[:, :-1, :], pyc[:, :-1, :], pxc[:, 1:, :], pyc[:, 1:, :], radc.unsqueeze(1))\n        hit = hit & alivec.unsqueeze(1)\n        planet_hit_step = torch.where(hit, step_h, torch.full_like(step_h, big)).amin(1)\n        first_planet_step = planet_hit_step.amin(1)\n        is_first = planet_hit_step == first_planet_step.unsqueeze(-1)\n        contact_planet = torch.where(is_first, real_slot, torch.full_like(real_slot, big)).amin(1)\n        nfx = fx[:, 1:]\n        nfy = fy[:, 1:]\n        ofx = fx[:, :-1]\n        ofy = fy[:, :-1]\n        oob = (nfx < 0) | (nfx > BOARD_SIZE) | (nfy < 0) | (nfy > BOARD_SIZE)\n        vx = nfx - ofx\n        vy = nfy - ofy\n        wx = CENTER - ofx\n        wy = CENTER - ofy\n        vv = (vx * vx + vy * vy).clamp(min=1e-12)\n        t = ((wx * vx + wy * vy) / vv).clamp(0.0, 1.0)\n        cxp = ofx + t * vx\n        cyp = ofy + t * vy\n        sun = (cxp - CENTER) ** 2 + (cyp - CENTER) ** 2 < SUN_RADIUS * SUN_RADIUS\n        env = oob | sun\n        death_step = torch.where(env, step_h.squeeze(-1), torch.full_like(env, big, dtype=dt)).amin(1)\n        ht = (first_planet_step <= death_step) & (first_planet_step < big)\n        contacts.append(torch.where(ht, contact_planet, torch.full_like(contact_planet, -1.0)).long())\n        etas.append(torch.where(ht, first_planet_step, torch.full_like(first_planet_step, float(H))))\n    contact = (contacts[0] if len(contacts) == 1 else torch.cat(contacts)).view(M)\n    eta = (etas[0] if len(etas) == 1 else torch.cat(etas)).view(M)\n    return (contact, eta)\nfrom dataclasses import dataclass\nfrom typing import Sequence\nimport torch\nfrom torch import Tensor\n\n@dataclass(frozen=True)\nclass PlannedLaunches:\n    source_slots: Tensor\n    angle: Tensor\n    ships: Tensor\n    target_slots: Tensor\n    eta_turns: Tensor\n    valid: Tensor\n    fleet_ids: Tensor\n\n@dataclass(frozen=True)\nclass LaunchEntries:\n    source_slots: Tensor\n    target_slots: Tensor\n    ships: Tensor\n    angle: Tensor\n    eta: Tensor\n    valid: Tensor\n\n    @property\n    def width(self) -> int:\n        return int(self.source_slots.shape[0])\n\ndef concat_launch_entries(entries: Sequence[LaunchEntries]) -> LaunchEntries:\n    if not entries:\n        raise ValueError(\'concat_launch_entries requires at least one entry table\')\n    if len(entries) == 1:\n        return entries[0]\n    return LaunchEntries(source_slots=torch.cat([e.source_slots for e in entries], dim=0), target_slots=torch.cat([e.target_slots for e in entries], dim=0), ships=torch.cat([e.ships for e in entries], dim=0), angle=torch.cat([e.angle for e in entries], dim=0), eta=torch.cat([e.eta for e in entries], dim=0), valid=torch.cat([e.valid for e in entries], dim=0))\n\ndef disambiguate_duplicate_launches(entries: LaunchEntries, *, epsilon: float=1e-05) -> LaunchEntries:\n    src = entries.source_slots\n    ang = entries.angle\n    ships = entries.ships\n    valid = entries.valid\n    L = src.shape[0]\n    if L < 2 or not bool(valid.any()):\n        return entries\n    device = src.device\n    src_i = src.unsqueeze(1)\n    src_j = src.unsqueeze(0)\n    ang_i = ang.unsqueeze(1)\n    ang_j = ang.unsqueeze(0)\n    ships_i = ships.unsqueeze(1)\n    ships_j = ships.unsqueeze(0)\n    valid_i = valid.unsqueeze(1)\n    valid_j = valid.unsqueeze(0)\n    j_indices = torch.arange(L, device=device).view(1, L)\n    i_indices = torch.arange(L, device=device).view(L, 1)\n    earlier = j_indices < i_indices\n    match = valid_i & valid_j & (src_i == src_j) & (ang_i == ang_j) & (ships_i == ships_j) & earlier\n    if not bool(match.any()):\n        return entries\n    dup_count = match.sum(dim=1).to(ang.dtype)\n    new_angle = ang + dup_count * float(epsilon)\n    return LaunchEntries(source_slots=entries.source_slots, target_slots=entries.target_slots, ships=entries.ships, angle=new_angle, eta=entries.eta, valid=entries.valid)\n\ndef ensure_planet_movement(*, obs_tensors: dict, expected_cfg: MovementConfig, cached_movement: PlanetMovement | None) -> PlanetMovement:\n    if cached_movement is not None and cached_movement.config == expected_cfg:\n        cached_movement.update(obs_tensors)\n        return cached_movement\n    return PlanetMovement.from_obs_tensors(obs_tensors, config=expected_cfg)\n\ndef _resolve_player_next_fleet_id(obs_tensors: dict, *, device: torch.device) -> Tensor:\n    next_fleet_id = obs_tensors.get(\'player_next_fleet_id\', obs_tensors.get(\'next_fleet_id\'))\n    if next_fleet_id is None:\n        return torch.zeros((), dtype=torch.long, device=device)\n    return next_fleet_id.to(device=device, dtype=torch.long)\n\ndef infer_planned_launches_from_entries(*, obs_tensors: dict, movement: PlanetMovement, entries: LaunchEntries, player_id: int) -> PlannedLaunches:\n    source_slots = entries.source_slots\n    angle = entries.angle\n    ships = entries.ships\n    launch_valid = entries.valid\n    L = source_slots.shape[0]\n    device = source_slots.device\n    P = max(int(movement.P), 1)\n    next_fleet_id = _resolve_player_next_fleet_id(obs_tensors, device=device)\n    launch_long = launch_valid.to(torch.long)\n    launch_rank = launch_long.cumsum(0) - launch_long\n    fleet_ids = next_fleet_id + launch_rank\n    src_safe = source_slots.clamp(min=0, max=P - 1)\n    launch_x, launch_y = movement.position_at_slots(src_safe, 0)\n    source_r = movement.radii[src_safe]\n    start_x = launch_x + torch.cos(angle) * (source_r + 0.1)\n    start_y = launch_y + torch.sin(angle) * (source_r + 0.1)\n    source_planet_ids = movement.planet_ids[src_safe]\n    rows = torch.full((L, 7), -1.0, dtype=movement.dtype, device=device)\n    rows[..., 0] = fleet_ids.to(dtype=movement.dtype)\n    rows[..., 1] = float(player_id)\n    rows[..., 2] = start_x.to(dtype=movement.dtype)\n    rows[..., 3] = start_y.to(dtype=movement.dtype)\n    rows[..., 4] = angle.to(dtype=movement.dtype)\n    rows[..., 5] = source_planet_ids.to(dtype=movement.dtype)\n    rows[..., 6] = ships.to(dtype=movement.dtype)\n    rows[..., 0] = torch.where(launch_valid, rows[..., 0], torch.full_like(rows[..., 0], -1.0))\n    target_slots = torch.zeros(L, dtype=torch.long, device=device)\n    eta_turns = torch.zeros(L, dtype=torch.float32, device=device)\n    intent_valid = torch.zeros(L, dtype=torch.bool, device=device)\n    fleet_slot = torch.where(launch_valid)[0]\n    if int(fleet_slot.numel()) > 0:\n        estimate = _estimate_new_fleet_arrivals(movement=movement, obs_fleets=rows, fleet_slot=fleet_slot)\n        valid_hit = estimate[\'has_hit\']\n        if bool(valid_hit.any()):\n            src = fleet_slot[valid_hit]\n            target_slots[src] = estimate[\'target_slot\'][valid_hit]\n            eta_turns[src] = estimate[\'eta_index\'][valid_hit].to(dtype=torch.float32) + 1.0\n            intent_valid[src] = True\n    return PlannedLaunches(source_slots=source_slots, angle=angle, ships=ships, target_slots=target_slots, eta_turns=eta_turns, valid=intent_valid, fleet_ids=fleet_ids)\n\ndef apply_private_planned_launches(*, movement: PlanetMovement, launches: PlannedLaunches, owner_id: int, obs_tensors: dict) -> None:\n    if not movement.track_fleets:\n        return\n    movement.record_fleet_arrivals(target_slots=launches.target_slots, owner_ids=int(owner_id), ships=launches.ships, eta=launches.eta_turns, valid=launches.valid)\n    nfid = obs_tensors.get(\'next_fleet_id\')\n    if nfid is None:\n        raise ValueError("obs_tensors is missing \'next_fleet_id\'")\n    movement.stash_pending_own_launches(owner_id=int(owner_id), source_slots=launches.source_slots, ships=launches.ships, angle=launches.angle, target_slots=launches.target_slots, eta=launches.eta_turns, valid=launches.valid, prev_next_fleet_id=nfid)\n\'Flow-diff scored planner core: candidate scoring, shortlists, aim, selection.\\n\\n\\n\\nPure, tensor-only planning helpers for one game: the competitive net-ship-delta\\n\\nscorer, target/source shortlists, capture-floor sizing, the strict-superset\\n\\nreachability gate, the device-stable greedy selector, the hold-reserve cap\\n\\n``safe_drain``, and the pressure-gradient regrouper.\\n\\n\'\nimport torch\nfrom torch import Tensor\n\ndef largest_initial_player_count(obs_tensors: dict) -> int:\n    metadata_count = obs_tensors.get(\'player_count\')\n    if metadata_count is not None:\n        count = int(metadata_count.flatten()[0].item()) if isinstance(metadata_count, Tensor) else int(metadata_count)\n        if count in (2, 4):\n            return count\n    initial = obs_tensors[\'initial_planets\']\n    pid = initial[:, 0]\n    owner = initial[:, 1]\n    mask = (pid >= 0) & (owner >= 0)\n    owners = owner[mask]\n    n_max = 2\n    if owners.numel() > 0:\n        n_max = max(n_max, int(torch.unique(owners.long()).numel()))\n    return n_max\n\ndef make_launch_set(*, source_slots: Tensor, target_slots: Tensor, ships: Tensor, eta: Tensor, valid: Tensor, player_id: int) -> LaunchSet:\n    owner = torch.full_like(source_slots, int(player_id), dtype=torch.long)\n    return LaunchSet(source_slots=source_slots.to(torch.long), target_slots=target_slots.to(torch.long), ships=ships, eta=eta, owner=owner, valid=valid.to(torch.bool))\n\ndef competitive_score(diff: GarrisonFlowDiff, *, player_id: int) -> Tensor:\n    net = diff.net_ship_delta\n    me = net[..., int(player_id)]\n    opp = net.sum(dim=-1) - me\n    return me - opp\n\ndef score_candidates(status: PlanetGarrisonStatus, *, prod: Tensor, alive_by_step: Tensor, player_count: int, launches: LaunchSet, player_id: int) -> Tensor:\n    diff = sparse_launch_flow_delta(status, prod=prod, alive_by_step=alive_by_step, player_count=int(player_count), launches=launches, player_id=int(player_id))\n    return competitive_score(diff, player_id=int(player_id))\n\ndef _stable_topk_indices(ranked: Tensor, k: int) -> Tensor:\n    order = torch.argsort(ranked, dim=-1, descending=True, stable=True)\n    return order[..., :max(1, int(k))]\n\ndef _stable_argmax(scores: Tensor) -> Tensor:\n    C = int(scores.shape[-1])\n    is_max = scores == scores.max(dim=-1, keepdim=True).values\n    idx = torch.arange(C, device=scores.device).expand_as(scores)\n    return torch.where(is_max, idx, torch.full_like(idx, C)).argmin(dim=-1)\n\ndef _candidate_indices(values: Tensor, mask: Tensor, cap: int) -> tuple[Tensor, Tensor]:\n    p_count = values.shape[0]\n    k = p_count if cap <= 0 else min(int(cap), p_count)\n    neg_inf = torch.full_like(values, float(\'-inf\'))\n    ranked = torch.where(mask, values, neg_inf)\n    top_idx = _stable_topk_indices(ranked, max(1, k))\n    top_vals = ranked[top_idx]\n    return (top_idx, top_vals > float(\'-inf\'))\n\ndef is_comet_planet(obs_tensors: dict, P: int, device: torch.device) -> Tensor | None:\n    comet_ids = obs_tensors.get(\'comet_planet_ids\')\n    planets = obs_tensors.get(\'planets\')\n    if comet_ids is None or planets is None:\n        return None\n    planet_ids = planets[..., 0].long()\n    comet_ids = comet_ids.to(device=device)\n    mask = torch.zeros(P, dtype=torch.bool, device=device)\n    for c in range(int(comet_ids.shape[-1])):\n        cid = comet_ids[c]\n        mask = mask | (planet_ids == cid) & (cid >= 0)\n    return mask\n\ndef reinforcement_timing_factor(eta: Tensor, *, eta_free: float, eta_scale: float) -> Tensor:\n    scale = max(float(eta_scale), 1e-06)\n    return ((eta - float(eta_free)) / scale).clamp(0.0, 1.0)\n\ndef capture_floor(garrison_status: PlanetGarrisonStatus, *, target_idx: Tensor, k_max: int, capture_overhead: float, player_id: int, reinforcement: Tensor | None=None) -> Tensor:\n    ships = garrison_status.ships\n    owner = garrison_status.owner\n    dtype = ships.dtype if ships.is_floating_point() else torch.float32\n    T = target_idx.shape[0]\n    H_axis = int(ships.shape[-1])\n    P = int(ships.shape[0])\n    K = max(0, min(int(k_max), H_axis - 1))\n    if K == 0:\n        return torch.empty(T, 0, dtype=dtype, device=ships.device)\n    tgt = target_idx.clamp(min=0, max=max(P - 1, 0))\n    gathered = ships[tgt].to(dtype=dtype)\n    owner_g = owner[tgt]\n    k_idx = torch.arange(1, K + 1, device=ships.device).view(1, K).expand(T, K)\n    defenders = gathered.gather(-1, k_idx)\n    mine_at_k = owner_g.gather(-1, k_idx) == int(player_id)\n    if reinforcement is not None:\n        assert reinforcement.shape[-1] >= K, f\'reinforcement last dim {reinforcement.shape[-1]} < capture_floor K={K}\'\n        extra = reinforcement[..., :K].to(dtype=dtype, device=ships.device)\n    else:\n        extra = 0.0\n    cap = (defenders + float(capture_overhead) + extra).clamp(min=1.0).ceil()\n    return torch.where(mine_at_k, torch.ones_like(cap), cap)\n\ndef attack_target_mask(obs, obs_tensors: dict) -> Tensor:\n    mask = (obs.is_enemy | obs.is_neutral) & obs.alive\n    comet = is_comet_planet(obs_tensors, obs.P, obs.device)\n    if comet is not None:\n        mask = mask & ~comet\n    return mask\n\ndef friendly_flip_targets(obs, garrison_status: PlanetGarrisonStatus, *, H: int, prod: Tensor) -> tuple[Tensor, Tensor]:\n    P = obs.P\n    device = obs.device\n    pid = int(obs.player_id)\n    if H <= 0:\n        z = torch.zeros(P, device=device)\n        return (torch.zeros(P, dtype=torch.bool, device=device), z)\n    owner_h = garrison_status.owner[..., 1:]\n    flips = obs.owned.unsqueeze(-1) & (owner_h != pid)\n    any_flip = flips.any(dim=-1)\n    flip_turn = _stable_argmax(flips.to(torch.int64)) + 1\n    remaining = (float(H) - flip_turn.to(prod.dtype)).clamp(min=0.0)\n    urgency = prod * remaining + obs.ships\n    urgency = torch.where(any_flip, urgency, torch.full_like(urgency, float(\'-inf\')))\n    return (any_flip, urgency)\n\ndef build_target_shortlist(obs, obs_tensors, garrison_status, cache, *, config, K_eta, H, prod, source_mask):\n    P = obs.P\n    device = obs.device\n    n_attack = max(1, min(int(config.max_offensive_targets), P))\n    R = max(0, min(int(config.max_defensive_targets), P))\n    attack_mask = attack_target_mask(obs, obs_tensors)\n    proximity = min_distance_to_targets(cache, source_mask, attack_mask, max_k=K_eta)\n    prod_weight = (\n        float(TARGET_SHORTLIST_PROD_WEIGHT)\n        if int(obs.step.item()) < TARGET_SHORTLIST_PROD_TURN_LIMIT else 0.0\n    )\n    attack_pref = -proximity + prod_weight * prod\n    attack_pref = torch.where(attack_mask, attack_pref, torch.full_like(proximity, float(\'-inf\')))\n    atk_idx, atk_exists = _candidate_indices(attack_pref, attack_mask, n_attack)\n    if R > 0:\n        flip_mask, urgency = friendly_flip_targets(obs, garrison_status, H=H, prod=prod)\n        def_idx, def_exists = _candidate_indices(urgency, flip_mask, R)\n        target_idx = torch.cat([atk_idx, def_idx], dim=0)\n        target_exists = torch.cat([atk_exists, def_exists], dim=0)\n    else:\n        target_idx, target_exists = (atk_idx, atk_exists)\n    return (target_idx, target_exists)\n\ndef reachable_mask(movement: PlanetMovement, *, source_idx: Tensor, target_idx: Tensor, fleet_sizes: Tensor, eta_cap: Tensor, eps: float=0.0001) -> Tensor:\n    S, T, G = fleet_sizes.shape\n    P = int(movement.P)\n    dt = movement.dtype\n    K = max(1, min(int(movement.movement_horizon), int(torch.ceil(eta_cap.max()).item())))\n    src = source_idx.clamp(0, P - 1)\n    tgt = target_idx.clamp(0, P - 1)\n    sx = movement.x[0][src].view(S, 1, 1)\n    sy = movement.y[0][src].view(S, 1, 1)\n    tx = movement.x[:K + 1].gather(1, tgt.view(1, T).expand(K + 1, T))\n    ty = movement.y[:K + 1].gather(1, tgt.view(1, T).expand(K + 1, T))\n    ax = tx[:K, :].view(1, K, T)\n    ay = ty[:K, :].view(1, K, T)\n    bx = tx[1:, :].view(1, K, T)\n    by = ty[1:, :].view(1, K, T)\n    abx = bx - ax\n    aby = by - ay\n    apx = sx - ax\n    apy = sy - ay\n    denom = (abx * abx + aby * aby).clamp(min=1e-12)\n    u = ((apx * abx + apy * aby) / denom).clamp(0.0, 1.0)\n    cx = ax + u * abx\n    cy = ay + u * aby\n    seg_dist = torch.sqrt(((sx - cx) ** 2 + (sy - cy) ** 2).clamp(min=0.0))\n    src_r = movement.radii[src].view(S, 1, 1)\n    tgt_r = movement.radii[tgt].view(1, 1, T)\n    gap = src_r + tgt_r + (LAUNCH_SURFACE_OFFSET + TARGET_HIT_SURFACE_OFFSET)\n    surf = (seg_dist - gap).clamp(min=0.0)\n    kv = torch.arange(1, K + 1, device=movement.device, dtype=dt).view(1, K, 1)\n    ratio = surf / kv\n    within = kv <= eta_cap.view(1, 1, T)\n    ratio = torch.where(within, ratio, torch.full_like(ratio, float(\'inf\')))\n    min_ratio = ratio.amin(dim=1)\n    speed = fleet_speed(fleet_sizes.clamp(min=1.0))\n    reachable = min_ratio.unsqueeze(-1) <= speed * (1.0 + float(eps))\n    distinct = (src.view(S, 1) != tgt.view(1, T)).unsqueeze(-1)\n    return reachable & distinct\n\ndef _greedy_select(*, P, W, device, dtype, score, cand_src, cand_send, cand_angle, cand_eta, cand_active, cand_tgt_slot, cand_tgt_short, cand_is_def, source_budget, target_exists, roi_threshold) -> LaunchEntries:\n    C, L = (int(cand_src.shape[0]), int(cand_src.shape[1]))\n    target_taken = ~target_exists.clone()\n    defended = torch.zeros(P, dtype=torch.bool, device=device)\n    used_src = torch.zeros(P, dtype=torch.bool, device=device)\n    w_src = torch.zeros(W, L, dtype=torch.long, device=device)\n    w_send = torch.zeros(W, L, dtype=dtype, device=device)\n    w_angle = torch.zeros(W, L, dtype=dtype, device=device)\n    w_eta = torch.ones(W, L, dtype=dtype, device=device)\n    w_tgt = torch.zeros(W, L, dtype=torch.long, device=device)\n    w_active = torch.zeros(W, L, dtype=torch.bool, device=device)\n    for w in range(W):\n        taken_cand = target_taken[cand_tgt_short]\n        budget_at = source_budget[cand_src]\n        can_fund = ((cand_send <= budget_at) | ~cand_active).all(dim=-1)\n        tgt_used_as_src = used_src[cand_tgt_slot]\n        contrib_defended = (defended[cand_src] & cand_active).any(dim=-1)\n        mask = torch.isfinite(score) & ~taken_cand & can_fund & ~tgt_used_as_src & ~contrib_defended\n        masked = torch.where(mask, score, torch.full_like(score, float(\'-inf\')))\n        best_c = _stable_argmax(masked)\n        best_score = masked[best_c]\n        fired = bool(torch.isfinite(best_score) & (best_score > roi_threshold))\n        if not fired:\n            break\n        sel_src = cand_src[best_c]\n        sel_send = cand_send[best_c]\n        sel_active = cand_active[best_c]\n        w_src[w] = sel_src\n        w_send[w] = torch.where(sel_active, sel_send, torch.zeros_like(sel_send))\n        w_angle[w] = cand_angle[best_c]\n        w_eta[w] = cand_eta[best_c]\n        w_tgt[w] = cand_tgt_slot[best_c]\n        w_active[w] = sel_active\n        debit = torch.zeros_like(source_budget)\n        debit.scatter_add_(0, sel_src, torch.where(sel_active, sel_send, torch.zeros_like(sel_send)))\n        source_budget = (source_budget - debit).clamp(min=0.0)\n        target_taken[cand_tgt_short[best_c]] = True\n        src_mark = torch.zeros(P, dtype=torch.long, device=device)\n        src_mark.scatter_add_(0, sel_src, sel_active.to(torch.long))\n        used_src = used_src | (src_mark > 0)\n        sel_tgt = cand_tgt_slot[best_c]\n        sel_is_def = bool(cand_is_def[best_c])\n        defended[sel_tgt] = defended[sel_tgt] | sel_is_def\n    WL = W * L\n    entries = LaunchEntries(source_slots=w_src.reshape(WL), target_slots=w_tgt.reshape(WL), ships=torch.where(w_active, w_send, torch.zeros_like(w_send)).reshape(WL), angle=torch.where(w_active, w_angle, torch.zeros_like(w_angle)).reshape(WL), eta=torch.where(w_active, w_eta, torch.ones_like(w_eta)).reshape(WL), valid=w_active.reshape(WL))\n    return (entries, source_budget)\n\ndef _plan_regroup(*, movement, obs, obs_tensors, garrison_status, leftover, original_ships, pressure, config, H) -> LaunchEntries:\n    P = obs.P\n    device = obs.device\n    dtype = original_ships.dtype\n    pid = int(obs.player_id)\n    min_send = float(config.min_ships_to_launch)\n    src_mask = obs.owned & obs.alive & (leftover >= min_send)\n    if not bool(src_mask.any()):\n        return _empty_entries(device, dtype)\n    S_cap = max(1, min(int(config.max_regroup_sources_per_lane), P))\n    src_idx, src_exists = _candidate_indices(leftover, src_mask, S_cap)\n    S = int(src_idx.shape[0])\n    leftover_s = leftover[src_idx.clamp(0, P - 1)]\n    orig_s = original_ships[src_idx.clamp(0, P - 1)]\n    H_eff = torch.full((), float(H), dtype=dtype, device=device)\n    drain_s = safe_drain(garrison_status, source_idx=src_idx, source_ships=orig_s, H_eff=H_eff, player_id=pid)\n    committed_s = (orig_s - leftover_s).clamp(min=0.0)\n    regroup_cap = torch.minimum(leftover_s, (drain_s - committed_s).clamp(min=0.0)).floor()\n    can_send = src_exists & (regroup_cap >= min_send)\n    if not bool(can_send.any()):\n        return _empty_entries(device, dtype)\n    dst_mask = obs.owned & obs.alive\n    comet = is_comet_planet(obs_tensors, P, device)\n    if comet is not None:\n        dst_mask = dst_mask & ~comet\n    T_cap = max(1, min(int(config.max_regroup_targets_per_source), P))\n    dst_idx, dst_exists = _candidate_indices(pressure, dst_mask, T_cap)\n    T = int(dst_idx.shape[0])\n    regroup_active = reachable_mask(movement, source_idx=src_idx, target_idx=dst_idx, fleet_sizes=regroup_cap.view(S, 1, 1).expand(S, T, 1), eta_cap=torch.full((T,), float(config.max_regroup_time), device=device)).squeeze(-1)\n    aim = intercept_angle(movement, src_idx.unsqueeze(1), dst_idx.unsqueeze(0), regroup_cap.unsqueeze(1), active=regroup_active)\n    angle = aim[\'angle\']\n    eta = aim[\'eta\']\n    viable = aim[\'viable\']\n    src_pres = pressure[src_idx.clamp(0, P - 1)].view(S, 1)\n    dst_pres = pressure[dst_idx.clamp(0, P - 1)].view(1, T)\n    gap = dst_pres - src_pres\n    owner = garrison_status.owner\n    H_axis = int(owner.shape[-1])\n    dst_owner = owner[dst_idx.clamp(0, P - 1)]\n    k = torch.ceil(eta).clamp(min=0, max=H_axis - 1).to(torch.long)\n    owner_at_k = dst_owner.unsqueeze(0).expand(S, T, H_axis).gather(-1, k.unsqueeze(-1)).squeeze(-1)\n    still_mine = owner_at_k == pid\n    src_neq_dst = src_idx.view(S, 1) != dst_idx.view(1, T)\n    valid = viable & still_mine & src_neq_dst & (gap > float(config.regroup_pressure_delta_min)) & (eta <= float(config.max_regroup_time)) & can_send.view(S, 1) & dst_exists.view(1, T)\n    sc = torch.where(valid, gap - float(config.regroup_time_penalty_weight) * eta, torch.full_like(gap, float(\'-inf\')))\n    best_t = _stable_argmax(sc)\n    best_score = sc.gather(-1, best_t.unsqueeze(-1)).squeeze(-1)\n    best_valid = torch.isfinite(best_score)\n    s_ar = torch.arange(S, device=device)\n    best_dst = dst_idx[best_t]\n    best_angle = angle[s_ar, best_t]\n    best_eta = eta[s_ar, best_t]\n    return LaunchEntries(source_slots=src_idx, target_slots=best_dst, ships=torch.where(best_valid, regroup_cap, torch.zeros_like(regroup_cap)), angle=torch.where(best_valid, best_angle, torch.zeros_like(best_angle)), eta=torch.where(best_valid, best_eta, torch.ones_like(best_eta)), valid=best_valid)\n\ndef _empty_entries(device: torch.device, dtype: torch.dtype) -> LaunchEntries:\n    z = torch.zeros(0, dtype=dtype, device=device)\n    zl = torch.zeros(0, dtype=torch.long, device=device)\n    return LaunchEntries(source_slots=zl, target_slots=zl, ships=z, angle=z, eta=z, valid=torch.zeros(0, dtype=torch.bool, device=device))\n\ndef entries_to_sparse_payload(entries: LaunchEntries, *, planet_ids: Tensor) -> dict[str, Tensor]:\n    L = entries.source_slots.shape[0]\n    device = entries.source_slots.device\n    P = int(planet_ids.shape[0])\n    valid_long = entries.valid.to(torch.int64)\n    counts = valid_long.sum().to(torch.int32)\n    max_count = int(counts.item())\n    out_from = torch.full((max_count,), -1, dtype=torch.int32, device=device)\n    out_angle = torch.zeros((max_count,), dtype=torch.float32, device=device)\n    out_ships = torch.zeros((max_count,), dtype=torch.float32, device=device)\n    if max_count == 0:\n        return {\'from_planet_id\': out_from, \'angle\': out_angle, \'num_ships\': out_ships, \'counts\': counts}\n    safe_src = entries.source_slots.clamp(min=0, max=max(P - 1, 0))\n    from_pid_full = planet_ids[safe_src].to(torch.int32)\n    launch_rank = valid_long.cumsum(0) - valid_long\n    l_idx = torch.where(entries.valid)[0]\n    pos = launch_rank[l_idx]\n    out_from[pos] = from_pid_full[l_idx]\n    out_angle[pos] = entries.angle[l_idx].to(torch.float32)\n    out_ships[pos] = entries.ships[l_idx].to(torch.float32)\n    return {\'from_planet_id\': out_from, \'angle\': out_angle, \'num_ships\': out_ships, \'counts\': counts}\n\ndef empty_action_row(device: torch.device) -> dict[str, Tensor]:\n    return {\'from_planet_id\': torch.full((0,), -1, dtype=torch.int32, device=device), \'angle\': torch.zeros((0,), dtype=torch.float32, device=device), \'num_ships\': torch.zeros((0,), dtype=torch.float32, device=device), \'counts\': torch.zeros((), dtype=torch.int32, device=device)}\n\ndef safe_drain(garrison_status: PlanetGarrisonStatus, *, source_idx: Tensor, source_ships: Tensor, H_eff: Tensor, player_id: int=0) -> Tensor:\n    S = source_idx.shape[0]\n    ships_cache = garrison_status.ships\n    dtype = ships_cache.dtype if ships_cache.is_floating_point() else torch.float32\n    device = ships_cache.device\n    H_axis = int(ships_cache.shape[-1])\n    H = max(H_axis - 1, 0)\n    P = int(ships_cache.shape[0])\n    if H == 0:\n        return torch.zeros(S, dtype=dtype, device=device)\n    src_idx_safe = source_idx.clamp(min=0, max=max(P - 1, 0))\n    src_ships_traj = ships_cache[src_idx_safe][..., 1:].to(dtype=dtype)\n    src_owner_traj = garrison_status.owner[src_idx_safe][..., 1:]\n    me_owned = src_owner_traj == int(player_id)\n    turn_grid = torch.arange(1, H + 1, device=device, dtype=dtype).view(1, H)\n    within_horizon = turn_grid <= H_eff\n    held = me_owned & within_horizon & (src_ships_traj > 0.0)\n    inf_fill = torch.full_like(src_ships_traj, float(\'inf\'))\n    cap_traj = torch.where(held, src_ships_traj, inf_fill)\n    min_slack = cap_traj.min(dim=-1).values\n    return torch.minimum(min_slack, source_ships.to(dtype)).clamp(min=0.0)\n\'Observation/action adapter between the move-list format and tensors.\\n\\n\\n\\nConverts an observation dict (``{"planets": [...], "fleets": [...], ...}``) into\\n\\nthe named tensor observation the planner consumes, and converts the planner\\\'s\\n\\nsparse launch payload\\n\\n(``{"from_planet_id": [L], "angle": [L], "num_ships": [L], "counts": scalar}``)\\n\\nback into a move list (``[[from_planet_id, angle, ships], ...]``).\\n\\n\'\nfrom typing import Any\nimport torch\n\ndef _infer_player_count_from_obs(planets: list[Any], fleets: list[Any], player_id: int) -> int:\n    owners: list[int] = [int(player_id)]\n    for row in planets:\n        if len(row) >= 2 and int(row[0]) >= 0 and (int(row[1]) >= 0):\n            owners.append(int(row[1]))\n    for row in fleets:\n        if len(row) >= 2 and int(row[0]) >= 0 and (int(row[1]) >= 0):\n            owners.append(int(row[1]))\n    return 4 if max(owners, default=0) >= 2 else 2\n\ndef _dict_obs_to_tensor(obs: dict[str, Any], player_id: int, P: int=P_MAX, F: int=F_MAX, device: Any=\'cpu\') -> dict[str, Any]:\n    dev = torch.device(device)\n    planets_raw = obs.get(\'planets\', [])\n    initial_planets_raw = obs.get(\'initial_planets\', planets_raw)\n    fleets_raw = obs.get(\'fleets\', [])\n    comets_raw = obs.get(\'comets\', [])\n    comet_planet_ids_raw = obs.get(\'comet_planet_ids\', [])\n    step = int(obs.get(\'step\', 0))\n    angvel = float(obs.get(\'angular_velocity\', 0.03))\n    max_steps = int(obs.get(\'episode_steps\', DEFAULT_EPISODE_STEPS))\n    remaining_overtime = float(obs.get(\'remainingOverageTime\', 2.0))\n    next_fleet_id = int(obs.get(\'next_fleet_id\', 0))\n    planet_t = torch.zeros(P, 7, dtype=torch.float32, device=dev)\n    planet_t[..., 0] = -1.0\n    for i, p in enumerate(planets_raw[:P]):\n        pid, owner, x, y, r, ships, prod = p[:7]\n        planet_t[i, 0] = float(pid)\n        planet_t[i, 1] = float(owner)\n        planet_t[i, 2] = float(x)\n        planet_t[i, 3] = float(y)\n        planet_t[i, 4] = float(r)\n        planet_t[i, 5] = float(ships)\n        planet_t[i, 6] = float(prod)\n    initial_planet_t = torch.zeros(P, 7, dtype=torch.float32, device=dev)\n    initial_planet_t[..., 0] = -1.0\n    for i, p in enumerate(initial_planets_raw[:P]):\n        pid, owner, x, y, r, ships, prod = p[:7]\n        initial_planet_t[i, 0] = float(pid)\n        initial_planet_t[i, 1] = float(owner)\n        initial_planet_t[i, 2] = float(x)\n        initial_planet_t[i, 3] = float(y)\n        initial_planet_t[i, 4] = float(r)\n        initial_planet_t[i, 5] = float(ships)\n        initial_planet_t[i, 6] = float(prod)\n    fleet_t = torch.zeros(F, 7, dtype=torch.float32, device=dev)\n    fleet_t[..., 0] = -1.0\n    fleet_t[..., 5] = -1.0\n    for i, f in enumerate(fleets_raw[:F]):\n        fid, owner, x, y, angle, from_id, ships = f[:7]\n        fleet_t[i, 0] = float(fid)\n        fleet_t[i, 1] = float(owner)\n        fleet_t[i, 2] = float(x)\n        fleet_t[i, 3] = float(y)\n        fleet_t[i, 4] = float(angle)\n        fleet_t[i, 5] = float(from_id)\n        fleet_t[i, 6] = float(ships)\n    comet_ids = torch.full((COMET_EVENTS, COMETS_PER_EVENT), -1, dtype=torch.int32, device=dev)\n    comet_paths = torch.full((COMET_EVENTS, COMETS_PER_EVENT, COMET_PATH_MAX, 2), float(\'nan\'), dtype=torch.float32, device=dev)\n    comet_path_index = torch.full((COMET_EVENTS,), -1, dtype=torch.int32, device=dev)\n    for group_idx, group in enumerate(comets_raw[:COMET_EVENTS]):\n        comet_path_index[group_idx] = int(group.get(\'path_index\', -1))\n        group_ids = group.get(\'planet_ids\', [])\n        group_paths = group.get(\'paths\', [])\n        for comet_idx, pid in enumerate(group_ids[:COMETS_PER_EVENT]):\n            comet_ids[group_idx, comet_idx] = int(pid)\n        for comet_idx, path in enumerate(group_paths[:COMETS_PER_EVENT]):\n            for point_idx, point in enumerate(path[:COMET_PATH_MAX]):\n                comet_paths[group_idx, comet_idx, point_idx, 0] = float(point[0])\n                comet_paths[group_idx, comet_idx, point_idx, 1] = float(point[1])\n    comet_planet_ids = torch.full((COMET_EVENTS * COMETS_PER_EVENT,), -1, dtype=torch.int32, device=dev)\n    for idx, pid in enumerate(comet_planet_ids_raw[:COMET_EVENTS * COMETS_PER_EVENT]):\n        comet_planet_ids[idx] = int(pid)\n    return {\'planets\': planet_t, \'fleets\': fleet_t, \'player\': torch.tensor(player_id, dtype=torch.int32, device=dev), \'player_count\': torch.tensor(_infer_player_count_from_obs(planets_raw, fleets_raw, player_id), dtype=torch.int32, device=dev), \'angular_velocity\': torch.tensor(angvel, dtype=torch.float32, device=dev), \'initial_planets\': initial_planet_t, \'next_fleet_id\': torch.tensor(next_fleet_id, dtype=torch.int32, device=dev), \'comets\': {\'planet_ids\': comet_ids, \'paths\': comet_paths, \'path_index\': comet_path_index}, \'comet_planet_ids\': comet_planet_ids, \'step\': torch.tensor(step, dtype=torch.int32, device=dev), \'episode_steps\': torch.tensor(max_steps, dtype=torch.int32, device=dev), \'remainingOverageTime\': torch.tensor(remaining_overtime, dtype=torch.float32, device=dev)}\n\ndef _sparse_actions_to_list(action_payload: dict[str, Any], obs: dict[str, Any], player_id: int) -> list[list[Any]]:\n    from_pid_t = action_payload[\'from_planet_id\']\n    angle_t = action_payload[\'angle\']\n    num_ships_t = action_payload[\'num_ships\']\n    counts = int(action_payload[\'counts\'].item())\n    planets_by_id = {int(p[0]): p for p in obs.get(\'planets\', []) if len(p) >= 7}\n    moves: list[list[Any]] = []\n    for launch_idx in range(counts):\n        from_pid = int(from_pid_t[launch_idx].item())\n        ships = float(num_ships_t[launch_idx].item())\n        angle = float(angle_t[launch_idx].item())\n        if ships < 1.0:\n            continue\n        source = planets_by_id.get(from_pid)\n        if source is None:\n            continue\n        owner = int(source[1])\n        available = float(source[5])\n        if owner != int(player_id):\n            continue\n        if ships != float(round(ships)) or ships > available:\n            raise ValueError(f\'Invalid launch ship count in sparse action payload at from_planet_id={from_pid}: requested={ships}, available={available}. Counts must be finite, integer-valued, >= 0, and <= available planet ships.\')\n        moves.append([from_pid, angle, int(ships)])\n    return moves\n\ndef single_obs_to_tensor(obs: dict[str, Any], *, player_id: int, P: int=P_MAX, F: int=F_MAX, device: Any=\'cpu\') -> dict[str, Any]:\n    return _dict_obs_to_tensor(obs, player_id=player_id, P=P, F=F, device=device)\n\ndef sparse_action_row_to_moves(action_payload: dict[str, Any], obs: dict[str, Any], *, player_id: int) -> list[list[Any]]:\n    return _sparse_actions_to_list(action_payload, obs, player_id=int(player_id))\nimport dataclasses\nimport os\nimport sys\nfrom dataclasses import dataclass\ntry:\n    _HERE = os.path.dirname(os.path.abspath(__file__))\nexcept NameError:\n    _HERE = os.getcwd()\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\nimport torch\nfrom torch import Tensor\n\n@dataclass(frozen=True)\nclass ProducerLiteConfig:\n    horizon: int = 18\n    max_sources_per_lane: int = 12\n    max_offensive_targets: int = 12\n    max_defensive_targets: int = 4\n    max_waves_per_turn: int = 6\n    roi_threshold: float = 1.5\n    min_ships_to_launch: float = 4.0\n    enable_regroup: bool = True\n    max_regroup_time: float = 7.0\n    regroup_pressure_delta_min: float = 0.25\n    max_regroup_sources_per_lane: int = 6\n    max_regroup_targets_per_source: int = 7\n    regroup_pressure_norm: str = \'none\'\n    regroup_time_penalty_weight: float = 0.001\n    enable_potential_risk: bool = False\n    risk_blend_weight: float = 1.0\n    risk_enemy_prod_weight: float = 2.0\n    risk_self_prod_weight: float = 2.0\n    risk_support_weight: float = 0.5\n    enable_focus_fire: bool = True\n    max_strike_sources: int = 4\n\ndef _movement_config(config: ProducerLiteConfig, *, player_count: int) -> MovementConfig:\n    return MovementConfig(movement_horizon=int(config.horizon), drift_epsilon=0.001, track_fleets=True, player_count=int(player_count), max_tracked_fleets=128)\n\ndef cheap_enemy_pressure(obs, cache, *, horizon: float, player_id: int) -> Tensor:\n    P = int(obs.P)\n    device = obs.device\n    dtype = obs.ships.dtype\n    if P == 0:\n        return torch.zeros(P, dtype=dtype, device=device)\n    d0 = cache.cross_dist[0].to(dtype)\n    ships = obs.ships.to(dtype)\n    speeds = fleet_speed(ships.clamp(min=1e-06))\n    reach_dist = (speeds.view(P, 1) * float(horizon)).clamp(min=1e-06)\n    enemy = obs.alive & (obs.owner_abs >= 0) & (obs.owner_abs != int(player_id))\n    eye = torch.eye(P, device=device, dtype=torch.bool)\n    valid = enemy.view(P, 1) & obs.alive.view(1, P) & ~eye\n    decay = (1.0 - d0 / reach_dist).clamp(min=0.0)\n    contrib = torch.where(valid, ships.view(P, 1) * decay, torch.zeros_like(decay))\n    return contrib.sum(dim=0)\n\ndef potential_attack_risk(obs, cache, *, horizon: float, player_id: int, config) -> Tensor:\n    P = int(obs.P)\n    device = obs.device\n    dtype = obs.ships.dtype\n    if P == 0:\n        return torch.zeros(P, dtype=dtype, device=device)\n    H = max(float(horizon), 1e-06)\n    d0 = cache.cross_dist[0].to(dtype)\n    ships = obs.ships.to(dtype)\n    prod = obs.prod.to(dtype)\n    speeds = fleet_speed(ships.clamp(min=1e-06))\n    reach = (speeds.view(P, 1) * H).clamp(min=1e-06)\n    decay = (1.0 - d0 / reach).clamp(min=0.0)\n    eye = torch.eye(P, device=device, dtype=torch.bool)\n    x = obs.x.to(dtype)\n    y = obs.y.to(dtype)\n    ax = x.view(P, 1)\n    ay = y.view(P, 1)\n    bx = x.view(1, P)\n    by = y.view(1, P)\n    abx = bx - ax\n    aby = by - ay\n    denom = (abx * abx + aby * aby).clamp(min=1e-12)\n    u = (((CENTER - ax) * abx + (CENTER - ay) * aby) / denom).clamp(0.0, 1.0)\n    cx = ax + u * abx\n    cy = ay + u * aby\n    sun_dist = torch.sqrt(((cx - CENTER) ** 2 + (cy - CENTER) ** 2).clamp(min=0.0))\n    los_clear = (sun_dist >= float(SUN_RADIUS)).to(dtype)\n    enemy = obs.alive & (obs.owner_abs >= 0) & (obs.owner_abs != int(player_id))\n    strength = ships + float(config.risk_enemy_prod_weight) * prod\n    valid_e = enemy.view(P, 1) & obs.alive.view(1, P) & ~eye\n    threat = torch.where(valid_e, strength.view(P, 1) * decay * los_clear, torch.zeros_like(decay))\n    enemy_threat = threat.sum(dim=0)\n    own = obs.owned & obs.alive\n    valid_o = own.view(P, 1) & obs.alive.view(1, P) & ~eye\n    support = torch.where(valid_o, (1.0 + ships).view(P, 1) * decay, torch.zeros_like(decay)).sum(dim=0)\n    value = 1.0 + float(config.risk_self_prod_weight) * prod\n    return value * enemy_threat / (1.0 + float(config.risk_support_weight) * support)\n\ndef plan_lite_waves(*, movement: PlanetMovement, obs, obs_tensors: dict, cache, garrison_status, prod: Tensor, alive_by_step: Tensor, config: ProducerLiteConfig, player_count: int):\n    P = obs.P\n    device = obs.device\n    dtype = obs.ships.dtype\n    pid = int(obs.player_id)\n    H_axis = int(garrison_status.ships.shape[-1])\n    H = max(H_axis - 1, 0)\n    K_eta = max(1, min(int(config.horizon), H))\n    W = max(1, int(config.max_waves_per_turn))\n    source_mask = obs.owned & obs.alive & (obs.ships >= float(config.min_ships_to_launch))\n    if not bool(source_mask.any()):\n        return _empty_entries(device, dtype)\n    S_cap = max(1, min(int(config.max_sources_per_lane), P))\n    source_idx, source_exists = _candidate_indices(obs.ships, source_mask, S_cap)\n    target_idx, target_exists = build_target_shortlist(obs, obs_tensors, garrison_status, cache, config=config, K_eta=K_eta, H=H, prod=prod, source_mask=source_mask)\n    if not bool(target_exists.any()):\n        return _empty_entries(device, dtype)\n    S = int(source_idx.shape[0])\n    T = int(target_idx.shape[0])\n    target_is_mine = obs.owned[target_idx.clamp(0, P - 1)]\n    source_ships = obs.ships[source_idx.clamp(0, P - 1)].to(dtype)\n    H_eff = torch.full((), float(H), dtype=dtype, device=device)\n    drain = safe_drain(garrison_status, source_idx=source_idx, source_ships=source_ships, H_eff=H_eff, player_id=pid)\n    eta_cap = torch.full((T,), float(K_eta), dtype=dtype, device=device)\n    floor = capture_floor(garrison_status, target_idx=target_idx, k_max=K_eta, capture_overhead=1.0, player_id=pid)\n    K = int(floor.shape[-1])\n    sizes = drain.view(S, 1).expand(S, T).floor()\n    active = reachable_mask(movement, source_idx=source_idx, target_idx=target_idx, fleet_sizes=sizes.unsqueeze(-1), eta_cap=eta_cap).squeeze(-1)\n    aim = intercept_angle(movement, source_idx.unsqueeze(1), target_idx.unsqueeze(0), sizes, active=active)\n    angle = aim[\'angle\']\n    eta = aim[\'eta\']\n    viable = aim[\'viable\'] & (eta <= eta_cap.view(1, T))\n    if K > 0:\n        k_arr = (eta.clamp(min=1.0, max=float(K)).ceil().long() - 1).clamp(0, K - 1)\n        floor_at_arr = floor.unsqueeze(0).expand(S, T, K).gather(-1, k_arr.unsqueeze(-1)).squeeze(-1)\n    else:\n        floor_at_arr = torch.ones(S, T, dtype=dtype, device=device)\n    clears_floor = sizes >= floor_at_arr\n    src_neq_tgt = source_idx.view(S, 1) != target_idx.view(1, T)\n    valid = viable & clears_floor & (sizes >= 1.0) & src_neq_tgt & source_exists.view(S, 1) & target_exists.view(1, T)\n    if not bool(config.enable_focus_fire):\n        L = 1\n        C = S * T\n        cand_src = source_idx.view(S, 1).expand(S, T).reshape(C, L)\n        cand_tgt_slot = target_idx.view(1, T).expand(S, T).reshape(C)\n        cand_tgt_short = torch.arange(T, device=device).view(1, T).expand(S, T).reshape(C)\n        cand_send = torch.where(valid, sizes, torch.zeros_like(sizes)).reshape(C, L)\n        cand_angle = angle.reshape(C, L)\n        cand_eta = torch.where(valid, eta, torch.ones_like(eta)).reshape(C, L)\n        cand_active = valid.reshape(C, L)\n        cand_valid = valid.reshape(C)\n    else:\n        L = max(1, int(config.max_strike_sources))\n        ST = S * T\n        ss_src = torch.zeros(ST, L, dtype=torch.long, device=device)\n        ss_src[:, 0] = source_idx.view(S, 1).expand(S, T).reshape(-1)\n        ss_send = torch.zeros(ST, L, dtype=dtype, device=device)\n        ss_send[:, 0] = torch.where(valid, sizes, torch.zeros_like(sizes)).reshape(-1)\n        ss_angle = torch.zeros(ST, L, dtype=dtype, device=device)\n        ss_angle[:, 0] = angle.reshape(-1)\n        ss_eta = torch.ones(ST, L, dtype=dtype, device=device)\n        ss_eta[:, 0] = torch.where(valid, eta, torch.ones_like(eta)).reshape(-1)\n        ss_active = torch.zeros(ST, L, dtype=torch.bool, device=device)\n        ss_active[:, 0] = valid.reshape(-1)\n        ss_tgt_slot = target_idx.view(1, T).expand(S, T).reshape(-1)\n        ss_tgt_short = torch.arange(T, device=device).view(1, T).expand(S, T).reshape(-1)\n        ss_valid = valid.reshape(-1)\n        eligible = viable & (sizes >= 1.0) & src_neq_tgt & source_exists.view(S, 1) & target_exists.view(1, T)\n        step_arr = eta.clamp(min=1.0, max=float(K_eta)).ceil().long()\n        pooled = []\n        if L >= 2 and K > 0:\n            for t in range(T):\n                if bool(target_is_mine[t]):\n                    continue\n                rows = torch.nonzero(eligible[:, t], as_tuple=False).flatten()\n                if int(rows.numel()) < 2:\n                    continue\n                steps_t = step_arr[rows, t]\n                for k in torch.unique(steps_t).tolist():\n                    k = int(k)\n                    if k < 1 or k - 1 >= K:\n                        continue\n                    grp = rows[steps_t == k]\n                    if int(grp.numel()) < 2:\n                        continue\n                    gd = sizes[grp, t]\n                    order = torch.argsort(gd, descending=True, stable=True)\n                    grp = grp[order]\n                    csum = torch.cumsum(gd[order], dim=0)\n                    need = floor[t, k - 1]\n                    hit = torch.nonzero(csum >= need, as_tuple=False)\n                    if int(hit.numel()) == 0:\n                        continue\n                    j = int(hit[0].item()) + 1\n                    if j < 2 or j > L:\n                        continue\n                    pooled.append((t, grp[:j]))\n        if pooled:\n            C2 = len(pooled)\n            p_src = torch.zeros(C2, L, dtype=torch.long, device=device)\n            p_send = torch.zeros(C2, L, dtype=dtype, device=device)\n            p_angle = torch.zeros(C2, L, dtype=dtype, device=device)\n            p_eta = torch.ones(C2, L, dtype=dtype, device=device)\n            p_active = torch.zeros(C2, L, dtype=torch.bool, device=device)\n            p_tgt_slot = torch.zeros(C2, dtype=torch.long, device=device)\n            p_tgt_short = torch.zeros(C2, dtype=torch.long, device=device)\n            for i, (t, grp) in enumerate(pooled):\n                j = int(grp.numel())\n                p_src[i, :j] = source_idx[grp]\n                p_send[i, :j] = sizes[grp, t]\n                p_angle[i, :j] = angle[grp, t]\n                p_eta[i, :j] = eta[grp, t]\n                p_active[i, :j] = True\n                p_tgt_slot[i] = target_idx[t]\n                p_tgt_short[i] = t\n            cand_src = torch.cat([ss_src, p_src], dim=0)\n            cand_send = torch.cat([ss_send, p_send], dim=0)\n            cand_angle = torch.cat([ss_angle, p_angle], dim=0)\n            cand_eta = torch.cat([ss_eta, p_eta], dim=0)\n            cand_active = torch.cat([ss_active, p_active], dim=0)\n            cand_tgt_slot = torch.cat([ss_tgt_slot, p_tgt_slot], dim=0)\n            cand_tgt_short = torch.cat([ss_tgt_short, p_tgt_short], dim=0)\n            cand_valid = torch.cat([ss_valid, torch.ones(C2, dtype=torch.bool, device=device)], dim=0)\n        else:\n            cand_src, cand_send, cand_angle = (ss_src, ss_send, ss_angle)\n            cand_eta, cand_active = (ss_eta, ss_active)\n            cand_tgt_slot, cand_tgt_short, cand_valid = (ss_tgt_slot, ss_tgt_short, ss_valid)\n        C = int(cand_src.shape[0])\n    cand_is_def = target_is_mine[cand_tgt_short]\n    launches = make_launch_set(source_slots=cand_src, target_slots=cand_tgt_slot.unsqueeze(-1).expand(C, L), ships=cand_send, eta=cand_eta, valid=cand_active & cand_valid.unsqueeze(-1), player_id=pid)\n    score = score_candidates(garrison_status, prod=prod, alive_by_step=alive_by_step, player_count=int(player_count), launches=launches, player_id=pid)\n    score = torch.where(cand_valid, score, torch.full_like(score, float(\'-inf\')))\n    wave_entries, leftover = _greedy_select(P=P, W=W, device=device, dtype=dtype, score=score, cand_src=cand_src, cand_send=cand_send, cand_angle=cand_angle, cand_eta=cand_eta, cand_active=cand_active, cand_tgt_slot=cand_tgt_slot, cand_tgt_short=cand_tgt_short, cand_is_def=cand_is_def, source_budget=obs.ships.to(dtype).clone(), target_exists=target_exists, roi_threshold=float(config.roi_threshold))\n    if not bool(config.enable_regroup):\n        return wave_entries\n    enemy_mass = cheap_enemy_pressure(obs, cache, horizon=float(K_eta), player_id=pid)\n    if bool(config.enable_potential_risk):\n        enemy_mass = enemy_mass + float(config.risk_blend_weight) * potential_attack_risk(obs, cache, horizon=float(K_eta), player_id=pid, config=config)\n    regroup_entries = _plan_regroup(movement=movement, obs=obs, obs_tensors=obs_tensors, garrison_status=garrison_status, leftover=leftover, original_ships=obs.ships.to(dtype), pressure=enemy_mass, config=config, H=H)\n    return concat_launch_entries([wave_entries, regroup_entries])\n\ndef run_turn(obs_tensors: dict, *, config: ProducerLiteConfig, player_count: int, memory) -> dict:\n    device = obs_tensors[\'planets\'].device\n    obs = parse_obs(obs_tensors)\n    P = obs.P\n    if P == 0:\n        return empty_action_row(device)\n    movement = ensure_planet_movement(obs_tensors=obs_tensors, expected_cfg=_movement_config(config, player_count=int(player_count)), cached_movement=getattr(memory, \'movement\', None))\n    memory.movement = movement\n    cache = build_distance_cache(movement, max_k=int(config.horizon))\n    H = int(config.horizon)\n    status = movement.garrison_status(max_horizon=H)\n    alive_by_step = movement.alive_by_step[:H + 1]\n    entries = plan_lite_waves(movement=movement, obs=obs, obs_tensors=obs_tensors, cache=cache, garrison_status=status, prod=movement.planet_prod, alive_by_step=alive_by_step, config=config, player_count=int(player_count))\n    entries = disambiguate_duplicate_launches(entries)\n    launches = infer_planned_launches_from_entries(obs_tensors=obs_tensors, movement=movement, entries=entries, player_id=int(obs.player_id))\n    apply_private_planned_launches(movement=movement, launches=launches, owner_id=int(obs.player_id), obs_tensors=obs_tensors)\n    planet_ids = obs_tensors[\'planets\'][..., 0].long()\n    return entries_to_sparse_payload(entries, planet_ids=planet_ids)\nCONFIG_4P = dataclasses.replace(ProducerLiteConfig(), horizon=13, max_sources_per_lane=6, max_defensive_targets=2, max_regroup_time=6.0, max_regroup_targets_per_source=8, risk_blend_weight=0.5, max_strike_sources=3)\n\ndef _config_for(player_count: int) -> ProducerLiteConfig:\n    return CONFIG_4P if int(player_count) >= 4 else ProducerLiteConfig()\n\nclass ProducerLiteMemory:\n\n    def __init__(self) -> None:\n        self.movement = None\n        self.cached_player_count: int | None = None\n        self.last_sparse_action_row: dict | None = None\n\n    def reset(self) -> None:\n        self.movement = None\n        self.cached_player_count = None\n        self.last_sparse_action_row = None\n\nclass ProducerLiteRuntime:\n\n    def __init__(self, memory: ProducerLiteMemory | None=None) -> None:\n        self.memory = memory if memory is not None else ProducerLiteMemory()\n\n    def reset(self) -> None:\n        self.memory.reset()\n\n    def tensor_action(self, obs_tensors: dict):\n        mem = self.memory\n        if bool((obs_tensors[\'step\'] == 0).all()):\n            mem.cached_player_count = None\n        if mem.cached_player_count is None:\n            mem.cached_player_count = largest_initial_player_count(obs_tensors)\n        config = _config_for(mem.cached_player_count)\n        row = run_turn(obs_tensors, config=config, player_count=int(mem.cached_player_count), memory=mem)\n        mem.last_sparse_action_row = row\n        return row\n_RUNTIME = ProducerLiteRuntime()\n\ndef agent(obs):\n    player = obs.get(\'player\', 0) if isinstance(obs, dict) else obs.player\n    player_id = int(player)\n    obs_tensors = single_obs_to_tensor(obs, player_id=player_id)\n    with torch.no_grad():\n        sparse_row = _RUNTIME.tensor_action(obs_tensors)\n    return sparse_action_row_to_moves(sparse_row, obs, player_id=player_id)\n', encoding='utf-8')
print('heuristic written:', HEURISTIC_PATH, HEURISTIC_PATH.stat().st_size)


heuristic written: /kaggle/working/whole_plan_heuristic.py 138276


In [3]:
MODULE_PATH = WORK_DIR / 'whole_plan_ppo.py'
MODULE_PATH.write_text('from __future__ import annotations\n\nimport copy\nimport dataclasses\nimport importlib.util\nimport math\nimport random\nimport sys\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.distributions import Categorical\n\n\ndef obs_get(observation: Any, key: str, default: Any) -> Any:\n    if isinstance(observation, dict):\n        return observation.get(key, default)\n    return getattr(observation, key, default)\n\n\ndef load_heuristic_module(path: str | Path):\n    path = Path(path)\n    spec = importlib.util.spec_from_file_location("whole_plan_heuristic", path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Cannot import heuristic from {path}")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[spec.name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\n@dataclass(slots=True)\nclass TrainConfig:\n    seed: int = 42\n    device: str = "auto"\n    max_planets: int = 64\n    max_fleets: int = 256\n    hidden_size: int = 192\n    rollout_steps: int = 128\n    num_envs: int = 4\n    total_updates: int = 1000\n    ppo_epochs: int = 8\n    minibatch_size: int = 128\n    gamma: float = 0.995\n    gae_lambda: float = 0.95\n    clip_coef: float = 0.2\n    ent_coef: float = 0.02\n    vf_coef: float = 0.5\n    learning_rate: float = 1e-3\n    max_grad_norm: float = 0.5\n    shaping_coef: float = 0.03\n    checkpoint_every: int = 25\n    eval_every: int = 25\n    eval_games: int = 20\n    save_dir: str = "/kaggle/working/whole_plan_artifacts"\n    alternate_sides: bool = True\n\n\n@dataclass(slots=True)\nclass PlanProposal:\n    name: str\n    moves: list[list[float | int]]\n    memory: Any\n    valid: bool = True\n\n\n@dataclass(slots=True)\nclass EncodedDecision:\n    planet_features: np.ndarray\n    planet_mask: np.ndarray\n    fleet_features: np.ndarray\n    fleet_mask: np.ndarray\n    global_features: np.ndarray\n    plan_features: np.ndarray\n    plan_mask: np.ndarray\n\n\n@dataclass(slots=True)\nclass Transition:\n    encoded: EncodedDecision\n    action: int\n    log_prob: float\n    value: float\n    reward: float\n    done: bool\n\n\n@dataclass(slots=True)\nclass Batch:\n    planet_features: torch.Tensor\n    planet_mask: torch.Tensor\n    fleet_features: torch.Tensor\n    fleet_mask: torch.Tensor\n    global_features: torch.Tensor\n    plan_features: torch.Tensor\n    plan_mask: torch.Tensor\n    actions: torch.Tensor\n    old_log_probs: torch.Tensor\n    old_values: torch.Tensor\n    returns: torch.Tensor\n    advantages: torch.Tensor\n\n\ndef resolve_device(name: str) -> torch.device:\n    if name == "auto":\n        return torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    return torch.device(name)\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\nclass WholePlanLibrary:\n    """Generates complete turn plans while committing only the selected planner state."""\n\n    def __init__(self, heuristic_module: Any):\n        self.h = heuristic_module\n        base = self.h.ProducerLiteConfig()\n        self.variant_configs = [\n            ("baseline", base),\n            ("aggressive", dataclasses.replace(base, roi_threshold=0.5)),\n            ("very_aggressive", dataclasses.replace(base, roi_threshold=-0.5)),\n            ("conservative", dataclasses.replace(base, roi_threshold=3.0)),\n            ("no_regroup", dataclasses.replace(base, enable_regroup=False)),\n            (\n                "risk_regroup",\n                dataclasses.replace(\n                    base,\n                    enable_potential_risk=True,\n                    risk_blend_weight=0.5,\n                ),\n            ),\n            ("short_horizon", dataclasses.replace(base, horizon=10)),\n            ("long_horizon", dataclasses.replace(base, horizon=24)),\n            ("no_focus_fire", dataclasses.replace(base, enable_focus_fire=False)),\n        ]\n        self.names = ["pass"] + [name for name, _ in self.variant_configs]\n        self.memory = self.h.ProducerLiteMemory()\n\n    @property\n    def action_count(self) -> int:\n        return len(self.names)\n\n    def reset(self) -> None:\n        self.memory = self.h.ProducerLiteMemory()\n\n    def propose(self, observation: Any) -> list[PlanProposal]:\n        player = int(obs_get(observation, "player", 0))\n        obs_tensors = self.h.single_obs_to_tensor(observation, player_id=player)\n        player_count = int(obs_tensors["player_count"].item())\n        proposals = [\n            PlanProposal(\n                name="pass",\n                moves=[],\n                memory=copy.deepcopy(self.memory),\n                valid=True,\n            )\n        ]\n        seen = {()}\n        for name, configured in self.variant_configs:\n            config = configured\n            if player_count >= 4:\n                config = dataclasses.replace(\n                    configured,\n                    horizon=min(configured.horizon, 13),\n                    max_sources_per_lane=min(configured.max_sources_per_lane, 6),\n                    max_defensive_targets=min(configured.max_defensive_targets, 2),\n                    max_strike_sources=min(configured.max_strike_sources, 3),\n                )\n            candidate_memory = copy.deepcopy(self.memory)\n            with torch.inference_mode():\n                row = self.h.run_turn(\n                    obs_tensors,\n                    config=config,\n                    player_count=player_count,\n                    memory=candidate_memory,\n                )\n            moves = self.h.sparse_action_row_to_moves(\n                row,\n                observation,\n                player_id=player,\n            )\n            key = tuple(\n                (int(move[0]), round(float(move[1]), 5), int(move[2]))\n                for move in moves\n            )\n            valid = key not in seen\n            seen.add(key)\n            proposals.append(\n                PlanProposal(\n                    name=name,\n                    moves=moves,\n                    memory=candidate_memory,\n                    valid=valid,\n                )\n            )\n        return proposals\n\n    def commit(self, proposal: PlanProposal) -> None:\n        self.memory = proposal.memory\n\n\nPLANET_DIM = 11\nFLEET_DIM = 9\nGLOBAL_DIM = 12\nPLAN_STATS_DIM = 16\n\n\ndef encode_decision(\n    observation: Any,\n    proposals: list[PlanProposal],\n    variant_count: int,\n    cfg: TrainConfig,\n) -> EncodedDecision:\n    player = int(obs_get(observation, "player", 0))\n    planets = list(obs_get(observation, "planets", []))\n    fleets = list(obs_get(observation, "fleets", []))\n    step = int(obs_get(observation, "step", 0))\n    episode_steps = int(obs_get(observation, "episode_steps", 500))\n\n    planet_features = np.zeros((cfg.max_planets, PLANET_DIM), dtype=np.float32)\n    planet_mask = np.zeros((cfg.max_planets,), dtype=bool)\n    planet_by_id: dict[int, list[float]] = {}\n    for idx, row in enumerate(planets[: cfg.max_planets]):\n        pid, owner, x, y, radius, ships, production = row[:7]\n        pid = int(pid)\n        owner = int(owner)\n        planet_by_id[pid] = row\n        mine = owner == player\n        neutral = owner == -1\n        enemy = not mine and not neutral\n        dx = float(x) - 50.0\n        dy = float(y) - 50.0\n        planet_features[idx] = np.asarray(\n            [\n                float(mine),\n                float(enemy),\n                float(neutral),\n                float(x) / 100.0,\n                float(y) / 100.0,\n                float(radius) / 5.0,\n                math.log1p(max(float(ships), 0.0)) / math.log(1001.0),\n                float(production) / 5.0,\n                math.hypot(dx, dy) / 70.71,\n                math.sin(math.atan2(dy, dx)),\n                math.cos(math.atan2(dy, dx)),\n            ],\n            dtype=np.float32,\n        )\n        planet_mask[idx] = True\n\n    fleet_features = np.zeros((cfg.max_fleets, FLEET_DIM), dtype=np.float32)\n    fleet_mask = np.zeros((cfg.max_fleets,), dtype=bool)\n    for idx, row in enumerate(fleets[: cfg.max_fleets]):\n        _, owner, x, y, angle, from_pid, ships = row[:7]\n        from_row = planet_by_id.get(int(from_pid))\n        source_mine = from_row is not None and int(from_row[1]) == player\n        fleet_features[idx] = np.asarray(\n            [\n                float(int(owner) == player),\n                float(int(owner) != player),\n                float(x) / 100.0,\n                float(y) / 100.0,\n                math.sin(float(angle)),\n                math.cos(float(angle)),\n                math.log1p(max(float(ships), 0.0)) / math.log(1001.0),\n                float(source_mine),\n                math.hypot(float(x) - 50.0, float(y) - 50.0) / 70.71,\n            ],\n            dtype=np.float32,\n        )\n        fleet_mask[idx] = True\n\n    mine = [p for p in planets if int(p[1]) == player]\n    neutral = [p for p in planets if int(p[1]) == -1]\n    enemy = [p for p in planets if int(p[1]) not in {-1, player}]\n    my_fleets = [f for f in fleets if int(f[1]) == player]\n    enemy_fleets = [f for f in fleets if int(f[1]) != player]\n    my_ships = sum(float(p[5]) for p in mine)\n    enemy_ships = sum(float(p[5]) for p in enemy)\n    my_prod = sum(float(p[6]) for p in mine)\n    enemy_prod = sum(float(p[6]) for p in enemy)\n    global_features = np.asarray(\n        [\n            step / max(episode_steps, 1),\n            len(mine) / max(cfg.max_planets, 1),\n            len(enemy) / max(cfg.max_planets, 1),\n            len(neutral) / max(cfg.max_planets, 1),\n            math.log1p(my_ships) / math.log(10001.0),\n            math.log1p(enemy_ships) / math.log(10001.0),\n            my_prod / max(5.0 * cfg.max_planets, 1.0),\n            enemy_prod / max(5.0 * cfg.max_planets, 1.0),\n            len(my_fleets) / max(cfg.max_fleets, 1),\n            len(enemy_fleets) / max(cfg.max_fleets, 1),\n            math.log1p(sum(float(f[6]) for f in my_fleets)) / math.log(10001.0),\n            math.log1p(sum(float(f[6]) for f in enemy_fleets)) / math.log(10001.0),\n        ],\n        dtype=np.float32,\n    )\n\n    plan_features = np.zeros(\n        (variant_count, variant_count + PLAN_STATS_DIM),\n        dtype=np.float32,\n    )\n    plan_mask = np.zeros((variant_count,), dtype=bool)\n    for idx, proposal in enumerate(proposals):\n        plan_features[idx, idx] = 1.0\n        plan_features[idx, variant_count:] = plan_statistics(\n            proposal.moves,\n            planets,\n            player,\n            my_ships,\n            my_prod,\n        )\n        plan_mask[idx] = proposal.valid\n    plan_mask[0] = True\n    return EncodedDecision(\n        planet_features=planet_features,\n        planet_mask=planet_mask,\n        fleet_features=fleet_features,\n        fleet_mask=fleet_mask,\n        global_features=global_features,\n        plan_features=plan_features,\n        plan_mask=plan_mask,\n    )\n\n\ndef plan_statistics(\n    moves: list[list[float | int]],\n    planets: list[list[float]],\n    player: int,\n    my_ships: float,\n    my_prod: float,\n) -> np.ndarray:\n    if not moves:\n        return np.zeros((PLAN_STATS_DIM,), dtype=np.float32)\n    by_id = {int(p[0]): p for p in planets}\n    sent = np.asarray([float(move[2]) for move in moves], dtype=np.float32)\n    source_ids = [int(move[0]) for move in moves]\n    angles = np.asarray([float(move[1]) for move in moves], dtype=np.float32)\n    source_ships = []\n    source_prod = []\n    target_types = []\n    target_prod = []\n    distances = []\n    for source_id, angle in zip(source_ids, angles):\n        source = by_id.get(source_id)\n        if source is None:\n            continue\n        source_ships.append(float(source[5]))\n        source_prod.append(float(source[6]))\n        target = infer_target(source, angle, planets)\n        if target is None:\n            continue\n        owner = int(target[1])\n        target_types.append(\n            (\n                float(owner == player),\n                float(owner == -1),\n                float(owner not in {-1, player}),\n            )\n        )\n        target_prod.append(float(target[6]))\n        distances.append(math.hypot(float(target[2]) - float(source[2]), float(target[3]) - float(source[3])))\n    total_sent = float(sent.sum())\n    source_ships_arr = np.asarray(source_ships or [1.0], dtype=np.float32)\n    type_arr = np.asarray(target_types or [(0.0, 0.0, 0.0)], dtype=np.float32)\n    return np.asarray(\n        [\n            len(moves) / max(len(planets), 1),\n            total_sent / max(my_ships, 1.0),\n            float(sent.mean()) / 400.0,\n            float(sent.max()) / 400.0,\n            len(set(source_ids)) / max(len(moves), 1),\n            float(np.mean(source_ships_arr)) / 400.0,\n            float(np.mean(source_prod or [0.0])) / 5.0,\n            float(np.mean(np.sin(angles))),\n            float(np.mean(np.cos(angles))),\n            float(type_arr[:, 0].mean()),\n            float(type_arr[:, 1].mean()),\n            float(type_arr[:, 2].mean()),\n            float(np.mean(target_prod or [0.0])) / 5.0,\n            float(np.mean(distances or [0.0])) / 100.0,\n            total_sent / max(my_prod * 10.0, 1.0),\n            float(len(moves) > 1),\n        ],\n        dtype=np.float32,\n    )\n\n\ndef infer_target(\n    source: list[float],\n    angle: float,\n    planets: list[list[float]],\n) -> list[float] | None:\n    best = None\n    best_score = float("inf")\n    for target in planets:\n        if int(target[0]) == int(source[0]):\n            continue\n        dx = float(target[2]) - float(source[2])\n        dy = float(target[3]) - float(source[3])\n        target_angle = math.atan2(dy, dx)\n        delta = abs(math.atan2(math.sin(angle - target_angle), math.cos(angle - target_angle)))\n        distance = math.hypot(dx, dy)\n        score = delta + 0.0005 * distance\n        if score < best_score:\n            best_score = score\n            best = target\n    return best\n\n\nclass SetEncoder(nn.Module):\n    def __init__(self, input_dim: int, hidden_size: int):\n        super().__init__()\n        self.item = nn.Sequential(\n            nn.Linear(input_dim, hidden_size),\n            nn.SiLU(),\n            nn.Linear(hidden_size, hidden_size),\n            nn.SiLU(),\n        )\n        self.out = nn.Sequential(\n            nn.Linear(hidden_size * 2, hidden_size),\n            nn.SiLU(),\n        )\n\n    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n        hidden = self.item(x)\n        mask_f = mask.unsqueeze(-1).to(hidden.dtype)\n        denom = mask_f.sum(dim=1).clamp_min(1.0)\n        mean = (hidden * mask_f).sum(dim=1) / denom\n        neg_inf = torch.finfo(hidden.dtype).min\n        max_pool = hidden.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values\n        empty = ~mask.any(dim=1)\n        max_pool[empty] = 0.0\n        return self.out(torch.cat([mean, max_pool], dim=-1))\n\n\n@dataclass(slots=True)\nclass PolicyOutput:\n    logits: torch.Tensor\n    value: torch.Tensor\n\n\nclass WholePlanPolicy(nn.Module):\n    def __init__(self, action_count: int, hidden_size: int = 192):\n        super().__init__()\n        self.action_count = action_count\n        self.planet_encoder = SetEncoder(PLANET_DIM, hidden_size)\n        self.fleet_encoder = SetEncoder(FLEET_DIM, hidden_size)\n        self.global_encoder = nn.Sequential(\n            nn.Linear(GLOBAL_DIM, hidden_size),\n            nn.SiLU(),\n            nn.Linear(hidden_size, hidden_size),\n            nn.SiLU(),\n        )\n        self.state_encoder = nn.Sequential(\n            nn.Linear(hidden_size * 3, hidden_size),\n            nn.SiLU(),\n            nn.LayerNorm(hidden_size),\n        )\n        self.plan_encoder = nn.Sequential(\n            nn.Linear(action_count + PLAN_STATS_DIM, hidden_size),\n            nn.SiLU(),\n            nn.Linear(hidden_size, hidden_size),\n            nn.SiLU(),\n        )\n        self.policy_head = nn.Sequential(\n            nn.Linear(hidden_size * 2, hidden_size),\n            nn.SiLU(),\n            nn.Linear(hidden_size, 1),\n        )\n        self.value_head = nn.Sequential(\n            nn.Linear(hidden_size, hidden_size),\n            nn.SiLU(),\n            nn.Linear(hidden_size, 1),\n        )\n\n    def forward(\n        self,\n        planet_features: torch.Tensor,\n        planet_mask: torch.Tensor,\n        fleet_features: torch.Tensor,\n        fleet_mask: torch.Tensor,\n        global_features: torch.Tensor,\n        plan_features: torch.Tensor,\n        plan_mask: torch.Tensor,\n    ) -> PolicyOutput:\n        state = self.state_encoder(\n            torch.cat(\n                [\n                    self.planet_encoder(planet_features, planet_mask),\n                    self.fleet_encoder(fleet_features, fleet_mask),\n                    self.global_encoder(global_features),\n                ],\n                dim=-1,\n            )\n        )\n        plans = self.plan_encoder(plan_features)\n        expanded = state.unsqueeze(1).expand(-1, self.action_count, -1)\n        logits = self.policy_head(torch.cat([expanded, plans], dim=-1)).squeeze(-1)\n        logits = logits.masked_fill(~plan_mask, torch.finfo(logits.dtype).min)\n        value = self.value_head(state).squeeze(-1)\n        return PolicyOutput(logits=logits, value=value)\n\n\ndef encoded_to_tensors(encoded: EncodedDecision, device: torch.device) -> dict[str, torch.Tensor]:\n    return {\n        "planet_features": torch.from_numpy(encoded.planet_features).unsqueeze(0).to(device),\n        "planet_mask": torch.from_numpy(encoded.planet_mask).unsqueeze(0).to(device),\n        "fleet_features": torch.from_numpy(encoded.fleet_features).unsqueeze(0).to(device),\n        "fleet_mask": torch.from_numpy(encoded.fleet_mask).unsqueeze(0).to(device),\n        "global_features": torch.from_numpy(encoded.global_features).unsqueeze(0).to(device),\n        "plan_features": torch.from_numpy(encoded.plan_features).unsqueeze(0).to(device),\n        "plan_mask": torch.from_numpy(encoded.plan_mask).unsqueeze(0).to(device),\n    }\n\n\ndef act(\n    policy: WholePlanPolicy,\n    encoded: EncodedDecision,\n    device: torch.device,\n    deterministic: bool,\n) -> tuple[int, float, float]:\n    with torch.inference_mode():\n        output = policy(**encoded_to_tensors(encoded, device))\n        dist = Categorical(logits=output.logits)\n        action = output.logits.argmax(dim=-1) if deterministic else dist.sample()\n        log_prob = dist.log_prob(action)\n    return int(action.item()), float(log_prob.item()), float(output.value.item())\n\n\ndef potential(observation: Any) -> float:\n    player = int(obs_get(observation, "player", 0))\n    planets = list(obs_get(observation, "planets", []))\n    fleets = list(obs_get(observation, "fleets", []))\n    mine = [p for p in planets if int(p[1]) == player]\n    enemy = [p for p in planets if int(p[1]) not in {-1, player}]\n    my_planets = len(mine)\n    enemy_planets = len(enemy)\n    my_prod = sum(float(p[6]) for p in mine)\n    enemy_prod = sum(float(p[6]) for p in enemy)\n    my_ships = sum(float(p[5]) for p in mine) + sum(float(f[6]) for f in fleets if int(f[1]) == player)\n    enemy_ships = sum(float(p[5]) for p in enemy) + sum(float(f[6]) for f in fleets if int(f[1]) != player)\n    scale = max(len(planets), 1)\n    return (\n        0.35 * (my_planets - enemy_planets) / scale\n        + 0.45 * (my_prod - enemy_prod) / max(5.0 * scale, 1.0)\n        + 0.20 * (my_ships - enemy_ships) / max(my_ships + enemy_ships, 100.0)\n    )\n\n\nclass HeuristicOpponent:\n    def __init__(self, heuristic_module: Any):\n        self.h = heuristic_module\n        self.memory = self.h.ProducerLiteMemory()\n\n    def reset(self) -> None:\n        self.memory = self.h.ProducerLiteMemory()\n\n    def act(self, observation: Any) -> list[list[float | int]]:\n        player = int(obs_get(observation, "player", 0))\n        obs_tensors = self.h.single_obs_to_tensor(observation, player_id=player)\n        player_count = int(obs_tensors["player_count"].item())\n        config = self.h._config_for(player_count)\n        with torch.inference_mode():\n            row = self.h.run_turn(\n                obs_tensors,\n                config=config,\n                player_count=player_count,\n                memory=self.memory,\n            )\n        return self.h.sparse_action_row_to_moves(\n            row,\n            observation,\n            player_id=player,\n        )\n\n\nclass OrbitWarsWholePlanEnv:\n    def __init__(\n        self,\n        cfg: TrainConfig,\n        heuristic_module: Any,\n        env_index: int,\n        make_fn: Any | None = None,\n    ):\n        self.cfg = cfg\n        self.h = heuristic_module\n        self.env_index = env_index\n        self.make_fn = make_fn\n        self.library = WholePlanLibrary(heuristic_module)\n        self.opponent = HeuristicOpponent(heuristic_module)\n        self.env = None\n        self.learner_player = 0\n        self.episode_index = 0\n        self.observation = None\n        self.last_potential = 0.0\n\n    def reset(self, seed: int) -> tuple[list[PlanProposal], EncodedDecision]:\n        if self.make_fn is None:\n            from kaggle_environments import make\n\n            make_fn = make\n        else:\n            make_fn = self.make_fn\n        self.learner_player = (\n            (self.env_index + self.episode_index) % 2\n            if self.cfg.alternate_sides\n            else 0\n        )\n        self.episode_index += 1\n        self.library.reset()\n        self.opponent.reset()\n        self.env = make_fn(\n            "orbit_wars",\n            configuration={"seed": int(seed), "randomSeed": int(seed)},\n            debug=False,\n        )\n        self.env.reset(num_agents=2)\n        states = self.env.step([[], []])\n        self.observation = extract_observation(states[self.learner_player])\n        self.last_potential = potential(self.observation)\n        return self.current_decision()\n\n    def current_decision(self) -> tuple[list[PlanProposal], EncodedDecision]:\n        proposals = self.library.propose(self.observation)\n        encoded = encode_decision(\n            self.observation,\n            proposals,\n            self.library.action_count,\n            self.cfg,\n        )\n        return proposals, encoded\n\n    def step(\n        self,\n        proposal: PlanProposal,\n    ) -> tuple[float, bool, list[PlanProposal] | None, EncodedDecision | None, dict[str, Any]]:\n        self.library.commit(proposal)\n        opponent_obs = extract_observation(\n            self.env.steps[-1][1 - self.learner_player]\n        )\n        opponent_action = self.opponent.act(opponent_obs)\n        actions = [None, None]\n        actions[self.learner_player] = proposal.moves\n        actions[1 - self.learner_player] = opponent_action\n        states = self.env.step(actions)\n        player_state = states[self.learner_player]\n        opponent_state = states[1 - self.learner_player]\n        self.observation = extract_observation(player_state)\n        done = extract_status(player_state) != "ACTIVE"\n        next_potential = potential(self.observation)\n        shaped = self.cfg.shaping_coef * (\n            self.cfg.gamma * next_potential - self.last_potential\n        )\n        self.last_potential = next_potential\n        terminal = terminal_outcome(player_state, opponent_state) if done else 0.0\n        reward = terminal + shaped\n        info = {\n            "terminal": terminal,\n            "raw_reward": extract_reward(player_state),\n            "opponent_raw_reward": extract_reward(opponent_state),\n        }\n        if done:\n            return reward, True, None, None, info\n        proposals, encoded = self.current_decision()\n        return reward, False, proposals, encoded, info\n\n\ndef extract_observation(state: Any) -> Any:\n    if isinstance(state, dict):\n        return state.get("observation")\n    return getattr(state, "observation")\n\n\ndef extract_status(state: Any) -> str:\n    if isinstance(state, dict):\n        return str(state.get("status", "UNKNOWN"))\n    return str(getattr(state, "status", "UNKNOWN"))\n\n\ndef extract_reward(state: Any) -> float:\n    value = state.get("reward", 0.0) if isinstance(state, dict) else getattr(state, "reward", 0.0)\n    return 0.0 if value is None else float(value)\n\n\ndef terminal_outcome(player_state: Any, opponent_state: Any) -> float:\n    player_reward = extract_reward(player_state)\n    opponent_reward = extract_reward(opponent_state)\n    if player_reward > opponent_reward:\n        return 1.0\n    if player_reward < opponent_reward:\n        return -1.0\n    return 0.0\n\n\ndef stack_batch(\n    trajectories: list[list[Transition]],\n    bootstrap_values: list[float],\n    cfg: TrainConfig,\n) -> Batch:\n    flat: list[Transition] = []\n    all_returns: list[float] = []\n    all_advantages: list[float] = []\n    for trajectory, bootstrap in zip(trajectories, bootstrap_values):\n        advantages = [0.0] * len(trajectory)\n        last_gae = 0.0\n        next_value = bootstrap\n        for idx in reversed(range(len(trajectory))):\n            transition = trajectory[idx]\n            nonterminal = 0.0 if transition.done else 1.0\n            delta = (\n                transition.reward\n                + cfg.gamma * next_value * nonterminal\n                - transition.value\n            )\n            last_gae = (\n                delta\n                + cfg.gamma * cfg.gae_lambda * nonterminal * last_gae\n            )\n            advantages[idx] = last_gae\n            next_value = transition.value\n        flat.extend(trajectory)\n        all_advantages.extend(advantages)\n        all_returns.extend(\n            advantage + transition.value\n            for advantage, transition in zip(advantages, trajectory)\n        )\n    return Batch(\n        planet_features=torch.from_numpy(np.stack([t.encoded.planet_features for t in flat])),\n        planet_mask=torch.from_numpy(np.stack([t.encoded.planet_mask for t in flat])),\n        fleet_features=torch.from_numpy(np.stack([t.encoded.fleet_features for t in flat])),\n        fleet_mask=torch.from_numpy(np.stack([t.encoded.fleet_mask for t in flat])),\n        global_features=torch.from_numpy(np.stack([t.encoded.global_features for t in flat])),\n        plan_features=torch.from_numpy(np.stack([t.encoded.plan_features for t in flat])),\n        plan_mask=torch.from_numpy(np.stack([t.encoded.plan_mask for t in flat])),\n        actions=torch.tensor([t.action for t in flat], dtype=torch.long),\n        old_log_probs=torch.tensor([t.log_prob for t in flat], dtype=torch.float32),\n        old_values=torch.tensor([t.value for t in flat], dtype=torch.float32),\n        returns=torch.tensor(all_returns, dtype=torch.float32),\n        advantages=torch.tensor(all_advantages, dtype=torch.float32),\n    )\n\n\ndef explained_variance(y_pred: torch.Tensor, y_true: torch.Tensor) -> float:\n    var_y = torch.var(y_true, unbiased=False)\n    if float(var_y) < 1e-12:\n        return float("nan")\n    return float((1.0 - torch.var(y_true - y_pred, unbiased=False) / var_y).cpu())\n\n\ndef ppo_update(\n    policy: WholePlanPolicy,\n    optimizer: torch.optim.Optimizer,\n    batch: Batch,\n    cfg: TrainConfig,\n    device: torch.device,\n) -> dict[str, float]:\n    advantages = batch.advantages\n    advantages = (advantages - advantages.mean()) / (\n        advantages.std(unbiased=False) + 1e-8\n    )\n    size = batch.actions.shape[0]\n    metrics = {\n        "loss": 0.0,\n        "policy_loss": 0.0,\n        "value_loss": 0.0,\n        "entropy": 0.0,\n        "approx_kl": 0.0,\n        "clipfrac": 0.0,\n    }\n    updates = 0\n    for _ in range(cfg.ppo_epochs):\n        order = torch.randperm(size)\n        for start in range(0, size, cfg.minibatch_size):\n            idx = order[start : start + cfg.minibatch_size]\n            output = policy(\n                batch.planet_features[idx].to(device),\n                batch.planet_mask[idx].to(device),\n                batch.fleet_features[idx].to(device),\n                batch.fleet_mask[idx].to(device),\n                batch.global_features[idx].to(device),\n                batch.plan_features[idx].to(device),\n                batch.plan_mask[idx].to(device),\n            )\n            dist = Categorical(logits=output.logits)\n            new_log_prob = dist.log_prob(batch.actions[idx].to(device))\n            entropy = dist.entropy().mean()\n            old_log_prob = batch.old_log_probs[idx].to(device)\n            log_ratio = new_log_prob - old_log_prob\n            ratio = log_ratio.exp()\n            adv = advantages[idx].to(device)\n            policy_loss = torch.maximum(\n                -adv * ratio,\n                -adv * ratio.clamp(1.0 - cfg.clip_coef, 1.0 + cfg.clip_coef),\n            ).mean()\n            value_loss = 0.5 * (\n                output.value - batch.returns[idx].to(device)\n            ).pow(2).mean()\n            loss = policy_loss + cfg.vf_coef * value_loss - cfg.ent_coef * entropy\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            nn.utils.clip_grad_norm_(policy.parameters(), cfg.max_grad_norm)\n            optimizer.step()\n            with torch.no_grad():\n                approx_kl = ((ratio - 1.0) - log_ratio).mean()\n                clipfrac = ((ratio - 1.0).abs() > cfg.clip_coef).float().mean()\n            metrics["loss"] += float(loss.detach().cpu())\n            metrics["policy_loss"] += float(policy_loss.detach().cpu())\n            metrics["value_loss"] += float(value_loss.detach().cpu())\n            metrics["entropy"] += float(entropy.detach().cpu())\n            metrics["approx_kl"] += float(approx_kl.cpu())\n            metrics["clipfrac"] += float(clipfrac.cpu())\n            updates += 1\n    metrics = {key: value / max(updates, 1) for key, value in metrics.items()}\n    metrics["explained_variance"] = explained_variance(\n        batch.old_values,\n        batch.returns,\n    )\n    return metrics\n\n\ndef bootstrap_value(\n    policy: WholePlanPolicy,\n    encoded: EncodedDecision | None,\n    device: torch.device,\n) -> float:\n    if encoded is None:\n        return 0.0\n    with torch.inference_mode():\n        return float(policy(**encoded_to_tensors(encoded, device)).value.item())\n\n\ndef collect_rollout(\n    envs: list[OrbitWarsWholePlanEnv],\n    current: list[tuple[list[PlanProposal], EncodedDecision]],\n    policy: WholePlanPolicy,\n    cfg: TrainConfig,\n    device: torch.device,\n    next_seed: int,\n) -> tuple[Batch, list[tuple[list[PlanProposal], EncodedDecision]], int, dict[str, float]]:\n    trajectories: list[list[Transition]] = [[] for _ in envs]\n    episode_outcomes: list[float] = []\n    action_counts = np.zeros((envs[0].library.action_count,), dtype=np.int64)\n    optional_pass_count = 0\n    optional_decision_count = 0\n    for _ in range(cfg.rollout_steps):\n        for env_idx, env in enumerate(envs):\n            proposals, encoded = current[env_idx]\n            action, log_prob, value = act(policy, encoded, device, deterministic=False)\n            action_counts[action] += 1\n            if bool(encoded.plan_mask[1:].any()):\n                optional_decision_count += 1\n                optional_pass_count += int(action == 0)\n            reward, done, next_proposals, next_encoded, info = env.step(proposals[action])\n            trajectories[env_idx].append(\n                Transition(\n                    encoded=encoded,\n                    action=action,\n                    log_prob=log_prob,\n                    value=value,\n                    reward=reward,\n                    done=done,\n                )\n            )\n            if done:\n                episode_outcomes.append(float(info["terminal"]))\n                next_seed += 1\n                current[env_idx] = env.reset(next_seed)\n            else:\n                current[env_idx] = (next_proposals, next_encoded)\n    bootstraps = [\n        bootstrap_value(policy, encoded, device)\n        for _, encoded in current\n    ]\n    batch = stack_batch(trajectories, bootstraps, cfg)\n    stats = {\n        "episodes": float(len(episode_outcomes)),\n        "mean_outcome": float(np.mean(episode_outcomes)) if episode_outcomes else 0.0,\n        "win_rate": float(np.mean(np.asarray(episode_outcomes) > 0.0)) if episode_outcomes else 0.0,\n        "optional_pass_frac": float(\n            optional_pass_count / max(optional_decision_count, 1)\n        ),\n        "optional_decisions": float(optional_decision_count),\n    }\n    for idx, count in enumerate(action_counts):\n        stats[f"action_{idx}_frac"] = float(count / max(action_counts.sum(), 1))\n    return batch, current, next_seed, stats\n\n\ndef evaluate(\n    policy: WholePlanPolicy,\n    cfg: TrainConfig,\n    heuristic_module: Any,\n    device: torch.device,\n    games: int,\n    seed_start: int,\n) -> dict[str, float]:\n    outcomes = []\n    lengths = []\n    for game_idx in range(games):\n        env = OrbitWarsWholePlanEnv(cfg, heuristic_module, env_index=game_idx)\n        proposals, encoded = env.reset(seed_start + game_idx)\n        done = False\n        steps = 0\n        while not done:\n            action, _, _ = act(policy, encoded, device, deterministic=True)\n            reward, done, proposals_next, encoded_next, info = env.step(proposals[action])\n            steps += 1\n            if not done:\n                proposals, encoded = proposals_next, encoded_next\n        outcomes.append(float(info["terminal"]))\n        lengths.append(steps)\n    arr = np.asarray(outcomes)\n    return {\n        "eval_win_rate": float(np.mean(arr > 0.0)),\n        "eval_loss_rate": float(np.mean(arr < 0.0)),\n        "eval_draw_rate": float(np.mean(arr == 0.0)),\n        "eval_mean_outcome": float(arr.mean()),\n        "eval_mean_length": float(np.mean(lengths)),\n    }\n\n\ndef train(\n    cfg: TrainConfig,\n    heuristic_path: str | Path,\n) -> tuple[WholePlanPolicy, list[dict[str, float]]]:\n    seed_everything(cfg.seed)\n    device = resolve_device(cfg.device)\n    heuristic_module = load_heuristic_module(heuristic_path)\n    action_library = WholePlanLibrary(heuristic_module)\n    action_count = action_library.action_count\n    action_names = action_library.names\n    policy = WholePlanPolicy(action_count, cfg.hidden_size).to(device)\n    optimizer = torch.optim.AdamW(\n        policy.parameters(),\n        lr=cfg.learning_rate,\n        weight_decay=1e-4,\n    )\n    envs = [\n        OrbitWarsWholePlanEnv(cfg, heuristic_module, env_index=idx)\n        for idx in range(cfg.num_envs)\n    ]\n    next_seed = cfg.seed\n    current = []\n    for env in envs:\n        current.append(env.reset(next_seed))\n        next_seed += 1\n    save_dir = Path(cfg.save_dir)\n    save_dir.mkdir(parents=True, exist_ok=True)\n    history: list[dict[str, float]] = []\n    best_eval = -float("inf")\n    for update in range(1, cfg.total_updates + 1):\n        batch, current, next_seed, rollout_stats = collect_rollout(\n            envs,\n            current,\n            policy,\n            cfg,\n            device,\n            next_seed,\n        )\n        metrics = ppo_update(policy, optimizer, batch, cfg, device)\n        row = {"update": float(update), **rollout_stats, **metrics}\n        if update % cfg.eval_every == 0 or update == 1:\n            eval_stats = evaluate(\n                policy,\n                cfg,\n                heuristic_module,\n                device,\n                cfg.eval_games,\n                seed_start=100_000 + update * cfg.eval_games,\n            )\n            row.update(eval_stats)\n            if eval_stats["eval_mean_outcome"] > best_eval:\n                best_eval = eval_stats["eval_mean_outcome"]\n                torch.save(\n                    {\n                        "policy": policy.state_dict(),\n                        "config": dataclasses.asdict(cfg),\n                        "action_names": action_names,\n                        "update": update,\n                    },\n                    save_dir / "best.pt",\n                )\n        history.append(row)\n        if update % cfg.checkpoint_every == 0 or update == cfg.total_updates:\n            torch.save(\n                {\n                    "policy": policy.state_dict(),\n                    "optimizer": optimizer.state_dict(),\n                    "config": dataclasses.asdict(cfg),\n                    "action_names": action_names,\n                    "update": update,\n                    "history": history,\n                },\n                save_dir / "last.pt",\n            )\n        action_fractions = [\n            row.get(f"action_{idx}_frac", 0.0)\n            for idx in range(action_count)\n        ]\n        top_action_idx = int(np.argmax(action_fractions))\n        progress = (\n            f"update={update:04d} outcome={row[\'mean_outcome\']:+.3f} "\n            f"loss={row[\'loss\']:.4f} ev={row[\'explained_variance\']:+.3f} "\n            f"kl={row[\'approx_kl\']:.5f} clipfrac={row[\'clipfrac\']:.3f} "\n            f"entropy={row[\'entropy\']:.3f} "\n            f"pass={action_fractions[0]:.3f} "\n            f"optional_pass={row[\'optional_pass_frac\']:.3f} "\n            f"baseline={action_fractions[1]:.3f} "\n            f"top={action_names[top_action_idx]}:{action_fractions[top_action_idx]:.3f}"\n        )\n        if "eval_win_rate" in row:\n            progress += (\n                f" eval_w/l/d={row[\'eval_win_rate\']:.3f}/"\n                f"{row[\'eval_loss_rate\']:.3f}/"\n                f"{row[\'eval_draw_rate\']:.3f}"\n            )\n        if (\n            row["optional_decisions"] > 0\n            and row["optional_pass_frac"] >= 0.75\n        ):\n            progress += " WARNING=pass_collapse"\n        print(progress)\n    return policy, history\n', encoding='utf-8')
print('trainer written:', MODULE_PATH, MODULE_PATH.stat().st_size)


trainer written: /kaggle/working/whole_plan_ppo.py 37402


## Config

In [5]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location("whole_plan_ppo", MODULE_PATH)
wp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = wp
spec.loader.exec_module(wp)

cfg = wp.TrainConfig(
    seed=42,
    device="auto",
    rollout_steps=128,
    num_envs=4,
    total_updates=1000,
    ppo_epochs=8,
    minibatch_size=128,
    gamma=0.995,
    gae_lambda=0.95,
    clip_coef=0.2,
    ent_coef=0.02,
    vf_coef=0.5,
    learning_rate=1e-3,
    shaping_coef=0.03,
    checkpoint_every=25,
    eval_every=25,
    eval_games=20,
    save_dir=str(WORK_DIR / "whole_plan_artifacts"),
)

heuristic = wp.load_heuristic_module(HEURISTIC_PATH)
library = wp.WholePlanLibrary(heuristic)
print("actions:", library.names)
device = wp.resolve_device(cfg.device)
print("parameters will use device:", device)
if device.type != "cuda":
    raise RuntimeError(
        "CUDA is not available in this notebook kernel. "
        "Select the RL .venv kernel and restart the notebook."
    )


actions: ['pass', 'baseline', 'aggressive', 'very_aggressive', 'conservative', 'no_regroup', 'risk_regroup', 'short_horizon', 'long_horizon', 'no_focus_fire']
parameters will use device: cuda


In [7]:
from kaggle_environments import make

env = make("orbit_wars", configuration={"randomSeed": cfg.seed}, debug=False)
env.reset(num_agents=2)
states = env.step([[], []])
obs = states[0].observation
proposals = library.propose(obs)
encoded = wp.encode_decision(obs, proposals, library.action_count, cfg)
policy = wp.WholePlanPolicy(library.action_count, cfg.hidden_size).to(wp.resolve_device(cfg.device))
action, log_prob, value = wp.act(
    policy,
    encoded,
    wp.resolve_device(cfg.device),
    deterministic=False,
)

print("planet features:", encoded.planet_features.shape)
print("fleet features:", encoded.fleet_features.shape)
print("plan features:", encoded.plan_features.shape)
print("valid plans:", [p.name for p, valid in zip(proposals, encoded.plan_mask) if valid])
print("sampled:", proposals[action].name, "log_prob:", log_prob, "value:", value)
print("moves:", proposals[action].moves)


planet features: (64, 11)
fleet features: (256, 9)
plan features: (10, 26)
valid plans: ['pass', 'baseline']
sampled: pass log_prob: -0.6938596963882446 value: -0.03003571927547455
moves: []


## Train


In [8]:
policy, history = wp.train(cfg, HEURISTIC_PATH)

HISTORY_PATH = Path(cfg.save_dir) / "history.json"
HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")
print("history:", HISTORY_PATH)
print("best checkpoint:", Path(cfg.save_dir) / "best.pt")
print("last checkpoint:", Path(cfg.save_dir) / "last.pt")


update=0001 outcome=+0.000 loss=0.0006 ev=-0.035 kl=0.00245 clipfrac=0.018 entropy=0.678 pass=0.588 optional_pass=0.392 baseline=0.105 top=pass:0.588 eval_w/l/d=0.000/1.000/0.000
update=0002 outcome=+0.000 loss=-0.0196 ev=+0.736 kl=0.00401 clipfrac=0.035 entropy=0.863 pass=0.551 optional_pass=0.406 baseline=0.109 top=pass:0.551
update=0003 outcome=+0.000 loss=-0.0154 ev=+0.885 kl=0.00368 clipfrac=0.041 entropy=0.511 pass=0.709 optional_pass=0.452 baseline=0.092 top=pass:0.709
update=0004 outcome=-0.200 loss=-0.0318 ev=+0.930 kl=0.00998 clipfrac=0.164 entropy=0.739 pass=0.596 optional_pass=0.419 baseline=0.125 top=pass:0.596
update=0005 outcome=+0.600 loss=-0.0240 ev=+0.961 kl=0.00732 clipfrac=0.106 entropy=0.721 pass=0.596 optional_pass=0.433 baseline=0.104 top=pass:0.596
update=0006 outcome=-0.333 loss=-0.0261 ev=+0.983 kl=0.00776 clipfrac=0.110 entropy=0.712 pass=0.623 optional_pass=0.484 baseline=0.121 top=pass:0.623
update=0007 outcome=-0.200 loss=-0.0258 ev=+0.978 kl=0.00656 clipf

KeyboardInterrupt: 

## Benchmark

In [11]:
BEST_PATH = Path(cfg.save_dir) / "best.pt"
OPPONENTS = {
    "ver12": "/kaggle/input/datasets/cunmincut/opponents/ver12.py",
    "top_now": "/kaggle/input/datasets/cunmincut/opponents/top_now.py",
    "h3b1": "/kaggle/input/datasets/cunmincut/opponents/h3b1.py",
    "multi-focus-new": "/kaggle/input/datasets/cunmincut/opponents/multi_focus_hybrid_v1a_enemy_prod_bias_v2.py"
}
BENCHMARK_SEEDS = list(range(201, 211))

import importlib.util
import statistics
import sys

import pandas as pd
from IPython.display import display
from kaggle_environments import make


def load_python_agent(path, module_name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import agent: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    if not hasattr(module, "agent"):
        raise AttributeError(f"{path} does not expose agent(obs, config=None)")
    return module.agent


class BestWholePlanAgent:
    def __init__(self, checkpoint_path):
        self.device = wp.resolve_device(cfg.device)
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.heuristic = wp.load_heuristic_module(HEURISTIC_PATH)
        self.library = wp.WholePlanLibrary(self.heuristic)
        expected_names = checkpoint.get("action_names")
        if expected_names is not None and list(expected_names) != self.library.names:
            raise ValueError("Checkpoint action names do not match this planner library")
        checkpoint_cfg = checkpoint.get("config", {})
        hidden_size = int(checkpoint_cfg.get("hidden_size", cfg.hidden_size))
        self.policy = wp.WholePlanPolicy(
            self.library.action_count,
            hidden_size,
        ).to(self.device)
        self.policy.load_state_dict(checkpoint["policy"])
        self.policy.eval()

    def __call__(self, obs, config=None):
        if int(wp.obs_get(obs, "step", 0)) == 0:
            self.library.reset()
        proposals = self.library.propose(obs)
        encoded = wp.encode_decision(
            obs,
            proposals,
            self.library.action_count,
            cfg,
        )
        action, _, _ = wp.act(
            self.policy,
            encoded,
            self.device,
            deterministic=True,
        )
        proposal = proposals[action]
        self.library.commit(proposal)
        return proposal.moves


def slot_metrics(env, slot):
    total_actions = acted_turns = observable_turns = 0
    for step in env.steps:
        state = step[slot]
        if getattr(state, "observation", None) is None:
            continue
        observable_turns += 1
        actions = state.action or []
        total_actions += len(actions)
        acted_turns += int(bool(actions))
    reward = float(env.steps[-1][slot].reward or 0.0)
    return {
        "reward": reward,
        "steps": len(env.steps),
        "moves_per_turn": total_actions / max(observable_turns, 1),
        "idle_rate": 1.0 - acted_turns / max(observable_turns, 1),
    }


def summarize_benchmark(raw_df):
    rows = []
    for opponent, group in raw_df.groupby("opponent", sort=False):
        wins = int((group.reward > 0).sum())
        losses = int((group.reward < 0).sum())
        draws = len(group) - wins - losses
        rows.append({
            "opponent": opponent,
            "games": len(group),
            "wins": wins,
            "losses": losses,
            "draws": draws,
            "win_rate": wins / len(group),
            "avg_reward": statistics.fmean(group.reward),
            "avg_steps": statistics.fmean(group.steps),
            "avg_moves_per_turn": statistics.fmean(group.moves_per_turn),
            "avg_idle_rate": statistics.fmean(group.idle_rate),
        })
    return pd.DataFrame(rows).sort_values("opponent").reset_index(drop=True)


if not BEST_PATH.exists():
    raise FileNotFoundError(f"best.pt not found: {BEST_PATH}")
if not OPPONENTS:
    raise ValueError("Fill at least one opponent path in OPPONENTS")

loaded_opponents = {
    name: load_python_agent(path, f"whole_plan_opp_{idx}_{name}")
    for idx, (name, path) in enumerate(OPPONENTS.items())
}

benchmark_rows = []
total_games = len(loaded_opponents) * len(BENCHMARK_SEEDS) * 2
game_idx = 0
for opponent_name, opponent_agent in loaded_opponents.items():
    for seed in BENCHMARK_SEEDS:
        for learner_slot in (0, 1):
            # A fresh object is required for every game so planner memory cannot leak.
            best_agent = BestWholePlanAgent(BEST_PATH)
            players = (
                [best_agent, opponent_agent]
                if learner_slot == 0
                else [opponent_agent, best_agent]
            )
            env = make(
                "orbit_wars",
                debug=False,
                configuration={"randomSeed": seed},
            )
            env.run(players)
            metrics = slot_metrics(env, learner_slot)
            game_idx += 1
            verdict = (
                "WIN" if metrics["reward"] > 0
                else "LOSS" if metrics["reward"] < 0
                else "DRAW"
            )
            print(
                f"[{game_idx}/{total_games}] best@{learner_slot} "
                f"vs {opponent_name} seed={seed} {verdict} "
                f"reward={metrics['reward']:+.1f} "
                f"steps={metrics['steps']} "
                f"moves/t={metrics['moves_per_turn']:.3f} "
                f"idle={metrics['idle_rate']:.3f}",
                flush=True,
            )
            benchmark_rows.append({
                "agent": "whole_plan_best",
                "opponent": opponent_name,
                "seed": seed,
                "seat": learner_slot,
                **metrics,
            })

BEST_BENCHMARK_RAW = pd.DataFrame(benchmark_rows)
BEST_BENCHMARK_SUMMARY = summarize_benchmark(BEST_BENCHMARK_RAW)
BENCHMARK_DIR = Path(cfg.save_dir) / "benchmark"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
BEST_BENCHMARK_RAW.to_csv(BENCHMARK_DIR / "best_vs_opponents_raw.csv", index=False)
BEST_BENCHMARK_SUMMARY.to_csv(
    BENCHMARK_DIR / "best_vs_opponents_summary.csv",
    index=False,
)

display(BEST_BENCHMARK_SUMMARY)
print("Saved:", BENCHMARK_DIR)

[1/80] best@0 vs ver12 seed=201 LOSS reward=-1.0 steps=109 moves/t=1.477 idle=0.266
[2/80] best@1 vs ver12 seed=201 LOSS reward=-1.0 steps=110 moves/t=1.336 idle=0.145
[3/80] best@0 vs ver12 seed=202 LOSS reward=-1.0 steps=102 moves/t=1.451 idle=0.235
[4/80] best@1 vs ver12 seed=202 LOSS reward=-1.0 steps=112 moves/t=1.188 idle=0.188
[5/80] best@0 vs ver12 seed=203 WIN reward=+1.0 steps=155 moves/t=0.529 idle=0.555
[6/80] best@1 vs ver12 seed=203 LOSS reward=-1.0 steps=96 moves/t=1.594 idle=0.156
[7/80] best@0 vs ver12 seed=204 WIN reward=+1.0 steps=188 moves/t=1.090 idle=0.324
[8/80] best@1 vs ver12 seed=204 LOSS reward=-1.0 steps=121 moves/t=1.314 idle=0.264
[9/80] best@0 vs ver12 seed=205 LOSS reward=-1.0 steps=100 moves/t=0.830 idle=0.370
[10/80] best@1 vs ver12 seed=205 LOSS reward=-1.0 steps=128 moves/t=1.891 idle=0.133
[11/80] best@0 vs ver12 seed=206 LOSS reward=-1.0 steps=111 moves/t=1.180 idle=0.360
[12/80] best@1 vs ver12 seed=206 LOSS reward=-1.0 steps=203 moves/t=0.966 idl

,opponent,games,wins,losses,draws,win_rate,avg_reward,avg_steps,avg_moves_per_turn,avg_idle_rate
0,h3b1,20,14,6,0,0.70,0.4,151.40,1.564053,0.244384
1,multi-focus-new,20,3,17,0,0.15,-0.7,127.05,1.304574,0.268095
2,top_now,20,12,8,0,0.60,0.2,148.15,1.723880,0.178369
3,ver12,20,3,17,0,0.15,-0.7,123.40,1.303704,0.283643


Saved: /kaggle/working/whole_plan_artifacts/benchmark


## Package

In [12]:
import importlib.util
import shutil
import sys
import time

from kaggle_environments import make

SUBMISSION_DIR = WORK_DIR / "whole_plan_submission"
SUBMISSION_ZIP = WORK_DIR / "whole_plan_submission.zip"
BEST_PATH = Path(cfg.save_dir) / "best.pt"

if not BEST_PATH.exists():
    raise FileNotFoundError(f"Cannot export without best checkpoint: {BEST_PATH}")

if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

(SUBMISSION_DIR / "main.py").write_text(
    'from __future__ import annotations\n\nimport importlib.util\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\n\n\nHERE = Path(__file__).resolve().parent\n\n\ndef _load_module(name: str, path: Path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Cannot import {name} from {path}")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\nwp = _load_module("submission_whole_plan_ppo", HERE / "whole_plan_ppo.py")\nheuristic = wp.load_heuristic_module(HERE / "whole_plan_heuristic.py")\n\n\nclass SubmissionAgent:\n    def __init__(self) -> None:\n        self.device = torch.device("cpu")\n        try:\n            checkpoint = torch.load(\n                HERE / "best.pt",\n                map_location=self.device,\n                weights_only=False,\n            )\n        except TypeError:\n            checkpoint = torch.load(\n                HERE / "best.pt",\n                map_location=self.device,\n            )\n        checkpoint_cfg = checkpoint.get("config", {})\n        self.cfg = wp.TrainConfig()\n        for key, value in checkpoint_cfg.items():\n            if hasattr(self.cfg, key):\n                setattr(self.cfg, key, value)\n        self.cfg.device = "cpu"\n        self.library = wp.WholePlanLibrary(heuristic)\n        checkpoint_actions = checkpoint.get("action_names")\n        if (\n            checkpoint_actions is not None\n            and list(checkpoint_actions) != self.library.names\n        ):\n            raise ValueError("Checkpoint action library does not match package")\n        self.policy = wp.WholePlanPolicy(\n            self.library.action_count,\n            int(self.cfg.hidden_size),\n        ).to(self.device)\n        self.policy.load_state_dict(checkpoint["policy"])\n        self.policy.eval()\n\n    def act(self, observation: Any) -> list[list[float | int]]:\n        if int(wp.obs_get(observation, "step", 0)) == 0:\n            self.library.reset()\n        proposals = self.library.propose(observation)\n        encoded = wp.encode_decision(\n            observation,\n            proposals,\n            self.library.action_count,\n            self.cfg,\n        )\n        action, _, _ = wp.act(\n            self.policy,\n            encoded,\n            self.device,\n            deterministic=True,\n        )\n        proposal = proposals[action]\n        self.library.commit(proposal)\n        return proposal.moves\n\n\n_AGENT: SubmissionAgent | None = None\n\n\ndef agent(obs: Any, config: Any = None) -> list[list[float | int]]:\n    global _AGENT\n    if _AGENT is None:\n        _AGENT = SubmissionAgent()\n    return _AGENT.act(obs)\n',
    encoding="utf-8",
)
shutil.copy2(BEST_PATH, SUBMISSION_DIR / "best.pt")
shutil.copy2(MODULE_PATH, SUBMISSION_DIR / "whole_plan_ppo.py")
shutil.copy2(HEURISTIC_PATH, SUBMISSION_DIR / "whole_plan_heuristic.py")

# Import exactly as Kaggle will: main.py with package files beside it.
module_name = "whole_plan_submission_smoke"
spec = importlib.util.spec_from_file_location(
    module_name,
    SUBMISSION_DIR / "main.py",
)
submission = importlib.util.module_from_spec(spec)
sys.modules[module_name] = submission
spec.loader.exec_module(submission)

smoke_env = make(
    "orbit_wars",
    debug=False,
    configuration={"randomSeed": 987654},
)
smoke_env.reset(num_agents=2)
smoke_states = smoke_env.step([[], []])
smoke_obs = smoke_states[0].observation
t0 = time.perf_counter()
smoke_action = submission.agent(smoke_obs)
smoke_ms = (time.perf_counter() - t0) * 1000.0
print("smoke action:", smoke_action)
print(f"first action latency: {smoke_ms:.1f} ms")

if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()
archive_path = shutil.make_archive(
    str(SUBMISSION_ZIP.with_suffix("")),
    "zip",
    root_dir=SUBMISSION_DIR,
)

print("\nPackage files:")
for path in sorted(SUBMISSION_DIR.iterdir()):
    print(f"  {path.name}: {path.stat().st_size / 1024:.1f} KiB")
print("\nSubmission ZIP:", archive_path)


smoke action: [[8, 1.9309114217758179, 13]]
first action latency: 272.0 ms

Package files:
  __pycache__: 4.0 KiB
  best.pt: 2083.0 KiB
  main.py: 2.7 KiB
  whole_plan_heuristic.py: 135.0 KiB
  whole_plan_ppo.py: 36.5 KiB

Submission ZIP: /kaggle/working/whole_plan_submission.zip
